# Uplift Modeling

This notebook is the central modeling notebook for the CRITEO-UPLIFTv2.1 causal
uplift study. It reads the frozen `f0`-`f11` feature contract, the sealed
70/15/15 train/validation/held-out split, and the frozen no-op preprocessing
transform established by the prior notebook, and evaluates uplift-ranking
methods through the frozen metric contract only.

Sections are added as each method's real, executed results exist -- this is a
running record, not a template with placeholder results. As of this run, T07
(Random reference + Response LightGBM baseline) is in progress: its
correctness/artifact-mechanism SMOKE rehearsal (D30) has executed; its FULL,
authoritative development run has not yet been authorized.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_PATH = REPO_ROOT / 'kaggle' / '02_uplift_modeling.ipynb'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
T04_CONFIG_PATH = REPO_ROOT / 'configs' / 't04_preprocessing.json'
T07_CONFIG_PATH = REPO_ROOT / 'configs' / 't07_baselines.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


## 0. Environment and reproducibility setup

In [ ]:
import importlib

import lightgbm as lgb

from src.lightgbm_baseline import FROZEN_LIGHTGBM_VERSION

if lgb.__version__ != FROZEN_LIGHTGBM_VERSION:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'lightgbm=={FROZEN_LIGHTGBM_VERSION}'])
    raise RuntimeError(
        f'LightGBM was {lgb.__version__}; installed {FROZEN_LIGHTGBM_VERSION}. '
        'A hot importlib.reload() is not sufficient reproducibility enforcement for an '
        'already-imported compiled extension module -- restart the kernel and re-run this '
        'notebook from the top so the freshly-installed binary is what actually loads.'
    )

assert lgb.__version__ == FROZEN_LIGHTGBM_VERSION, (
    f'LightGBM version mismatch after guard: runtime={lgb.__version__}, frozen={FROZEN_LIGHTGBM_VERSION}. '
    'Refusing to fit on an unverified version.'
)
print(f'LightGBM verified: {lgb.__version__}')


In [ ]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import psutil
import sklearn
from sklearn.model_selection import train_test_split

from src.data import (
    DataContractError, FEATURE_COLUMNS, PROCESSED_COLUMNS, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME,
    assert_model_feature_contract, finalize_artifact_manifest, load_selector,
    materialize_pandas, open_processed_dataset, sha256_file, write_bytes_new,
    write_json_new, write_text_new,
)
from src.split import SplitDataset, SplitContractError, membership_hash
from src.preprocessing import IdentityFeatureTransform, preprocessing_contract
import src.metrics as metrics
import src.lightgbm_baseline as lgb_baseline

process = psutil.Process()
notebook_baseline_rss_bytes = process.memory_info().rss
print('Setup complete.')


## 1. Modeling Objective

Two non-causal reference points are established before any causal estimator is
trained (D11, D12, D27): a **Random reference** (a seeded ranking independent
of `X`, `T`, and `Y`, giving the no-skill floor every method is compared
against), and a **Response LightGBM baseline** (`P(Y=1|X)`, a plain
factual-outcome classifier that never sees `T`).

Response probability and treatment effect are different quantities. A
"sure thing" -- someone who converts whether or not they are treated -- ranks
high on response but contributes zero incremental value; a "persuadable" --
who converts only if treated -- can rank low on response despite being exactly
who targeting should reach. A response model with strong AUC can therefore
still rank poorly on uplift, and that is the expected, reportable finding, not
an error to fix. Response's own diagnostics (ROC-AUC, average precision, log
loss) are evaluated separately from its uplift-ranking performance and never
select a causal winner (D27).

Both methods' scores are routed through the same frozen T06 metric interface
(`src/metrics.py`) used by every later estimator, so comparisons are on equal
footing from the start.


## 2. Data, Split & Preprocessing Contracts

In [ ]:
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))
t04_config = json.loads(T04_CONFIG_PATH.read_text(encoding='utf-8'))
t07_config = json.loads(T07_CONFIG_PATH.read_text(encoding='utf-8'))

assert t05_config['lifecycle_state'] == 'T05_SPLIT_ACCEPTED', t05_config['lifecycle_state']
assert t04_config['lifecycle_state'] == 'T04_ACCEPTED', t04_config['lifecycle_state']

t05_run_manifest_path = REPO_ROOT / t05_config['lifecycle_state_evidence']['authorizing_run_manifest']
t05_run_root = t05_run_manifest_path.parent.parent
membership = pd.read_csv(t05_run_root / 'audit' / 'split_membership.csv')
observed_membership_hash = membership_hash(membership)
expected_membership_hash = t05_config['lifecycle_state_evidence']['membership_sha256']
if observed_membership_hash != expected_membership_hash:
    raise SplitContractError(
        f'Split membership hash mismatch: expected {expected_membership_hash}, observed {observed_membership_hash}'
    )

split_dataset = SplitDataset(membership=membership)
train_ids_full = split_dataset.train_ids()
validation_ids_full = split_dataset.validation_ids()
print(f'Split membership verified. train={len(train_ids_full):,} validation={len(validation_ids_full):,}')


In [ ]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload['processed_sha256']

full_frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(full_frame.columns) != PROCESSED_COLUMNS:
    raise RuntimeError('Processed frame column order drifted from the frozen contract')

print(f'Processed dataset loaded: {len(full_frame):,} rows, sha256={processed_sha256[:16]}...')


In [ ]:
RUN_T07_STAGE = False  # T07 (Random reference + Response LightGBM baseline) is already accepted.
T07_FULL_RUN_ID_ACCEPTED = 't07_full_20260818T111446Z_593205'
T07_ERRATUM_RUN_ID_ACCEPTED = 't07_audit_erratum_20260818T150917Z_995999'
T07_T08_RANDOM_LABEL_ERRATUM_RUN_ID_ACCEPTED = 't07_t08_random_label_erratum_20260818T162445Z_587615'

if not RUN_T07_STAGE:
    t07_full_root = REPO_ROOT / 'outputs' / 'runs' / T07_FULL_RUN_ID_ACCEPTED
    t07_manifest_path = t07_full_root / 'audit' / 'artifact_manifest.json'
    if not t07_manifest_path.is_file():
        raise RuntimeError(
            f'RUN_T07_STAGE is False, but the accepted T07 FULL run evidence was not found at '
            f'{t07_full_root}. This environment does not have the governed T07 evidence mounted. '
            f'Refusing to silently set RUN_T07_STAGE = True and recompute T07 -- either provide/mount '
            f'the accepted governed run evidence (outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}/), or '
            f'explicitly set RUN_T07_STAGE = True in this cell to opt into reproducing T07 from scratch.'
        )
    t07_manifest = json.loads(t07_manifest_path.read_text(encoding='utf-8'))
    t07_artifact_hashes = {a['path']: a['sha256'] for a in t07_manifest['artifacts']}
    t07_model_summary_path = t07_full_root / 'tables' / 'model_summary.csv'
    t07_model_summary_actual_sha256 = hashlib.sha256(t07_model_summary_path.read_bytes()).hexdigest()
    t07_model_summary_expected_sha256 = t07_artifact_hashes.get('tables/model_summary.csv')
    if t07_model_summary_expected_sha256 is None or t07_model_summary_actual_sha256 != t07_model_summary_expected_sha256:
        raise RuntimeError(
            f'T07 FULL tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t07_model_summary_expected_sha256}, actual {t07_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    t07_reference_summary = pd.read_csv(t07_model_summary_path)
    # Label erratum (audit-only; numeric values unchanged, see
    # T07_T08_RANDOM_LABEL_ERRATUM_RUN_ID_ACCEPTED): the 'random' row reports the seed-42
    # illustrative draw, not the theoretical no-skill reference -- relabel in memory only,
    # the source file on disk is never touched.
    t07_reference_summary = t07_reference_summary.copy()
    t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random', 'ranking_method'] = 'random_seed_42'
    t07_theoretical_random_qini_area = float(t07_reference_summary['theoretical_random_qini_area'].iloc[0])
    print('Using hash-verified frozen development results (T07).')
    print(f'  Reproducibility: T07 FULL run_id={T07_FULL_RUN_ID_ACCEPTED}')
else:
    print('RUN_T07_STAGE = True: T07 SMOKE and FULL will be recomputed from scratch below.')


## 3. Scale-Gating (D30): SMOKE Verification

Under D30, T07 uses `SMOKE -> FULL` with `resource_gates = 0` (recorded and
justified in `configs/t07_baselines.json`: the full-scale data path is already
proven by T01, and a single Response LightGBM binary classifier over 12
numeric features does not meet D30's resource-risk trigger). SMOKE is a
bounded, development-only rehearsal of correctness, the feature/leakage
contract, row alignment, serialization/reload, and artifact mechanics -- it
never supports a performance claim or selects a model/config/seed. This
section executes SMOKE only; FULL is a separate, later-authorized run.


*(This section -- T07 Random reference and Response LightGBM baseline -- is already accepted. It is skipped by default on Run All; see `RUN_T07_STAGE` immediately above. Set it to `True` only to deliberately reproduce T07 from scratch.)*

In [ ]:
if RUN_T07_STAGE:
    SMOKE_SIZE = t07_config['scale_gating']['smoke_size']
    SMOKE_SEED = t07_config['scale_gating']['smoke_seed']
    RESOURCE_GATES = t07_config['scale_gating']['resource_gates']
    RUN_FULL_STAGE = True  # FULL authorized this run under D30 (resource_gates=0), after the SMOKE PASS above.

    smoke_started = datetime.now(timezone.utc)
    smoke_wall_start = __import__('time').perf_counter()

    RUN_ID = smoke_started.strftime('t07_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
    RUN_ROOT.mkdir(parents=True, exist_ok=False)

    try:
        git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        git_head, git_dirty = None, None

    print(f'RUN_ID = {RUN_ID}')
    print(f'resource_gates = {RESOURCE_GATES}, smoke_size = {SMOKE_SIZE}, seed = {SMOKE_SEED}')


In [ ]:
if RUN_T07_STAGE:
    def _joint_strata(frame, id_column, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_partition(partition_ids, quota, full_frame, seed):
        # One deterministic joint-(T,Y)-stratified draw of `quota` rows from
        # `partition_ids`, reusing the same train_test_split stratification
        # mechanism already used by src/split.py's assign_split() -- no new
        # sampling algorithm, no reusable scale-rung module.
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata(subset, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    p_train = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
    train_quota = round(SMOKE_SIZE * p_train)
    validation_quota = SMOKE_SIZE - train_quota

    smoke_train_ids = smoke_sample_partition(train_ids_full, train_quota, full_frame, SMOKE_SEED)
    smoke_validation_ids = smoke_sample_partition(validation_ids_full, validation_quota, full_frame, SMOKE_SEED)

    # Held-out isolation is proved positively, by construction, from the two
    # sanctioned development partitions only (`train_ids_full`/`validation_ids_full`,
    # obtained solely via SplitDataset.train_ids()/.validation_ids()). This never
    # reads split_membership.csv's held_out label, never calls
    # SplitDataset.held_out_ids(), and never inspects any held-out row, ID,
    # feature, label, or summary through any path -- the held-out partition is
    # simply absent from every set these assertions reference.
    smoke_total = len(smoke_train_ids) + len(smoke_validation_ids)
    development_ids_full = set(train_ids_full) | set(validation_ids_full)
    assert smoke_total == SMOKE_SIZE, f'SMOKE total {smoke_total} != {SMOKE_SIZE}'
    assert set(smoke_train_ids).issubset(set(train_ids_full)), 'smoke_train_ids must be a subset of the frozen train partition'
    assert set(smoke_validation_ids).issubset(set(validation_ids_full)), 'smoke_validation_ids must be a subset of the frozen validation partition'
    assert set(smoke_train_ids).isdisjoint(set(smoke_validation_ids)), 'smoke_train_ids and smoke_validation_ids must be disjoint'
    assert (set(smoke_train_ids) | set(smoke_validation_ids)).issubset(development_ids_full), (
        'every SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
    )

    print(f'SMOKE: train={len(smoke_train_ids):,} validation={len(smoke_validation_ids):,} total={smoke_total:,}')
    print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


In [ ]:
if RUN_T07_STAGE:
    def joint_ty_support(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support = joint_ty_support(smoke_train_ids, full_frame)
    smoke_validation_support = joint_ty_support(smoke_validation_ids, full_frame)
    smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support, **smoke_validation_support}.values())
    print('SMOKE train (T,Y) support:', smoke_train_support)
    print('SMOKE validation (T,Y) support:', smoke_validation_support)
    print('All joint-(T,Y) cells non-empty:', smoke_all_cells_nonempty)

    smoke_sample_ids_frame = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids, 'partition': 'train'}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids, 'partition': 'validation'}),
    ], ignore_index=True)
    smoke_sample_ids_sha256 = hashlib.sha256(
        smoke_sample_ids_frame.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
    ).hexdigest()

    write_bytes_new(RUN_ROOT, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame.to_parquet(index=False))
    write_json_new(RUN_ROOT, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID,
        'stage': 't07_smoke',
        'population': 'smoke_50000_rows',
        'smoke_size': SMOKE_SIZE,
        'smoke_seed': SMOKE_SEED,
        'train_quota': int(train_quota),
        'validation_quota': int(validation_quota),
        'train_count': int(len(smoke_train_ids)),
        'validation_count': int(len(smoke_validation_ids)),
        'total_count': int(smoke_total),
        'train_support': smoke_train_support,
        'validation_support': smoke_validation_support,
        'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256,
        'held_out_isolation_method': (
            'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids and '
            'smoke_validation_ids are proved subsets of train_ids_full/validation_ids_full (each obtained '
            'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
            'subset of train_ids_full union validation_ids_full -- held-out is never read via '
            'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
        ),
    })
    print('SMOKE sample identity persisted.')


In [ ]:
if RUN_T07_STAGE:
    smoke_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    smoke_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    transform = IdentityFeatureTransform()
    X_smoke_train = transform.fit_transform(smoke_train_frame)
    X_smoke_validation = transform.transform(smoke_validation_frame)
    assert_model_feature_contract(X_smoke_train.columns)
    assert_model_feature_contract(X_smoke_validation.columns)

    y_smoke_train = smoke_train_frame[PRIMARY_OUTCOME].astype('float64')
    y_smoke_validation = smoke_validation_frame[PRIMARY_OUTCOME].astype('float64')
    t_smoke_validation = smoke_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_smoke_validation_arr = y_smoke_validation.to_numpy()
    source_row_id_smoke_validation = smoke_validation_frame[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_train: {X_smoke_train.shape}, X_smoke_validation: {X_smoke_validation.shape}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_train.columns))


### 3.1 Response LightGBM baseline (SMOKE)

In [ ]:
if RUN_T07_STAGE:
    response_model = lgb_baseline.fit_binary_classifier(
        X_smoke_train, y_smoke_train, X_smoke_validation, y_smoke_validation,
    )
    response_probabilities_smoke = lgb_baseline.predict_probabilities(response_model, X_smoke_validation)

    assert np.isfinite(response_probabilities_smoke).all()
    assert (response_probabilities_smoke >= 0).all() and (response_probabilities_smoke <= 1).all()
    assert len(response_probabilities_smoke) == len(smoke_validation_ids)
    assert set(source_row_id_smoke_validation) == set(smoke_validation_ids)

    print(f'Response fit complete. best_iteration={response_model.best_iteration}, config_hash={response_model.config_hash[:16]}...')
    print('Predictions bounded/finite/aligned: OK')


### 3.2 Random reference (SMOKE) -- via the public T06 interface only

In [ ]:
if RUN_T07_STAGE:
    random_scores_smoke = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
    random_ranking_smoke = metrics.evaluate_ranking(
        random_scores_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    random_reference_distribution_smoke = metrics.random_ranking_reference_distribution(
        t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    assert len(random_reference_distribution_smoke) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

    response_ranking_smoke = metrics.evaluate_ranking(
        response_probabilities_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    response_diag_smoke = metrics.response_diagnostics(response_probabilities_smoke, y_smoke_validation_arr)
    ate_smoke = metrics.compute_ate(t_smoke_validation, y_smoke_validation_arr)

    print(f'Random: 1 illustrative draw + {len(random_reference_distribution_smoke)} reference draws computed via the public T06 interface.')
    print(f'Response ranking qini_above_random (SMOKE, non-substantive): {response_ranking_smoke.qini_above_random:.6f}')


### 3.3 SMOKE artifacts

In [ ]:
if RUN_T07_STAGE:
    def _rows_with_run_context(rows, population='smoke_50000_rows'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID)
            row.setdefault('stage', 't07_smoke')
            row.setdefault('population', population)
            yield row


    ate_summary_rows = list(_rows_with_run_context([{'method': 'assigned_arm', **ate_smoke.__dict__}]))
    response_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))

    uplift_at_k_rows = []
    for label in metrics.RANKING_K_LABELS:
        uplift_at_k_rows.append({'method': 'random', 'k': label, 'uplift': random_ranking_smoke.uplift_at_k[label],
                                  'incremental_conversions': random_ranking_smoke.incremental_conversions_at_k[label],
                                  'status': random_ranking_smoke.top_k_status[label]})
        uplift_at_k_rows.append({'method': 'response', 'k': label, 'uplift': response_ranking_smoke.uplift_at_k[label],
                                  'incremental_conversions': response_ranking_smoke.incremental_conversions_at_k[label],
                                  'status': response_ranking_smoke.top_k_status[label]})
    uplift_at_k_rows = list(_rows_with_run_context(uplift_at_k_rows))

    model_summary_rows = []
    for name, result in (('random', random_ranking_smoke), ('response', response_ranking_smoke)):
        model_summary_rows.append({
            'ranking_method': name,
            'qini_area': result.qini_area,
            'theoretical_random_qini_area': result.theoretical_random_qini_area,
            'qini_above_random': result.qini_above_random,
            'qini_above_random_permutation': random_ranking_smoke.qini_above_random,
            'uplift_at_10pct': result.uplift_at_k['10pct'],
            'uplift_at_20pct': result.uplift_at_k['20pct'],
            'uplift_at_30pct': result.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
        })
    model_summary_rows = list(_rows_with_run_context(model_summary_rows))

    random_deciles_rows = list(_rows_with_run_context(random_ranking_smoke.decile_table.to_dict('records')))
    response_deciles_rows = list(_rows_with_run_context(response_ranking_smoke.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/ate_summary.csv', ate_summary_rows),
        ('tables/response_diagnostics.csv', response_diagnostics_rows),
        ('tables/random_deciles.csv', random_deciles_rows),
        ('tables/response_deciles.csv', response_deciles_rows),
        ('tables/uplift_at_k.csv', uplift_at_k_rows),
        ('tables/model_summary.csv', model_summary_rows),
    ):
        write_text_new(RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    print('SMOKE tables written.')


In [ ]:
if RUN_T07_STAGE:
    response_predictions_frame = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation,
        'response_probability': response_probabilities_smoke,
    })
    random_scores_frame = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation,
        'random_score': random_scores_smoke,
    })
    response_predictions_bytes = response_predictions_frame.to_parquet(index=False)
    random_scores_bytes = random_scores_frame.to_parquet(index=False)

    write_bytes_new(RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes)
    write_bytes_new(RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes)

    response_model_text = response_model.booster.model_to_string()
    write_text_new(RUN_ROOT, 'models/response_model.txt', response_model_text)

    random_baseline_summary_rows = list(_rows_with_run_context([{
        'theoretical_random_qini_area': random_ranking_smoke.theoretical_random_qini_area,
        'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
        'illustrative_draw_qini_area': random_ranking_smoke.qini_area,
        'illustrative_draw_qini_above_random': random_ranking_smoke.qini_above_random,
    }]))
    write_text_new(RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows).to_csv(index=False, lineterminator='\n'))

    random_baseline_draws_rows = list(_rows_with_run_context(random_reference_distribution_smoke.to_dict('records')))
    write_text_new(RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows).to_csv(index=False, lineterminator='\n'))

    model_probability_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))
    write_text_new(RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows).to_csv(index=False, lineterminator='\n'))

    definitions = metrics.metric_definitions()
    definitions_sha256 = hashlib.sha256(json.dumps(definitions, sort_keys=True).encode()).hexdigest()

    def _flatten(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)

    definitions_rows = list(_rows_with_run_context(
        [{'field': k, 'value': _flatten(v)} for k, v in definitions.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256}]
    ))
    write_text_new(RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows).to_csv(index=False, lineterminator='\n'))

    response_predictions_sha256 = hashlib.sha256(response_predictions_bytes).hexdigest()
    random_scores_sha256 = hashlib.sha256(random_scores_bytes).hexdigest()
    print('SMOKE audit/model/prediction artifacts written.')


### 3.4 SMOKE reproducibility check (T07.7, SMOKE scope)

In [ ]:
if RUN_T07_STAGE:
    reloaded_booster = lgb.Booster(model_str=response_model_text)
    reloaded_config_hash = lgb_baseline.config_hash()
    config_hash_matches = reloaded_config_hash == response_model.config_hash

    X_smoke_validation_rebuilt = transform.transform(smoke_validation_frame)
    reloaded_probabilities = np.asarray(reloaded_booster.predict(X_smoke_validation_rebuilt, num_iteration=response_model.best_iteration), dtype=np.float64)
    prediction_reload_matches = np.allclose(reloaded_probabilities, response_probabilities_smoke, rtol=1e-6, atol=1e-8)

    row_identity_matches = set(source_row_id_smoke_validation) == set(smoke_validation_ids)

    random_scores_regenerated = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
    random_scores_exact_match = np.array_equal(random_scores_regenerated, random_scores_smoke)

    reload_verification = {
        'config_hash_matches': bool(config_hash_matches),
        'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches),
        'row_identity_matches': bool(row_identity_matches),
        'random_scores_exact_match': bool(random_scores_exact_match),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(reload_verification)
    assert all(reload_verification[k] for k in ('config_hash_matches', 'prediction_reload_matches_within_tolerance', 'row_identity_matches', 'random_scores_exact_match'))


In [ ]:
if RUN_T07_STAGE:
    smoke_wall_seconds = __import__('time').perf_counter() - smoke_wall_start
    smoke_peak_rss_bytes = process.memory_info().rss

    write_json_new(RUN_ROOT, 'audit/environment.json', {
        'run_id': RUN_ID,
        'created_at_utc': smoke_started.isoformat(),
        'git_head': git_head,
        'git_dirty': git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT, 'audit/run_config.json', {
        'run_id': RUN_ID,
        'created_at_utc': smoke_started.isoformat(),
        'stage': 't07_smoke',
        'population': 'smoke_50000_rows',
        'git_head': git_head,
        'git_dirty': git_dirty,
        'real_or_held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES, 'smoke_size': SMOKE_SIZE, 'smoke_seed': SMOKE_SEED},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_config_hash': response_model.config_hash,
        'lightgbm_best_iteration': response_model.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'response_predictions_sha256': response_predictions_sha256,
        'random_scores_sha256': random_scores_sha256,
        'definitions_sha256': definitions_sha256,
        'resource_evidence': {
            'wall_seconds': smoke_wall_seconds,
            'baseline_rss_bytes': notebook_baseline_rss_bytes,
            'peak_rss_bytes': smoke_peak_rss_bytes,
            'peak_rss_delta_bytes': smoke_peak_rss_bytes - notebook_baseline_rss_bytes,
        },
        'reload_verification': reload_verification,
    })

    print(f'SMOKE resource evidence: wall_seconds={smoke_wall_seconds:.2f}, peak_rss_delta_bytes={smoke_peak_rss_bytes - notebook_baseline_rss_bytes:,}')


In [ ]:
if RUN_T07_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT,
        run_id=RUN_ID,
        final_status='COMPLETED_T07_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t07_smoke',
        population='smoke_50000_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
        ],
    )
    print(f'SMOKE run finalized: {RUN_ID}')

    immutable_write_refused = False
    try:
        write_json_new(RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused = True
    except Exception:
        immutable_write_refused = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused}')
    assert immutable_write_refused


### 3.5 SMOKE summary

In [ ]:
if RUN_T07_STAGE:
    smoke_summary = {
        'run_id': RUN_ID,
        'smoke_total': int(smoke_total),
        'smoke_train_count': int(len(smoke_train_ids)),
        'smoke_validation_count': int(len(smoke_validation_ids)),
        'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
        'lightgbm_version': lgb.__version__,
        'lightgbm_config_hash': response_model.config_hash,
        'best_iteration': response_model.best_iteration,
        'random_reference_draws': len(random_reference_distribution_smoke),
        'reload_verification': reload_verification,
        'resource_wall_seconds': smoke_wall_seconds,
        'immutable_write_refused': immutable_write_refused,
    }
    print(json.dumps(smoke_summary, indent=2))


#### Reproducibility note (technical): T07 FULL execution

SMOKE passed every correctness/artifact-mechanism check in the section above.
Under D30, T07's approved path is `SMOKE -> FULL` with `resource_gates = 0`
(recorded and justified in `configs/t07_baselines.json`), so FULL is the next
and only remaining stage. FULL fits on the **complete** frozen train partition,
early-stops on the **complete** frozen validation partition, and is the
authoritative development run whose results the sections below report.
Held-out remains completely sealed throughout -- this section never reads
`SplitDataset.held_out_ids()` or any held-out row.

In [ ]:
if RUN_T07_STAGE:
    from IPython.display import Markdown, display
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    full_started = datetime.now(timezone.utc)
    full_wall_start = __import__('time').perf_counter()
    full_baseline_rss_bytes = process.memory_info().rss

    FULL_RUN_ID = full_started.strftime('t07_full_%Y%m%dT%H%M%SZ_%f')
    FULL_RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / FULL_RUN_ID
    FULL_RUN_ROOT.mkdir(parents=True, exist_ok=False)

    try:
        full_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        full_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        full_git_head, full_git_dirty = None, None

    print(f'FULL_RUN_ID = {FULL_RUN_ID}')


In [ ]:
if RUN_T07_STAGE:
    full_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    full_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(full_train_frame) == len(train_ids_full)
    assert len(full_validation_frame) == len(validation_ids_full)

    transform_full = IdentityFeatureTransform()
    X_full_train = transform_full.fit_transform(full_train_frame)
    X_full_validation = transform_full.transform(full_validation_frame)
    assert_model_feature_contract(X_full_train.columns)
    assert_model_feature_contract(X_full_validation.columns)

    y_full_train = full_train_frame[PRIMARY_OUTCOME].astype('float64')
    y_full_validation = full_validation_frame[PRIMARY_OUTCOME].astype('float64')
    t_full_validation = full_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_full_validation_arr = y_full_validation.to_numpy()
    source_row_id_full_validation = full_validation_frame[SOURCE_ROW_ID].to_numpy()

    print(f'FULL: X_train={X_full_train.shape}, X_validation={X_full_validation.shape}')


## 4. Random Reference

In [ ]:
if RUN_T07_STAGE:
    random_scores_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
    random_ranking_full = metrics.evaluate_ranking(
        random_scores_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    random_reference_distribution_full = metrics.random_ranking_reference_distribution(
        t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    assert len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

    display(Markdown(
        f"The theoretical expected-random Qini area on the frozen validation population is "
        f"**{random_ranking_full.theoretical_random_qini_area:.4f}**. One deterministic seed-42 illustrative "
        f"random ranking realizes `qini_area = {random_ranking_full.qini_area:.4f}` "
        f"(`qini_above_random = {random_ranking_full.qini_above_random:.4f}`), and the frozen "
        f"{len(random_reference_distribution_full)}-draw random-ranking reference distribution has "
        f"`qini_above_random` mean **{random_reference_distribution_full['qini_above_random'].mean():.4f}** "
        f"(std {random_reference_distribution_full['qini_above_random'].std():.4f}), consistent with a "
        f"no-skill ranking centered near zero. This distribution is secondary empirical context; the "
        f"theoretical line remains the primary random reference (D11)."
    ))


## 5. Response LightGBM Baseline

In [ ]:
if RUN_T07_STAGE:
    response_model_full = lgb_baseline.fit_binary_classifier(
        X_full_train, y_full_train, X_full_validation, y_full_validation,
    )
    response_probabilities_full = lgb_baseline.predict_probabilities(response_model_full, X_full_validation)

    assert np.isfinite(response_probabilities_full).all()
    assert (response_probabilities_full >= 0).all() and (response_probabilities_full <= 1).all()
    assert set(source_row_id_full_validation) == set(validation_ids_full)

    response_diag_full = metrics.response_diagnostics(response_probabilities_full, y_full_validation_arr)
    response_ranking_full = metrics.evaluate_ranking(
        response_probabilities_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    ate_full = metrics.compute_ate(t_full_validation, y_full_validation_arr)

    print(f'Response fit complete. best_iteration={response_model_full.best_iteration}, config_hash={response_model_full.config_hash[:16]}...')


In [ ]:
if RUN_T07_STAGE:
    display(Markdown(
        f"**Response diagnostics (factual-outcome prediction quality -- diagnostic only, D27; never a causal "
        f"ranking claim):** ROC-AUC = {response_diag_full.roc_auc:.4f}, average precision = "
        f"{response_diag_full.average_precision:.4f}, log loss = {response_diag_full.log_loss:.4f}.\n\n"
        f"**Response as an uplift ranking** (the same probability scores, evaluated as a policy via the "
        f"identical T06 interface used for every method): `qini_area = {response_ranking_full.qini_area:.4f}`, "
        f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` "
        f"(theoretical random = {response_ranking_full.theoretical_random_qini_area:.4f}). "
        f"`uplift@10% = {response_ranking_full.uplift_at_k['10pct']}`, "
        f"`uplift@20% = {response_ranking_full.uplift_at_k['20pct']}`, "
        f"`uplift@30% = {response_ranking_full.uplift_at_k['30pct']}` "
        f"(status: {response_ranking_full.top_k_status['10pct']}/{response_ranking_full.top_k_status['20pct']}/{response_ranking_full.top_k_status['30pct']}).\n\n"
        f"**Assigned-arm ATE** (population aggregate, not a ranking estimator; D24 methodology note): "
        f"{ate_full.ate:.4f} ({ate_full.ate_percentage_points:.2f} pp), 95% CI "
        f"[{ate_full.ci_95_low:.4f}, {ate_full.ci_95_high:.4f}]."
    ))


### FULL artifacts

In [ ]:
if RUN_T07_STAGE:
    def _full_rows_with_run_context(rows, population='full_development_population'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', FULL_RUN_ID)
            row.setdefault('stage', 't07_full')
            row.setdefault('population', population)
            yield row


    ate_summary_rows_full = list(_full_rows_with_run_context([{'method': 'assigned_arm', **ate_full.__dict__}]))
    response_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))

    uplift_at_k_rows_full = []
    for label in metrics.RANKING_K_LABELS:
        uplift_at_k_rows_full.append({'method': 'random', 'k': label, 'uplift': random_ranking_full.uplift_at_k[label],
                                       'incremental_conversions': random_ranking_full.incremental_conversions_at_k[label],
                                       'status': random_ranking_full.top_k_status[label]})
        uplift_at_k_rows_full.append({'method': 'response', 'k': label, 'uplift': response_ranking_full.uplift_at_k[label],
                                       'incremental_conversions': response_ranking_full.incremental_conversions_at_k[label],
                                       'status': response_ranking_full.top_k_status[label]})
    uplift_at_k_rows_full = list(_full_rows_with_run_context(uplift_at_k_rows_full))

    model_summary_rows_full = []
    for name, result in (('random', random_ranking_full), ('response', response_ranking_full)):
        model_summary_rows_full.append({
            'ranking_method': name,
            'qini_area': result.qini_area,
            'theoretical_random_qini_area': result.theoretical_random_qini_area,
            'qini_above_random': result.qini_above_random,
            'qini_above_random_permutation': random_ranking_full.qini_above_random,
            'uplift_at_10pct': result.uplift_at_k['10pct'],
            'uplift_at_20pct': result.uplift_at_k['20pct'],
            'uplift_at_30pct': result.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
        })
    model_summary_rows_full = list(_full_rows_with_run_context(model_summary_rows_full))

    random_deciles_rows_full = list(_full_rows_with_run_context(random_ranking_full.decile_table.to_dict('records')))
    response_deciles_rows_full = list(_full_rows_with_run_context(response_ranking_full.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/ate_summary.csv', ate_summary_rows_full),
        ('tables/response_diagnostics.csv', response_diagnostics_rows_full),
        ('tables/random_deciles.csv', random_deciles_rows_full),
        ('tables/response_deciles.csv', response_deciles_rows_full),
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full),
        ('tables/model_summary.csv', model_summary_rows_full),
    ):
        write_text_new(FULL_RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    print('FULL tables written.')


In [ ]:
if RUN_T07_STAGE:
    response_predictions_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation,
        'response_probability': response_probabilities_full,
    })
    random_scores_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation,
        'random_score': random_scores_full,
    })
    response_predictions_bytes_full = response_predictions_frame_full.to_parquet(index=False)
    random_scores_bytes_full = random_scores_frame_full.to_parquet(index=False)

    write_bytes_new(FULL_RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes_full)
    write_bytes_new(FULL_RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes_full)

    response_model_text_full = response_model_full.booster.model_to_string()
    write_text_new(FULL_RUN_ROOT, 'models/response_model.txt', response_model_text_full)

    random_baseline_summary_rows_full = list(_full_rows_with_run_context([{
        'theoretical_random_qini_area': random_ranking_full.theoretical_random_qini_area,
        'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
        'illustrative_draw_qini_area': random_ranking_full.qini_area,
        'illustrative_draw_qini_above_random': random_ranking_full.qini_above_random,
    }]))
    write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows_full).to_csv(index=False, lineterminator='\n'))

    random_baseline_draws_rows_full = list(_full_rows_with_run_context(random_reference_distribution_full.to_dict('records')))
    write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows_full).to_csv(index=False, lineterminator='\n'))

    model_probability_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))
    write_text_new(FULL_RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows_full).to_csv(index=False, lineterminator='\n'))

    definitions_full = metrics.metric_definitions()
    definitions_sha256_full = hashlib.sha256(json.dumps(definitions_full, sort_keys=True).encode()).hexdigest()
    definitions_rows_full = list(_full_rows_with_run_context(
        [{'field': k, 'value': _flatten(v)} for k, v in definitions_full.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_full}]
    ))
    write_text_new(FULL_RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full).to_csv(index=False, lineterminator='\n'))

    response_predictions_sha256_full = hashlib.sha256(response_predictions_bytes_full).hexdigest()
    random_scores_sha256_full = hashlib.sha256(random_scores_bytes_full).hexdigest()
    print('FULL audit/model/prediction artifacts written.')


In [ ]:
if RUN_T07_STAGE:
    import io


    def _cumulative_rate_curve(decile_table):
        ordered = decile_table.sort_values('decile')
        cum_n1 = ordered['n1'].cumsum()
        cum_n0 = ordered['n0'].cumsum()
        cum_y1 = ordered['y1'].cumsum()
        cum_y0 = ordered['y0'].cumsum()
        with np.errstate(divide='ignore', invalid='ignore'):
            cumulative_rate = (cum_y1 / cum_n1) - (cum_y0 / cum_n0)
        return ordered['decile'].to_numpy(), cumulative_rate.to_numpy()


    def _save_figure_bytes(fig, run_root, relative_path):
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
        plt.close(fig)
        write_bytes_new(run_root, relative_path, buffer.getvalue())


    fig, ax = plt.subplots(figsize=(7, 4))
    deciles = response_ranking_full.decile_table.sort_values('decile')['decile']
    ax.bar(deciles - 0.15, response_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Response')
    ax.bar(deciles + 0.15, random_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Random')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Decile (1 = highest-ranked)')
    ax.set_ylabel('Observed uplift (treated_rate - control_rate)')
    ax.set_title('Response vs. Random: observed uplift by decile')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/response_uplift_deciles.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    resp_d, resp_rate = _cumulative_rate_curve(response_ranking_full.decile_table)
    rand_d, rand_rate = _cumulative_rate_curve(random_ranking_full.decile_table)
    ax.plot(resp_d, resp_rate, marker='o', label='Response')
    ax.plot(rand_d, rand_rate, marker='o', label='Random')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Cumulative decile coverage')
    ax.set_ylabel('Cumulative uplift rate')
    ax.set_title('Cumulative uplift rate by coverage')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_rate.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
    ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
    ax.set_xlabel('Coverage')
    ax.set_ylabel('Qini gain (incremental conversions)')
    ax.set_title('Cumulative Qini gain by coverage')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_gain.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
    ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
    theoretical_x = np.array([0.0, 1.0])
    theoretical_y = theoretical_x * (response_ranking_full.qini_curve['qini_gain'].iloc[-1])
    ax.plot(theoretical_x, theoretical_y, linestyle='--', color='gray', label='Theoretical random line')
    ax.set_xlabel('Coverage')
    ax.set_ylabel('Qini gain')
    ax.set_title('Qini curve: Response vs. Random vs. theoretical random')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/qini_curve.png')

    print('FULL figures written.')


### FULL reproducibility check (T07.7)

In [ ]:
if RUN_T07_STAGE:
    reloaded_booster_full = lgb.Booster(model_str=response_model_text_full)
    reloaded_config_hash_full = lgb_baseline.config_hash()
    config_hash_matches_full = reloaded_config_hash_full == response_model_full.config_hash

    X_full_validation_rebuilt = transform_full.transform(full_validation_frame)
    reloaded_probabilities_full = np.asarray(
        reloaded_booster_full.predict(X_full_validation_rebuilt, num_iteration=response_model_full.best_iteration), dtype=np.float64
    )
    prediction_reload_matches_full = np.allclose(reloaded_probabilities_full, response_probabilities_full, rtol=1e-6, atol=1e-8)
    row_identity_matches_full = set(source_row_id_full_validation) == set(validation_ids_full)

    random_scores_regenerated_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
    random_scores_exact_match_full = np.array_equal(random_scores_regenerated_full, random_scores_full)

    # The frozen 200-draw distribution is not recomputed a second time to "prove" determinism here --
    # T06's own test suite (test_random_ranking_reference_is_deterministic_given_master_seed) already
    # covers that. This is a bounded self-consistency check of what was actually stored above.
    random_reference_self_consistent_full = (
        len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200
        and random_reference_distribution_full['draw_index'].nunique() == 200
    )

    reload_verification_full = {
        'config_hash_matches': bool(config_hash_matches_full),
        'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches_full),
        'row_identity_matches': bool(row_identity_matches_full),
        'random_scores_exact_match': bool(random_scores_exact_match_full),
        'random_reference_200_draws_self_consistent': bool(random_reference_self_consistent_full),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(reload_verification_full)
    assert all(reload_verification_full[k] for k in (
        'config_hash_matches', 'prediction_reload_matches_within_tolerance',
        'row_identity_matches', 'random_scores_exact_match', 'random_reference_200_draws_self_consistent',
    ))


In [ ]:
if RUN_T07_STAGE:
    full_wall_seconds = __import__('time').perf_counter() - full_wall_start
    full_peak_rss_bytes = process.memory_info().rss

    write_json_new(FULL_RUN_ROOT, 'audit/environment.json', {
        'run_id': FULL_RUN_ID,
        'created_at_utc': full_started.isoformat(),
        'git_head': full_git_head,
        'git_dirty': full_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(FULL_RUN_ROOT, 'audit/run_config.json', {
        'run_id': FULL_RUN_ID,
        'created_at_utc': full_started.isoformat(),
        'stage': 't07_full',
        'population': 'full_development_population',
        'git_head': full_git_head,
        'git_dirty': full_git_dirty,
        'real_or_held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES, 'preceding_smoke_run_id': RUN_ID},
        'train_count': int(len(train_ids_full)),
        'validation_count': int(len(validation_ids_full)),
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_config_hash': response_model_full.config_hash,
        'lightgbm_best_iteration': response_model_full.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'response_predictions_sha256': response_predictions_sha256_full,
        'random_scores_sha256': random_scores_sha256_full,
        'definitions_sha256': definitions_sha256_full,
        'resource_evidence': {
            'wall_seconds': full_wall_seconds,
            'baseline_rss_bytes': full_baseline_rss_bytes,
            'peak_rss_bytes': full_peak_rss_bytes,
            'peak_rss_delta_bytes': full_peak_rss_bytes - full_baseline_rss_bytes,
        },
        'reload_verification': reload_verification_full,
    })

    print(f'FULL resource evidence: wall_seconds={full_wall_seconds:.2f}, peak_rss_delta_bytes={full_peak_rss_bytes - full_baseline_rss_bytes:,}')


In [ ]:
if RUN_T07_STAGE:
    finalize_artifact_manifest(
        FULL_RUN_ROOT,
        run_id=FULL_RUN_ID,
        final_status='COMPLETED_T07_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t07_full',
        population='full_development_population',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{RUN_ID}', 'role': 'preceding_smoke_run', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'FULL run finalized: {FULL_RUN_ID}')

    immutable_write_refused_full = False
    try:
        write_json_new(FULL_RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_full = True
    except Exception:
        immutable_write_refused_full = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_full}')
    assert immutable_write_refused_full


In [ ]:
if RUN_T07_STAGE:
    full_summary = {
        'run_id': FULL_RUN_ID,
        'preceding_smoke_run_id': RUN_ID,
        'train_count': int(len(train_ids_full)),
        'validation_count': int(len(validation_ids_full)),
        'lightgbm_version': lgb.__version__,
        'lightgbm_config_hash': response_model_full.config_hash,
        'best_iteration': response_model_full.best_iteration,
        'response_diagnostics': response_diag_full.__dict__,
        'random_reference_draws': len(random_reference_distribution_full),
        'random_theoretical_qini_area': random_ranking_full.theoretical_random_qini_area,
        'random_illustrative_qini_above_random': random_ranking_full.qini_above_random,
        'response_qini_area': response_ranking_full.qini_area,
        'response_qini_above_random': response_ranking_full.qini_above_random,
        'response_uplift_at_k': response_ranking_full.uplift_at_k,
        'response_incremental_conversions_at_k': response_ranking_full.incremental_conversions_at_k,
        'ate': ate_full.__dict__,
        'reload_verification': reload_verification_full,
        'resource_wall_seconds': full_wall_seconds,
        'immutable_write_refused': immutable_write_refused_full,
    }
    print(json.dumps(full_summary, indent=2, default=str))


## 6. Interpretation

In [ ]:
if RUN_T07_STAGE:
    display(Markdown(
        f"Response's own diagnostics (ROC-AUC {response_diag_full.roc_auc:.4f}) describe how well "
        f"`P(Y=1|X)` predicts conversion -- a factual-outcome quality measure, never a causal ranking claim "
        f"(D12, D27). Whether that translates into a *useful uplift ranking* is a separate question, answered "
        f"only by routing the same scores through the frozen T06 Qini/uplift-at-K interface: Response reaches "
        f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` against a theoretical random "
        f"reference of {response_ranking_full.theoretical_random_qini_area:.4f}, and the illustrative seeded "
        f"random ranking reaches `qini_above_random = {random_ranking_full.qini_above_random:.4f}` of its own "
        f"-- both are read directly off the same metric interface with no separate formula for either method.\n\n"
        f"{'Response ranks above the random reference on this development population.' if response_ranking_full.qini_above_random > random_ranking_full.qini_above_random else 'Response does not clearly outrank the random reference on this development population -- a strong response model is not automatically a good uplift ranker, and that is a legitimate, reportable finding here, not an error.'} "
        f"This is a development-only observation on the validation partition; it is not a held-out claim, and "
        f"predicted uplift is not a true individual treatment effect. T-Learner, X-Learner, and Causal Forest "
        f"(T08-T11) extend this same comparison with genuinely causal estimators."
    ))


In [ ]:
RUN_T08_SMOKE_STAGE = False  # T08 SMOKE is already accepted.
T08_SMOKE_RUN_ID_ACCEPTED = 't08_smoke_20260818T154813Z_381508'
T08_SMOKE_ERRATUM_RUN_ID_ACCEPTED = 't08_smoke_audit_erratum_20260818T155806Z_642086'

if not RUN_T08_SMOKE_STAGE:
    t08_smoke_root = REPO_ROOT / 'outputs' / 'runs' / T08_SMOKE_RUN_ID_ACCEPTED
    t08_smoke_manifest_path = t08_smoke_root / 'audit' / 'artifact_manifest.json'
    if not t08_smoke_manifest_path.is_file():
        raise RuntimeError(
            f'RUN_T08_SMOKE_STAGE is False, but the accepted T08 SMOKE run evidence was not found at '
            f'{t08_smoke_root}. Refusing to silently set RUN_T08_SMOKE_STAGE = True and refit the SMOKE '
            f'mu1/mu0 models -- either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T08_SMOKE_RUN_ID_ACCEPTED}/), or explicitly set RUN_T08_SMOKE_STAGE = True '
            f'in this cell to opt into reproducing T08 SMOKE from scratch.'
        )
    t08_smoke_manifest = json.loads(t08_smoke_manifest_path.read_text(encoding='utf-8'))
    t08_smoke_artifact_hashes = {a['path']: a['sha256'] for a in t08_smoke_manifest['artifacts']}
    t08_smoke_model_summary_path = t08_smoke_root / 'tables' / 'model_summary.csv'
    t08_smoke_model_summary_actual_sha256 = hashlib.sha256(t08_smoke_model_summary_path.read_bytes()).hexdigest()
    t08_smoke_model_summary_expected_sha256 = t08_smoke_artifact_hashes.get('tables/model_summary.csv')
    if t08_smoke_model_summary_expected_sha256 is None or t08_smoke_model_summary_actual_sha256 != t08_smoke_model_summary_expected_sha256:
        raise RuntimeError(
            f'T08 SMOKE tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t08_smoke_model_summary_expected_sha256}, actual {t08_smoke_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    print('Using hash-verified frozen development results (T08 SMOKE).')
    print(f'  Reproducibility: T08 SMOKE run_id={T08_SMOKE_RUN_ID_ACCEPTED}')
else:
    print('RUN_T08_SMOKE_STAGE = True: T08 SMOKE will be recomputed from scratch below.')


#### Reproducibility note (technical): T08 SMOKE verification

T08 (T-Learner) uses the D30 path declared in `configs/t08_tlearner.json`:
`SMOKE -> FULL`, `resource_gates = 0`, `smoke_size = 200,000`. This is larger
than T07's SMOKE (50,000) because T-Learner's two arm-specific fits each see
only their own arm's slice of any draw -- the binding constraint is the
control arm's converter cell (`T=0,Y=1`), which is far rarer, per-arm, than
what a single pooled model like T07's Response needed. SMOKE here proves
correctness and artifact mechanics only, for **both** surfaces; it is not a
performance estimate, and this stage's SMOKE population is deliberately not
used for any performance claim.


*(This section -- T08 SMOKE -- is already accepted. It is skipped by default on Run All; see `RUN_T08_SMOKE_STAGE` immediately above. Set it to `True` only to deliberately reproduce T08 SMOKE from scratch.)*

In [ ]:
if RUN_T08_SMOKE_STAGE:
    from src.tlearner import (
        TLearnerContractError, assert_aligned_predictions, compute_tau,
        partition_by_arm, reconcile_reloaded_tau,
    )
    import src.tlearner as tlearner_module

    t08_config = json.loads((REPO_ROOT / 'configs' / 't08_tlearner.json').read_text(encoding='utf-8'))

    SMOKE_SIZE_T08 = t08_config['scale_gating']['smoke_size']
    SMOKE_SEED_T08 = t08_config['scale_gating']['smoke_seed']
    RESOURCE_GATES_T08 = t08_config['scale_gating']['resource_gates']

    t08_smoke_started = datetime.now(timezone.utc)
    t08_smoke_wall_start = __import__('time').perf_counter()
    t08_smoke_baseline_rss_bytes = process.memory_info().rss
    t08_smoke_baseline_peak_wset = getattr(process.memory_info(), 'peak_wset', None)

    RUN_ID_T08_SMOKE = t08_smoke_started.strftime('t08_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T08_SMOKE = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T08_SMOKE
    RUN_ROOT_T08_SMOKE.mkdir(parents=True, exist_ok=False)

    try:
        t08_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t08_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t08_git_head, t08_git_dirty = None, None

    print(f'RUN_ID_T08_SMOKE = {RUN_ID_T08_SMOKE}')
    print(f'resource_gates = {RESOURCE_GATES_T08}, smoke_size = {SMOKE_SIZE_T08}, seed = {SMOKE_SEED_T08}')


In [ ]:
if RUN_T08_SMOKE_STAGE:
    def _joint_strata_t08(frame, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_partition_t08(partition_ids, quota, full_frame, seed):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata_t08(subset, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    p_train_t08 = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
    train_quota_t08 = round(SMOKE_SIZE_T08 * p_train_t08)
    validation_quota_t08 = SMOKE_SIZE_T08 - train_quota_t08

    smoke_train_ids_t08 = smoke_sample_partition_t08(train_ids_full, train_quota_t08, full_frame, SMOKE_SEED_T08)
    smoke_validation_ids_t08 = smoke_sample_partition_t08(validation_ids_full, validation_quota_t08, full_frame, SMOKE_SEED_T08)

    smoke_total_t08 = len(smoke_train_ids_t08) + len(smoke_validation_ids_t08)
    assert smoke_total_t08 == SMOKE_SIZE_T08, f'T08 SMOKE total {smoke_total_t08} != {SMOKE_SIZE_T08}'
    assert set(smoke_train_ids_t08).issubset(set(train_ids_full)), 'smoke_train_ids_t08 must be a subset of the frozen train partition'
    assert set(smoke_validation_ids_t08).issubset(set(validation_ids_full)), 'smoke_validation_ids_t08 must be a subset of the frozen validation partition'
    assert set(smoke_train_ids_t08).isdisjoint(set(smoke_validation_ids_t08))
    development_ids_full_t08 = set(train_ids_full) | set(validation_ids_full)
    assert (set(smoke_train_ids_t08) | set(smoke_validation_ids_t08)).issubset(development_ids_full_t08), (
        'every T08 SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
    )
    # Held-out isolation proved by construction only, same as T07's corrected SMOKE -- never read
    # via split_membership.csv's held_out label or SplitDataset.held_out_ids().

    print(f'T08 SMOKE: train={len(smoke_train_ids_t08):,} validation={len(smoke_validation_ids_t08):,} total={smoke_total_t08:,}')
    print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


In [ ]:
if RUN_T08_SMOKE_STAGE:
    def joint_ty_support_t08(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support_t08 = joint_ty_support_t08(smoke_train_ids_t08, full_frame)
    smoke_validation_support_t08 = joint_ty_support_t08(smoke_validation_ids_t08, full_frame)
    print('T08 SMOKE train (T,Y) support:', smoke_train_support_t08)
    print('T08 SMOKE validation (T,Y) support:', smoke_validation_support_t08)

    t08_smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support_t08, **smoke_validation_support_t08}.values())
    print('All 8 partition x (T,Y) cells non-empty:', t08_smoke_all_cells_nonempty)
    if not t08_smoke_all_cells_nonempty:
        raise TLearnerContractError(
            'T08 SMOKE support is degenerate: at least one train/validation x (T,Y) cell is empty at '
            f'smoke_size={SMOKE_SIZE_T08}. SMOKE purpose is engineering/correctness validation, not '
            'performance estimation -- failing closed rather than adjusting the sample after seeing this.'
        )


In [ ]:
if RUN_T08_SMOKE_STAGE:
    smoke_sample_ids_frame_t08 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids_t08, 'partition': 'train'}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids_t08, 'partition': 'validation'}),
    ], ignore_index=True)
    smoke_sample_ids_sha256_t08 = hashlib.sha256(
        smoke_sample_ids_frame_t08.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
    ).hexdigest()

    write_bytes_new(RUN_ROOT_T08_SMOKE, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame_t08.to_parquet(index=False))
    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'stage': 't08_smoke',
        'population': 'smoke_200000_rows',
        'smoke_size': SMOKE_SIZE_T08,
        'smoke_seed': SMOKE_SEED_T08,
        'train_quota': int(train_quota_t08),
        'validation_quota': int(validation_quota_t08),
        'train_count': int(len(smoke_train_ids_t08)),
        'validation_count': int(len(smoke_validation_ids_t08)),
        'total_count': int(smoke_total_t08),
        'train_support': smoke_train_support_t08,
        'validation_support': smoke_validation_support_t08,
        'all_partition_ty_cells_nonempty': bool(t08_smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256_t08,
        'held_out_isolation_method': (
            'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids_t08 and '
            'smoke_validation_ids_t08 are proved subsets of train_ids_full/validation_ids_full (each obtained '
            'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
            'subset of train_ids_full union validation_ids_full -- held-out is never read via '
            'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
        ),
    })
    print('T08 SMOKE sample identity persisted.')


In [ ]:
if RUN_T08_SMOKE_STAGE:
    smoke_train_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids_t08)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    smoke_validation_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids_t08)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    transform_t08_smoke = IdentityFeatureTransform()
    X_smoke_train_t08 = transform_t08_smoke.fit_transform(smoke_train_frame_t08)
    X_smoke_validation_t08 = transform_t08_smoke.transform(smoke_validation_frame_t08)
    assert_model_feature_contract(X_smoke_train_t08.columns)
    assert_model_feature_contract(X_smoke_validation_t08.columns)

    y_smoke_train_t08 = smoke_train_frame_t08[PRIMARY_OUTCOME].astype('float64')
    y_smoke_validation_t08 = smoke_validation_frame_t08[PRIMARY_OUTCOME].astype('float64')
    t_smoke_train_t08 = smoke_train_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_smoke_validation_t08 = smoke_validation_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_smoke_validation_arr_t08 = y_smoke_validation_t08.to_numpy()
    source_row_id_smoke_validation_t08 = smoke_validation_frame_t08[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_train_t08: {X_smoke_train_t08.shape}, X_smoke_validation_t08: {X_smoke_validation_t08.shape}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_train_t08.columns))


### T08.1 Arm partitioning

In [ ]:
if RUN_T08_SMOKE_STAGE:
    (X_train_treated_t08,), (X_train_control_t08,) = partition_by_arm(t_smoke_train_t08, X_smoke_train_t08.to_numpy())
    X_train_treated_t08 = pd.DataFrame(X_train_treated_t08, columns=X_smoke_train_t08.columns)
    X_train_control_t08 = pd.DataFrame(X_train_control_t08, columns=X_smoke_train_t08.columns)
    (y_train_treated_t08,), (y_train_control_t08,) = partition_by_arm(t_smoke_train_t08, y_smoke_train_t08.to_numpy())

    (X_val_treated_t08,), (X_val_control_t08,) = partition_by_arm(t_smoke_validation_t08, X_smoke_validation_t08.to_numpy())
    X_val_treated_t08 = pd.DataFrame(X_val_treated_t08, columns=X_smoke_validation_t08.columns)
    X_val_control_t08 = pd.DataFrame(X_val_control_t08, columns=X_smoke_validation_t08.columns)
    (y_val_treated_t08,), (y_val_control_t08,) = partition_by_arm(t_smoke_validation_t08, y_smoke_validation_arr_t08)

    assert len(X_train_treated_t08) == smoke_train_support_t08['T=1,Y=0'] + smoke_train_support_t08['T=1,Y=1']
    assert len(X_train_control_t08) == smoke_train_support_t08['T=0,Y=0'] + smoke_train_support_t08['T=0,Y=1']
    print(f'mu1 train rows: {len(X_train_treated_t08):,} (arm-specific early-stopping validation: {len(X_val_treated_t08):,})')
    print(f'mu0 train rows: {len(X_train_control_t08):,} (arm-specific early-stopping validation: {len(X_val_control_t08):,})')


### T08.2 Fit mu1 and mu0 (identical frozen config)

In [ ]:
if RUN_T08_SMOKE_STAGE:
    mu1_model_smoke = lgb_baseline.fit_binary_classifier(
        X_train_treated_t08, y_train_treated_t08, X_val_treated_t08, y_val_treated_t08,
    )
    mu0_model_smoke = lgb_baseline.fit_binary_classifier(
        X_train_control_t08, y_train_control_t08, X_val_control_t08, y_val_control_t08,
    )
    assert mu1_model_smoke.config_hash == mu0_model_smoke.config_hash, 'mu1/mu0 must share the identical frozen config'
    print(f'mu1: best_iteration={mu1_model_smoke.best_iteration}, config_hash={mu1_model_smoke.config_hash[:16]}...')
    print(f'mu0: best_iteration={mu0_model_smoke.best_iteration}, config_hash={mu0_model_smoke.config_hash[:16]}...')


### T08.3 Score the full common SMOKE validation cohort

In [ ]:
if RUN_T08_SMOKE_STAGE:
    mu1_hat_smoke = lgb_baseline.predict_probabilities(mu1_model_smoke, X_smoke_validation_t08)
    mu0_hat_smoke = lgb_baseline.predict_probabilities(mu0_model_smoke, X_smoke_validation_t08)

    assert_aligned_predictions(source_row_id_smoke_validation_t08, source_row_id_smoke_validation_t08, source_row_id_smoke_validation_t08)
    assert len(mu1_hat_smoke) == len(smoke_validation_ids_t08) == len(mu0_hat_smoke)
    print('Both surfaces scored the full SMOKE validation cohort:', len(mu1_hat_smoke), 'rows')


### T08.4 tau_hat = mu1_hat - mu0_hat

In [ ]:
if RUN_T08_SMOKE_STAGE:
    tau_hat_smoke = compute_tau(mu1_hat_smoke, mu0_hat_smoke)
    np.testing.assert_array_equal(tau_hat_smoke, mu1_hat_smoke - mu0_hat_smoke)  # exact, elementwise, asserted not just constructed

    positive_count = int((tau_hat_smoke > 0).sum())
    negative_count = int((tau_hat_smoke < 0).sum())
    zero_count = int((tau_hat_smoke == 0).sum())
    assert positive_count == int(((mu1_hat_smoke - mu0_hat_smoke) > 0).sum())  # sign orientation: higher mu1 -> positive tau
    print(f'tau_hat: positive={positive_count:,} negative={negative_count:,} zero={zero_count:,}')


### T08.5 Evaluate via T06 only

In [ ]:
if RUN_T08_SMOKE_STAGE:
    tlearner_ranking_smoke = metrics.evaluate_ranking(
        tau_hat_smoke, t_smoke_validation_t08, y_smoke_validation_arr_t08, source_row_id_smoke_validation_t08,
    )
    print(f'T-Learner SMOKE qini_above_random (non-substantive): {tlearner_ranking_smoke.qini_above_random:.6f}')
    # metrics.random_ranking_reference_distribution() and metrics.seeded_random_scores() are never called
    # here -- T08 does not recompute T07's random reference or Response diagnostics.


### T08.6 SMOKE artifacts

In [ ]:
if RUN_T08_SMOKE_STAGE:
    def _rows_with_run_context_t08(rows, population='smoke_200000_rows'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T08_SMOKE)
            row.setdefault('stage', 't08_smoke')
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_t08 = list(_rows_with_run_context_t08([
        {'method': 'tlearner', 'k': label, 'uplift': tlearner_ranking_smoke.uplift_at_k[label],
         'incremental_conversions': tlearner_ranking_smoke.incremental_conversions_at_k[label],
         'status': tlearner_ranking_smoke.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_t08 = list(_rows_with_run_context_t08([{
        'ranking_method': 'tlearner',
        'qini_area': tlearner_ranking_smoke.qini_area,
        'theoretical_random_qini_area': tlearner_ranking_smoke.theoretical_random_qini_area,
        'qini_above_random': tlearner_ranking_smoke.qini_above_random,
        'uplift_at_10pct': tlearner_ranking_smoke.uplift_at_k['10pct'],
        'uplift_at_20pct': tlearner_ranking_smoke.uplift_at_k['20pct'],
        'uplift_at_30pct': tlearner_ranking_smoke.uplift_at_k['30pct'],
        'incremental_conversions_at_10pct': tlearner_ranking_smoke.incremental_conversions_at_k['10pct'],
        'incremental_conversions_at_20pct': tlearner_ranking_smoke.incremental_conversions_at_k['20pct'],
        'incremental_conversions_at_30pct': tlearner_ranking_smoke.incremental_conversions_at_k['30pct'],
    }]))
    tlearner_deciles_rows_t08 = list(_rows_with_run_context_t08(tlearner_ranking_smoke.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_t08),
        ('tables/model_summary.csv', model_summary_rows_t08),
        ('tables/tlearner_deciles.csv', tlearner_deciles_rows_t08),
    ):
        write_text_new(RUN_ROOT_T08_SMOKE, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    tlearner_predictions_frame_smoke = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation_t08,
        'mu1_hat': mu1_hat_smoke,
        'mu0_hat': mu0_hat_smoke,
        'tau_hat': tau_hat_smoke,
    })
    tlearner_predictions_bytes_smoke = tlearner_predictions_frame_smoke.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T08_SMOKE, 'predictions/development/tlearner/seed_42/validation_predictions.parquet', tlearner_predictions_bytes_smoke)

    mu1_model_text_smoke = mu1_model_smoke.booster.model_to_string()
    mu0_model_text_smoke = mu0_model_smoke.booster.model_to_string()
    write_text_new(RUN_ROOT_T08_SMOKE, 'models/tlearner_mu1.txt', mu1_model_text_smoke)
    write_text_new(RUN_ROOT_T08_SMOKE, 'models/tlearner_mu0.txt', mu0_model_text_smoke)

    def _flatten_t08(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_t08 = metrics.metric_definitions()
    definitions_sha256_t08 = hashlib.sha256(json.dumps(definitions_t08, sort_keys=True).encode()).hexdigest()
    definitions_rows_t08 = list(_rows_with_run_context_t08(
        [{'field': k, 'value': _flatten_t08(v)} for k, v in definitions_t08.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_t08}]
    ))
    write_text_new(RUN_ROOT_T08_SMOKE, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_t08).to_csv(index=False, lineterminator='\n'))

    tlearner_predictions_sha256_smoke = hashlib.sha256(tlearner_predictions_bytes_smoke).hexdigest()
    print('T08 SMOKE tables/models/predictions/audit written.')


### T08.7 Reload/reproducibility (derived tau tolerance)

In [ ]:
if RUN_T08_SMOKE_STAGE:
    reloaded_mu1_booster_smoke = lgb.Booster(model_str=mu1_model_text_smoke)
    reloaded_mu0_booster_smoke = lgb.Booster(model_str=mu0_model_text_smoke)
    reloaded_config_hash_mu1_smoke = lgb_baseline.config_hash()
    reloaded_config_hash_mu0_smoke = lgb_baseline.config_hash()
    config_hash_matches_smoke = (
        reloaded_config_hash_mu1_smoke == mu1_model_smoke.config_hash
        and reloaded_config_hash_mu0_smoke == mu0_model_smoke.config_hash
    )

    X_smoke_validation_rebuilt_t08 = transform_t08_smoke.transform(smoke_validation_frame_t08)
    mu1_reloaded_smoke = np.asarray(reloaded_mu1_booster_smoke.predict(X_smoke_validation_rebuilt_t08, num_iteration=mu1_model_smoke.best_iteration), dtype=np.float64)
    mu0_reloaded_smoke = np.asarray(reloaded_mu0_booster_smoke.predict(X_smoke_validation_rebuilt_t08, num_iteration=mu0_model_smoke.best_iteration), dtype=np.float64)

    mu1_reload_matches_smoke = bool(np.allclose(mu1_reloaded_smoke, mu1_hat_smoke, rtol=1e-6, atol=1e-8))
    mu0_reload_matches_smoke = bool(np.allclose(mu0_reloaded_smoke, mu0_hat_smoke, rtol=1e-6, atol=1e-8))

    tau_reload_matches_smoke, tau_reloaded_smoke, tau_reload_info_smoke = reconcile_reloaded_tau(
        mu1_reloaded_smoke, mu0_reloaded_smoke, tau_hat_smoke, mu1_hat_smoke, mu0_hat_smoke,
    )

    row_identity_matches_smoke = set(source_row_id_smoke_validation_t08) == set(smoke_validation_ids_t08)

    reload_verification_t08_smoke = {
        'config_hash_matches': bool(config_hash_matches_smoke),
        'mu1_reload_matches_within_tolerance': mu1_reload_matches_smoke,
        'mu0_reload_matches_within_tolerance': mu0_reload_matches_smoke,
        'tau_reload_matches_within_derived_tolerance': bool(tau_reload_matches_smoke),
        'tau_reload_info': tau_reload_info_smoke,
        'row_identity_matches': bool(row_identity_matches_smoke),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8, 'tau_derived_atol': 2e-8},
    }
    print(reload_verification_t08_smoke)
    assert all([
        reload_verification_t08_smoke['config_hash_matches'],
        reload_verification_t08_smoke['mu1_reload_matches_within_tolerance'],
        reload_verification_t08_smoke['mu0_reload_matches_within_tolerance'],
        reload_verification_t08_smoke['tau_reload_matches_within_derived_tolerance'],
        reload_verification_t08_smoke['row_identity_matches'],
    ])


### T08.8 T07 reuse verification (no recomputation)

In [ ]:
if RUN_T08_SMOKE_STAGE:
    t08_t07_reuse_hashes = {}
    for rel_path in ('models/response_model.txt',
                      'predictions/development/response/seed_42/validation_predictions.parquet',
                      'predictions/development/random/seed_42/validation_scores.parquet'):
        actual = hashlib.sha256((t07_full_root / rel_path).read_bytes()).hexdigest()
        expected = t07_artifact_hashes[rel_path]
        t08_t07_reuse_hashes[rel_path] = {'expected': expected, 'actual': actual, 'matches': actual == expected}
        assert actual == expected, f'T07 artifact {rel_path} hash mismatch -- refusing to cite unverified evidence'

    print('T07 reuse hash verification (all must match, none recomputed):')
    for path, info in t08_t07_reuse_hashes.items():
        print('  ' + path + ': matches=' + str(info['matches']))


In [ ]:
if RUN_T08_SMOKE_STAGE:
    t08_smoke_wall_seconds = __import__('time').perf_counter() - t08_smoke_wall_start
    t08_smoke_end_memory = process.memory_info()
    t08_smoke_end_peak_wset = getattr(t08_smoke_end_memory, 'peak_wset', None)

    if t08_smoke_baseline_peak_wset is not None and t08_smoke_end_peak_wset is not None:
        # peak_wset is an OS-tracked historical maximum (monotonically non-decreasing since process
        # start), not a before/after instantaneous sample -- valid because T07's own heavy computation
        # is skipped in this run (RUN_T07_STAGE=False), so no prior heavy stage contaminates this figure.
        t08_smoke_resource_evidence = {
            'wall_seconds': t08_smoke_wall_seconds,
            'execution_completed': True,
            'oom_or_termination_observed': False,
            'baseline_peak_wset_bytes': int(t08_smoke_baseline_peak_wset),
            'end_peak_wset_bytes': int(t08_smoke_end_peak_wset),
            'peak_rss_delta_bytes': int(t08_smoke_end_peak_wset) - int(t08_smoke_baseline_peak_wset),
            'measurement_mechanism': 'os_tracked_peak_working_set_since_process_start',
        }
    else:
        t08_smoke_resource_evidence = {
            'wall_seconds': t08_smoke_wall_seconds,
            'execution_completed': True,
            'oom_or_termination_observed': False,
            'peak_rss_delta_bytes': 'NOT_COMPARABLE',
            'peak_rss_delta_not_comparable_reason': 'psutil peak_wset unavailable in this environment; no valid stage-scoped peak measurement mechanism was available.',
        }
    print(t08_smoke_resource_evidence)


In [ ]:
if RUN_T08_SMOKE_STAGE:
    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/environment.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'created_at_utc': t08_smoke_started.isoformat(),
        'git_head': t08_git_head,
        'git_dirty': t08_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/run_config.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'created_at_utc': t08_smoke_started.isoformat(),
        'stage': 't08_smoke',
        'population': 'smoke_200000_rows',
        'git_head': t08_git_head,
        'git_dirty': t08_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_tlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t08_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t08_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        't07_reuse': {
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T07_ERRATUM_RUN_ID_ACCEPTED,
            'hash_verification': t08_t07_reuse_hashes,
            'recomputed': False,
        },
        'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES_T08, 'smoke_size': SMOKE_SIZE_T08, 'smoke_seed': SMOKE_SEED_T08},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'mu1_config_hash': mu1_model_smoke.config_hash,
        'mu0_config_hash': mu0_model_smoke.config_hash,
        'mu1_best_iteration': mu1_model_smoke.best_iteration,
        'mu0_best_iteration': mu0_model_smoke.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'tlearner_predictions_sha256': tlearner_predictions_sha256_smoke,
        'definitions_sha256': definitions_sha256_t08,
        'resource_evidence': t08_smoke_resource_evidence,
        'reload_verification': reload_verification_t08_smoke,
    })
    print('T08 SMOKE run_config.json / environment.json written.')


In [ ]:
if RUN_T08_SMOKE_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T08_SMOKE,
        run_id=RUN_ID_T08_SMOKE,
        final_status='COMPLETED_T08_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t08_smoke',
        population='smoke_200000_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/tlearner.py', 'role': 'reusable_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}', 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'T08 SMOKE run finalized: {RUN_ID_T08_SMOKE}')

    immutable_write_refused_t08_smoke = False
    try:
        write_json_new(RUN_ROOT_T08_SMOKE, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t08_smoke = True
    except Exception:
        immutable_write_refused_t08_smoke = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t08_smoke}')
    assert immutable_write_refused_t08_smoke


In [ ]:
if RUN_T08_SMOKE_STAGE:
    t08_smoke_summary = {
        'run_id': RUN_ID_T08_SMOKE,
        'smoke_total': int(smoke_total_t08),
        'smoke_train_count': int(len(smoke_train_ids_t08)),
        'smoke_validation_count': int(len(smoke_validation_ids_t08)),
        'all_partition_ty_cells_nonempty': bool(t08_smoke_all_cells_nonempty),
        'train_support': smoke_train_support_t08,
        'validation_support': smoke_validation_support_t08,
        'lightgbm_version': lgb.__version__,
        'mu1_config_hash': mu1_model_smoke.config_hash,
        'mu0_config_hash': mu0_model_smoke.config_hash,
        'mu1_best_iteration': mu1_model_smoke.best_iteration,
        'mu0_best_iteration': mu0_model_smoke.best_iteration,
        'tau_sign_counts': {'positive': positive_count, 'negative': negative_count, 'zero': zero_count},
        'tlearner_ranking_smoke_non_substantive': {
            'qini_area': tlearner_ranking_smoke.qini_area,
            'qini_above_random': tlearner_ranking_smoke.qini_above_random,
        },
        'reload_verification': reload_verification_t08_smoke,
        't07_reuse_hash_verification': t08_t07_reuse_hashes,
        'resource_evidence': t08_smoke_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t08_smoke,
    }
    print(json.dumps(t08_smoke_summary, indent=2, default=str))


In [ ]:
RUN_T08_FULL_STAGE = False  # T08 FULL is already accepted.
T08_FULL_RUN_ID_ACCEPTED = 't08_full_20260818T160823Z_081282'

if not RUN_T08_FULL_STAGE:
    t08_full_root_ref = REPO_ROOT / 'outputs' / 'runs' / T08_FULL_RUN_ID_ACCEPTED
    t08_full_manifest_path_ref = t08_full_root_ref / 'audit' / 'artifact_manifest.json'
    if not t08_full_manifest_path_ref.is_file():
        raise RuntimeError(
            f'RUN_T08_FULL_STAGE is False, but the accepted T08 FULL run evidence was not found at '
            f'{t08_full_root_ref}. Refusing to silently set RUN_T08_FULL_STAGE = True and refit mu1/mu0 -- '
            f'either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T08_FULL_RUN_ID_ACCEPTED}/), or explicitly set RUN_T08_FULL_STAGE = True '
            f'in this cell to opt into reproducing T08 FULL from scratch.'
        )
    t08_full_manifest_ref = json.loads(t08_full_manifest_path_ref.read_text(encoding='utf-8'))
    t08_full_artifact_hashes_ref = {a['path']: a['sha256'] for a in t08_full_manifest_ref['artifacts']}
    t08_full_model_summary_path_ref = t08_full_root_ref / 'tables' / 'model_summary.csv'
    t08_full_model_summary_actual_sha256 = hashlib.sha256(t08_full_model_summary_path_ref.read_bytes()).hexdigest()
    t08_full_model_summary_expected_sha256 = t08_full_artifact_hashes_ref.get('tables/model_summary.csv')
    if t08_full_model_summary_expected_sha256 is None or t08_full_model_summary_actual_sha256 != t08_full_model_summary_expected_sha256:
        raise RuntimeError(
            f'T08 FULL tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t08_full_model_summary_expected_sha256}, actual {t08_full_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    t08_reference_summary = pd.read_csv(t08_full_model_summary_path_ref)
    # Same audit-only label erratum applied in memory (T08's derived table copied the 'random' row
    # verbatim from T07); the stored file on disk is never touched.
    t08_reference_summary = t08_reference_summary.copy()
    t08_reference_summary.loc[t08_reference_summary['ranking_method'] == 'random', 'ranking_method'] = 'random_seed_42'
    tlearner_reference_row = t08_reference_summary.loc[t08_reference_summary['ranking_method'] == 'tlearner'].iloc[0]
    print('Using hash-verified frozen development results (T08 FULL).')
    print(f'  Reproducibility: T08 FULL run_id={T08_FULL_RUN_ID_ACCEPTED}')
else:
    print('RUN_T08_FULL_STAGE = True: T08 FULL will be recomputed from scratch below.')


#### Reproducibility note (technical): T08 FULL execution

SMOKE passed every correctness/artifact-mechanism check. Under D30, T08's
approved path is `SMOKE -> FULL` with `resource_gates = 0`, so FULL is the
only remaining stage. FULL fits `mu1` on the complete treated train partition
and `mu0` on the complete control train partition, early-stops each against
its own arm's complete validation subset, and both score the complete common
validation cohort. Held-out remains completely sealed.


*(This section -- T08 FULL -- is already accepted. It is skipped by default on Run All; see `RUN_T08_FULL_STAGE` immediately above. Set it to `True` only to deliberately reproduce T08 FULL from scratch.)*

In [ ]:
if RUN_T08_FULL_STAGE:
    from src.tlearner import (
        TLearnerContractError, assert_aligned_predictions, compute_tau,
        partition_by_arm, reconcile_reloaded_tau,
    )
    import src.tlearner as tlearner_module
    import threading
    import time as _time

    t08_config = json.loads((REPO_ROOT / 'configs' / 't08_tlearner.json').read_text(encoding='utf-8'))
    RESOURCE_GATES_T08 = t08_config['scale_gating']['resource_gates']


    class _PeakRSSSampler:
        # Continuous stage-scoped sampler: polls process.memory_info().rss at a
        # fixed interval and tracks the running maximum observed strictly within
        # this sampler's own start()/stop() window -- a genuinely valid mechanism
        # under ADR-experiment-artifacts' resource-measurement rule (distinct from
        # a same-process before/after peak_wset read, which is not).

        def __init__(self, proc, interval_seconds=1.0):
            self._proc = proc
            self._interval = interval_seconds
            self._max_rss_bytes = proc.memory_info().rss
            self._stop_event = threading.Event()
            self._thread = None

        def _run(self):
            while not self._stop_event.is_set():
                rss = self._proc.memory_info().rss
                if rss > self._max_rss_bytes:
                    self._max_rss_bytes = rss
                self._stop_event.wait(self._interval)

        def start(self):
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
            return self

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=5)
            return self._max_rss_bytes


    t08_full_started = datetime.now(timezone.utc)
    t08_full_wall_start = __import__('time').perf_counter()
    t08_full_baseline_rss_bytes = process.memory_info().rss
    t08_full_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T08_FULL = t08_full_started.strftime('t08_full_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T08_FULL = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T08_FULL
    RUN_ROOT_T08_FULL.mkdir(parents=True, exist_ok=False)

    try:
        t08_full_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t08_full_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t08_full_git_head, t08_full_git_dirty = None, None

    print(f'RUN_ID_T08_FULL = {RUN_ID_T08_FULL}')


In [ ]:
if RUN_T08_FULL_STAGE:
    full_train_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    full_validation_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(full_train_frame_t08) == len(train_ids_full)
    assert len(full_validation_frame_t08) == len(validation_ids_full)

    transform_t08_full = IdentityFeatureTransform()
    X_full_train_t08 = transform_t08_full.fit_transform(full_train_frame_t08)
    X_full_validation_t08 = transform_t08_full.transform(full_validation_frame_t08)
    assert_model_feature_contract(X_full_train_t08.columns)
    assert_model_feature_contract(X_full_validation_t08.columns)

    y_full_train_t08 = full_train_frame_t08[PRIMARY_OUTCOME].astype('float64')
    y_full_validation_t08 = full_validation_frame_t08[PRIMARY_OUTCOME].astype('float64')
    t_full_train_t08 = full_train_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_full_validation_t08 = full_validation_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_full_validation_arr_t08 = y_full_validation_t08.to_numpy()
    source_row_id_full_validation_t08 = full_validation_frame_t08[SOURCE_ROW_ID].to_numpy()

    print(f'X_full_train_t08: {X_full_train_t08.shape}, X_full_validation_t08: {X_full_validation_t08.shape}')


### T08.1 Arm partitioning (FULL)

In [ ]:
if RUN_T08_FULL_STAGE:
    (X_train_treated_full_t08,), (X_train_control_full_t08,) = partition_by_arm(t_full_train_t08, X_full_train_t08.to_numpy())
    X_train_treated_full_t08 = pd.DataFrame(X_train_treated_full_t08, columns=X_full_train_t08.columns)
    X_train_control_full_t08 = pd.DataFrame(X_train_control_full_t08, columns=X_full_train_t08.columns)
    (y_train_treated_full_t08,), (y_train_control_full_t08,) = partition_by_arm(t_full_train_t08, y_full_train_t08.to_numpy())

    (X_val_treated_full_t08,), (X_val_control_full_t08,) = partition_by_arm(t_full_validation_t08, X_full_validation_t08.to_numpy())
    X_val_treated_full_t08 = pd.DataFrame(X_val_treated_full_t08, columns=X_full_validation_t08.columns)
    X_val_control_full_t08 = pd.DataFrame(X_val_control_full_t08, columns=X_full_validation_t08.columns)
    (y_val_treated_full_t08,), (y_val_control_full_t08,) = partition_by_arm(t_full_validation_t08, y_full_validation_arr_t08)

    print(f'mu1 FULL train rows: {len(X_train_treated_full_t08):,} (early-stop validation: {len(X_val_treated_full_t08):,})')
    print(f'mu0 FULL train rows: {len(X_train_control_full_t08):,} (early-stop validation: {len(X_val_control_full_t08):,})')


### T08.2 Fit mu1 and mu0 (FULL, identical frozen config)

In [ ]:
if RUN_T08_FULL_STAGE:
    mu1_model_full = lgb_baseline.fit_binary_classifier(
        X_train_treated_full_t08, y_train_treated_full_t08, X_val_treated_full_t08, y_val_treated_full_t08,
    )
    mu0_model_full = lgb_baseline.fit_binary_classifier(
        X_train_control_full_t08, y_train_control_full_t08, X_val_control_full_t08, y_val_control_full_t08,
    )
    assert mu1_model_full.config_hash == mu0_model_full.config_hash
    print(f'mu1: best_iteration={mu1_model_full.best_iteration}, config_hash={mu1_model_full.config_hash[:16]}...')
    print(f'mu0: best_iteration={mu0_model_full.best_iteration}, config_hash={mu0_model_full.config_hash[:16]}...')


### T08.3 Score the full common validation cohort (FULL)

In [ ]:
if RUN_T08_FULL_STAGE:
    mu1_hat_full = lgb_baseline.predict_probabilities(mu1_model_full, X_full_validation_t08)
    mu0_hat_full = lgb_baseline.predict_probabilities(mu0_model_full, X_full_validation_t08)

    assert_aligned_predictions(source_row_id_full_validation_t08, source_row_id_full_validation_t08, source_row_id_full_validation_t08)
    assert len(mu1_hat_full) == len(validation_ids_full) == len(mu0_hat_full)
    assert set(source_row_id_full_validation_t08) == set(validation_ids_full)
    print('Both surfaces scored the full validation cohort:', len(mu1_hat_full), 'rows')


### T08.4 tau_hat = mu1_hat - mu0_hat (FULL)

In [ ]:
if RUN_T08_FULL_STAGE:
    tau_hat_full = compute_tau(mu1_hat_full, mu0_hat_full)
    np.testing.assert_array_equal(tau_hat_full, mu1_hat_full - mu0_hat_full)

    positive_count_full = int((tau_hat_full > 0).sum())
    negative_count_full = int((tau_hat_full < 0).sum())
    zero_count_full = int((tau_hat_full == 0).sum())
    assert positive_count_full == int(((mu1_hat_full - mu0_hat_full) > 0).sum())
    print(f'tau_hat (FULL): positive={positive_count_full:,} negative={negative_count_full:,} zero={zero_count_full:,}')


### T08.5 Evaluate via T06 only (FULL)

In [ ]:
if RUN_T08_FULL_STAGE:
    tlearner_ranking_full = metrics.evaluate_ranking(
        tau_hat_full, t_full_validation_t08, y_full_validation_arr_t08, source_row_id_full_validation_t08,
    )
    print(f'T-Learner FULL qini_area={tlearner_ranking_full.qini_area:.4f} qini_above_random={tlearner_ranking_full.qini_above_random:.4f}')
    # metrics.random_ranking_reference_distribution() and Response's fit_binary_classifier() are never
    # called here -- T08 FULL does not recompute T07's random reference or Response.


### T08.6 Nuisance diagnostics (each surface, own factual arm only)

In [ ]:
if RUN_T08_FULL_STAGE:
    mu1_diag_full = metrics.response_diagnostics(mu1_hat_full[t_full_validation_t08 == 1], y_full_validation_arr_t08[t_full_validation_t08 == 1])
    mu0_diag_full = metrics.response_diagnostics(mu0_hat_full[t_full_validation_t08 == 0], y_full_validation_arr_t08[t_full_validation_t08 == 0])
    print(f'mu1 diagnostics (treated-arm factual): ROC-AUC={mu1_diag_full.roc_auc:.4f} AP={mu1_diag_full.average_precision:.4f} logloss={mu1_diag_full.log_loss:.4f}')
    print(f'mu0 diagnostics (control-arm factual): ROC-AUC={mu0_diag_full.roc_auc:.4f} AP={mu0_diag_full.average_precision:.4f} logloss={mu0_diag_full.log_loss:.4f}')
    # Diagnostic only (D27) -- never a causal ranking claim; distinct from tlearner_ranking_full above.


### T08.7 T07 reuse verification (no recomputation)

In [ ]:
if RUN_T08_FULL_STAGE:
    t08_full_t07_reuse_hashes = {}
    for rel_path in ('models/response_model.txt',
                      'predictions/development/response/seed_42/validation_predictions.parquet',
                      'predictions/development/random/seed_42/validation_scores.parquet',
                      'tables/model_summary.csv'):
        actual = hashlib.sha256((t07_full_root / rel_path).read_bytes()).hexdigest()
        expected = t07_artifact_hashes[rel_path]
        t08_full_t07_reuse_hashes[rel_path] = {'expected': expected, 'actual': actual, 'matches': actual == expected}
        assert actual == expected, 'T07 artifact ' + rel_path + ' hash mismatch -- refusing to cite unverified evidence'

    t07_reference_random_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random'].iloc[0]
    t07_reference_response_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'response'].iloc[0]
    t07_model_summary_sha256 = t08_full_t07_reuse_hashes['tables/model_summary.csv']['actual']
    t07_response_predictions_sha256_ref = t08_full_t07_reuse_hashes['predictions/development/response/seed_42/validation_predictions.parquet']['actual']
    t07_random_scores_sha256_ref = t08_full_t07_reuse_hashes['predictions/development/random/seed_42/validation_scores.parquet']['actual']

    print('T07 reuse hash verification (all must match, none recomputed):')
    for path, info in t08_full_t07_reuse_hashes.items():
        print('  ' + path + ': matches=' + str(info['matches']))


### T08.8 Comparative model_summary (random + response + tlearner, explicit lineage)

In [ ]:
if RUN_T08_FULL_STAGE:
    def _full_rows_with_run_context_t08(rows, population='full_development_population'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T08_FULL)
            row.setdefault('stage', 't08_full')
            row.setdefault('population', population)
            yield row


    comparative_model_summary_rows = [
        {
            'ranking_method': 'random',
            'qini_area': float(t07_reference_random_row['qini_area']),
            'qini_above_random': float(t07_reference_random_row['qini_above_random']),
            'theoretical_random_qini_area': float(t07_reference_random_row['theoretical_random_qini_area']),
            'qini_above_random_permutation': float(t07_reference_random_row['qini_above_random_permutation']),
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_prediction_sha256': t07_random_scores_sha256_ref,
            'source_metric_artifact_sha256': t07_model_summary_sha256,
        },
        {
            'ranking_method': 'response',
            'qini_area': float(t07_reference_response_row['qini_area']),
            'qini_above_random': float(t07_reference_response_row['qini_above_random']),
            'theoretical_random_qini_area': float(t07_reference_response_row['theoretical_random_qini_area']),
            'qini_above_random_permutation': float(t07_reference_response_row['qini_above_random_permutation']),
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_prediction_sha256': t07_response_predictions_sha256_ref,
            'source_metric_artifact_sha256': t07_model_summary_sha256,
        },
        {
            'ranking_method': 'tlearner',
            'qini_area': tlearner_ranking_full.qini_area,
            'qini_above_random': tlearner_ranking_full.qini_above_random,
            'theoretical_random_qini_area': tlearner_ranking_full.theoretical_random_qini_area,
            'qini_above_random_permutation': float(t07_reference_random_row['qini_above_random_permutation']),
            'uplift_at_10pct': tlearner_ranking_full.uplift_at_k['10pct'],
            'uplift_at_20pct': tlearner_ranking_full.uplift_at_k['20pct'],
            'uplift_at_30pct': tlearner_ranking_full.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': tlearner_ranking_full.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': tlearner_ranking_full.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': tlearner_ranking_full.incremental_conversions_at_k['30pct'],
            'source_run_id': RUN_ID_T08_FULL,
            'source_prediction_sha256': None,  # populated below once the parquet is written and hashed
            'source_metric_artifact_sha256': None,  # self-originated, not reused from elsewhere
        },
    ]
    print('Random/response rows: source_run_id=' + T07_FULL_RUN_ID_ACCEPTED + ' (values inherited unchanged from T07, never recomputed)')
    print('T-Learner row: source_run_id=' + RUN_ID_T08_FULL + ' (genuinely new this run)')


### T08.9 Write FULL artifacts

In [ ]:
if RUN_T08_FULL_STAGE:
    tlearner_predictions_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation_t08,
        'mu1_hat': mu1_hat_full,
        'mu0_hat': mu0_hat_full,
        'tau_hat': tau_hat_full,
    })
    tlearner_predictions_bytes_full = tlearner_predictions_frame_full.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T08_FULL, 'predictions/development/tlearner/seed_42/validation_predictions.parquet', tlearner_predictions_bytes_full)
    tlearner_predictions_sha256_full = hashlib.sha256(tlearner_predictions_bytes_full).hexdigest()
    comparative_model_summary_rows[2]['source_prediction_sha256'] = tlearner_predictions_sha256_full

    mu1_model_text_full = mu1_model_full.booster.model_to_string()
    mu0_model_text_full = mu0_model_full.booster.model_to_string()
    write_text_new(RUN_ROOT_T08_FULL, 'models/tlearner_mu1.txt', mu1_model_text_full)
    write_text_new(RUN_ROOT_T08_FULL, 'models/tlearner_mu0.txt', mu0_model_text_full)

    uplift_at_k_rows_full_t08 = list(_full_rows_with_run_context_t08([
        {'method': 'tlearner', 'k': label, 'uplift': tlearner_ranking_full.uplift_at_k[label],
         'incremental_conversions': tlearner_ranking_full.incremental_conversions_at_k[label],
         'status': tlearner_ranking_full.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    tlearner_deciles_rows_full = list(_full_rows_with_run_context_t08(tlearner_ranking_full.decile_table.to_dict('records')))
    response_diagnostics_rows_full_t08 = list(_full_rows_with_run_context_t08([
        {'method': 'tlearner_mu1', **mu1_diag_full.__dict__},
        {'method': 'tlearner_mu0', **mu0_diag_full.__dict__},
    ]))
    model_summary_rows_full_t08 = list(_full_rows_with_run_context_t08(comparative_model_summary_rows))

    for name, rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full_t08),
        ('tables/tlearner_deciles.csv', tlearner_deciles_rows_full),
        ('tables/response_diagnostics.csv', response_diagnostics_rows_full_t08),
        ('tables/model_summary.csv', model_summary_rows_full_t08),
    ):
        write_text_new(RUN_ROOT_T08_FULL, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    def _flatten_t08_full(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_t08_full = metrics.metric_definitions()
    definitions_sha256_t08_full = hashlib.sha256(json.dumps(definitions_t08_full, sort_keys=True).encode()).hexdigest()
    definitions_rows_t08_full = list(_full_rows_with_run_context_t08(
        [{'field': k, 'value': _flatten_t08_full(v)} for k, v in definitions_t08_full.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_t08_full}]
    ))
    write_text_new(RUN_ROOT_T08_FULL, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_t08_full).to_csv(index=False, lineterminator='\n'))

    print('T08 FULL tables/models/predictions/audit written.')


### T08.10 Figure

In [ ]:
if RUN_T08_FULL_STAGE:
    import io as _io_full
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    deciles_full = tlearner_ranking_full.decile_table.sort_values('decile')
    ax.bar(deciles_full['decile'], deciles_full['observed_uplift'])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Decile (1 = highest tau_hat)')
    ax.set_ylabel('Observed uplift (treated_rate - control_rate)')
    ax.set_title('T-Learner: observed uplift by decile (FULL)')
    fig.tight_layout()
    buffer = _io_full.BytesIO()
    fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    write_bytes_new(RUN_ROOT_T08_FULL, 'figures/tlearner_uplift_deciles.png', buffer.getvalue())
    print('figures/tlearner_uplift_deciles.png written.')


### T08.11 Reload/reproducibility (FULL, derived tau tolerance)

In [ ]:
if RUN_T08_FULL_STAGE:
    reloaded_mu1_booster_full = lgb.Booster(model_str=mu1_model_text_full)
    reloaded_mu0_booster_full = lgb.Booster(model_str=mu0_model_text_full)
    config_hash_matches_full_t08 = (
        lgb_baseline.config_hash() == mu1_model_full.config_hash
        and lgb_baseline.config_hash() == mu0_model_full.config_hash
    )

    X_full_validation_rebuilt_t08 = transform_t08_full.transform(full_validation_frame_t08)
    mu1_reloaded_full = np.asarray(reloaded_mu1_booster_full.predict(X_full_validation_rebuilt_t08, num_iteration=mu1_model_full.best_iteration), dtype=np.float64)
    mu0_reloaded_full = np.asarray(reloaded_mu0_booster_full.predict(X_full_validation_rebuilt_t08, num_iteration=mu0_model_full.best_iteration), dtype=np.float64)

    mu1_reload_matches_full = bool(np.allclose(mu1_reloaded_full, mu1_hat_full, rtol=1e-6, atol=1e-8))
    mu0_reload_matches_full = bool(np.allclose(mu0_reloaded_full, mu0_hat_full, rtol=1e-6, atol=1e-8))
    tau_reload_matches_full, tau_reloaded_full_arr, tau_reload_info_full = reconcile_reloaded_tau(
        mu1_reloaded_full, mu0_reloaded_full, tau_hat_full, mu1_hat_full, mu0_hat_full,
    )
    row_identity_matches_full_t08 = set(source_row_id_full_validation_t08) == set(validation_ids_full)

    reload_verification_t08_full = {
        'config_hash_matches': bool(config_hash_matches_full_t08),
        'mu1_reload_matches_within_tolerance': mu1_reload_matches_full,
        'mu0_reload_matches_within_tolerance': mu0_reload_matches_full,
        'tau_reload_matches_within_derived_tolerance': bool(tau_reload_matches_full),
        'tau_reload_info': tau_reload_info_full,
        'row_identity_matches': bool(row_identity_matches_full_t08),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8, 'tau_derived_atol': 2e-8},
    }
    print(reload_verification_t08_full)
    assert all([
        reload_verification_t08_full['config_hash_matches'],
        reload_verification_t08_full['mu1_reload_matches_within_tolerance'],
        reload_verification_t08_full['mu0_reload_matches_within_tolerance'],
        reload_verification_t08_full['tau_reload_matches_within_derived_tolerance'],
        reload_verification_t08_full['row_identity_matches'],
    ])


In [ ]:
if RUN_T08_FULL_STAGE:
    t08_full_wall_seconds = __import__('time').perf_counter() - t08_full_wall_start
    t08_full_max_rss_bytes_observed = t08_full_rss_sampler.stop()

    t08_full_resource_evidence = {
        'wall_seconds': t08_full_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t08_full_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t08_full_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t08_full_max_rss_bytes_observed) - int(t08_full_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(t08_full_resource_evidence)


In [ ]:
if RUN_T08_FULL_STAGE:
    write_json_new(RUN_ROOT_T08_FULL, 'audit/environment.json', {
        'run_id': RUN_ID_T08_FULL,
        'created_at_utc': t08_full_started.isoformat(),
        'git_head': t08_full_git_head,
        'git_dirty': t08_full_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T08_FULL, 'audit/run_config.json', {
        'run_id': RUN_ID_T08_FULL,
        'created_at_utc': t08_full_started.isoformat(),
        'stage': 't08_full',
        'population': 'full_development_population',
        'git_head': t08_full_git_head,
        'git_dirty': t08_full_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_tlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t08_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t08_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        't07_reuse': {
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T07_ERRATUM_RUN_ID_ACCEPTED,
            'hash_verification': t08_full_t07_reuse_hashes,
            'recomputed': False,
        },
        't08_smoke_reuse': {
            'source_run_id': T08_SMOKE_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T08_SMOKE_ERRATUM_RUN_ID_ACCEPTED,
            'refit': False,
        },
        'train_treated_count': int(len(X_train_treated_full_t08)),
        'train_control_count': int(len(X_train_control_full_t08)),
        'validation_treated_count': int(len(X_val_treated_full_t08)),
        'validation_control_count': int(len(X_val_control_full_t08)),
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES_T08, 'preceding_smoke_run_id': T08_SMOKE_RUN_ID_ACCEPTED},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'mu1_config_hash': mu1_model_full.config_hash,
        'mu0_config_hash': mu0_model_full.config_hash,
        'mu1_best_iteration': mu1_model_full.best_iteration,
        'mu0_best_iteration': mu0_model_full.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'tlearner_predictions_sha256': tlearner_predictions_sha256_full,
        'definitions_sha256': definitions_sha256_t08_full,
        'resource_evidence': t08_full_resource_evidence,
        'reload_verification': reload_verification_t08_full,
    })
    print('T08 FULL run_config.json / environment.json written.')


In [ ]:
if RUN_T08_FULL_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T08_FULL,
        run_id=RUN_ID_T08_FULL,
        final_status='COMPLETED_T08_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t08_full',
        population='full_development_population',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/tlearner.py', 'role': 'reusable_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}', 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f'outputs/runs/{T08_SMOKE_RUN_ID_ACCEPTED}', 'role': 'preceding_t08_smoke_run', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'T08 FULL run finalized: {RUN_ID_T08_FULL}')

    immutable_write_refused_t08_full = False
    try:
        write_json_new(RUN_ROOT_T08_FULL, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t08_full = True
    except Exception:
        immutable_write_refused_t08_full = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t08_full}')
    assert immutable_write_refused_t08_full


In [ ]:
if RUN_T08_FULL_STAGE:
    t08_full_summary = {
        'run_id': RUN_ID_T08_FULL,
        'train_treated_count': int(len(X_train_treated_full_t08)),
        'train_control_count': int(len(X_train_control_full_t08)),
        'validation_treated_count': int(len(X_val_treated_full_t08)),
        'validation_control_count': int(len(X_val_control_full_t08)),
        'mu1_config_hash': mu1_model_full.config_hash,
        'mu0_config_hash': mu0_model_full.config_hash,
        'mu1_best_iteration': mu1_model_full.best_iteration,
        'mu0_best_iteration': mu0_model_full.best_iteration,
        'tau_sign_counts': {'positive': positive_count_full, 'negative': negative_count_full, 'zero': zero_count_full},
        'tlearner_ranking_full': {
            'qini_area': tlearner_ranking_full.qini_area,
            'qini_above_random': tlearner_ranking_full.qini_above_random,
            'theoretical_random_qini_area': tlearner_ranking_full.theoretical_random_qini_area,
            'uplift_at_k': tlearner_ranking_full.uplift_at_k,
            'incremental_conversions_at_k': tlearner_ranking_full.incremental_conversions_at_k,
        },
        'mu1_diagnostics': mu1_diag_full.__dict__,
        'mu0_diagnostics': mu0_diag_full.__dict__,
        'comparison': comparative_model_summary_rows,
        'reload_verification': reload_verification_t08_full,
        't07_reuse_hash_verification': t08_full_t07_reuse_hashes,
        'resource_evidence': t08_full_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t08_full,
    }
    print(json.dumps(t08_full_summary, indent=2, default=str))


## Comparison

Random, Response, and T-Learner are evaluated on the identical frozen
validation cohort, all through the same T06 metric interface -- no method has
its own separate formula. The theoretical random line is the primary no-skill
reference (D11) and has `qini_above_random = 0` by construction, since it is
compared against itself; the seed-42 draw below is one illustrative empirical
realization of a random ranking, not the benchmark itself.


In [ ]:
from IPython.display import Markdown, display

_random_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random_seed_42'].iloc[0]
_response_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'response'].iloc[0]

comparison_table = pd.DataFrame([
    {'method': 'theoretical_random (primary reference)', 'qini_area': None, 'qini_above_random': 0.0},
    {'method': 'random_seed_42 (illustrative draw)', 'qini_area': float(_random_row['qini_area']), 'qini_above_random': float(_random_row['qini_above_random'])},
    {'method': 'response', 'qini_area': float(_response_row['qini_area']), 'qini_above_random': float(_response_row['qini_above_random'])},
    {'method': 'tlearner', 'qini_area': float(tlearner_reference_row['qini_area']), 'qini_above_random': float(tlearner_reference_row['qini_above_random'])},
])
print(comparison_table.to_string(index=False))


## Interpretation and limitations

In [ ]:
display(Markdown(
    f"- **Response** predicts factual conversion well but does not rank uplift well on this cohort: "
    f"`qini_above_random = {float(_response_row['qini_above_random']):.2f}`.\n\n"
    f"- **T-Learner** estimates `tau_hat(x) = mu1_hat(x) - mu0_hat(x)` from two arm-specific factual "
    f"outcome surfaces (`mu1` fit on treated rows only, `mu0` fit on control rows only); treatment is "
    f"never a feature in either surface.\n\n"
    f"- On the frozen validation cohort, T-Learner reaches `qini_above_random = "
    f"{float(tlearner_reference_row['qini_above_random']):.2f}`, against the theoretical random "
    f"reference at exactly 0 by construction.\n\n"
    f"- **This fitted T-Learner did not produce a useful uplift ranking relative to the theoretical "
    f"random reference on this development validation cohort.**\n\n"
    f"- This does not show that treatment heterogeneity is absent -- it is a statement about this "
    f"fitted model's ranking on this cohort, not about the underlying causal structure.\n\n"
    f"- It does not invalidate T-Learner methodology or this implementation.\n\n"
    f"- It does not authorize post-hoc tuning outside the frozen D30/docs-06 protocol -- any future "
    f"model change follows its own CODE PLAN and scale-gating path.\n\n"
    f"- T-Learner's `qini_above_random` is less negative than Response's on this cohort; this is "
    f"reported as a numeric fact only, not a claim that T-Learner is \"better\" in any general sense -- "
    f"both rank below the theoretical no-skill reference here."
))


In [ ]:
RUN_T09_SMOKE_STAGE = False  # T09 corrected SMOKE is already accepted.
T09_SMOKE_RUN_ID_ACCEPTED = 't09_smoke_20260819T034244Z_369573'
T09_SMOKE_SUPERSEDED_RUN_ID = 't09_smoke_20260819T032427Z_083588'  # original SMOKE, artifact-path nonconforming, left untouched

if not RUN_T09_SMOKE_STAGE:
    t09_smoke_root_ref = REPO_ROOT / 'outputs' / 'runs' / T09_SMOKE_RUN_ID_ACCEPTED
    t09_smoke_manifest_path_ref = t09_smoke_root_ref / 'audit' / 'artifact_manifest.json'
    if not t09_smoke_manifest_path_ref.is_file():
        raise RuntimeError(
            f'RUN_T09_SMOKE_STAGE is False, but the accepted corrected T09 SMOKE run evidence was not '
            f'found at {t09_smoke_root_ref}. Refusing to silently set RUN_T09_SMOKE_STAGE = True and '
            f'refit the SMOKE mechanism -- either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T09_SMOKE_RUN_ID_ACCEPTED}/), or explicitly set RUN_T09_SMOKE_STAGE = True '
            f'in this cell to opt into reproducing T09 SMOKE from scratch.'
        )
    t09_smoke_manifest_ref = json.loads(t09_smoke_manifest_path_ref.read_text(encoding='utf-8'))
    t09_smoke_artifact_hashes_ref = {a['path']: a['sha256'] for a in t09_smoke_manifest_ref['artifacts']}
    t09_smoke_model_summary_path_ref = t09_smoke_root_ref / 'tables' / 'model_summary.csv'
    t09_smoke_model_summary_actual_sha256 = hashlib.sha256(t09_smoke_model_summary_path_ref.read_bytes()).hexdigest()
    t09_smoke_model_summary_expected_sha256 = t09_smoke_artifact_hashes_ref.get('tables/model_summary.csv')
    if t09_smoke_model_summary_expected_sha256 is None or t09_smoke_model_summary_actual_sha256 != t09_smoke_model_summary_expected_sha256:
        raise RuntimeError(
            f'T09 corrected SMOKE tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t09_smoke_model_summary_expected_sha256}, actual {t09_smoke_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    print(f'T09 corrected SMOKE already accepted at run_id {T09_SMOKE_RUN_ID_ACCEPTED} (hash-verified); skipping refit.')
    print(f'Superseded (artifact-path nonconforming, left untouched, scientifically valid): {T09_SMOKE_SUPERSEDED_RUN_ID}')
else:
    print('RUN_T09_SMOKE_STAGE = True: T09 SMOKE will be recomputed from scratch below.')


## Scale-Gating (D30): T09 SMOKE

T09 (X-Learner) uses the D30 path declared in `configs/t09_xlearner.json`:
`SMOKE -> FULL`, `resource_gates = 0`, `smoke_size = 200,000`. X-Learner fits
four nuisance models (`mu0_A`, `mu1_A`, `mu0_B`, `mu1_B`) via training-only
two-fold cross-fitting, assembles both out-of-fold surfaces
(`mu0_oof`, `mu1_oof`) onto every training row from the *opposite* fold's
models, constructs signed pseudo-effects (`D1 = Y - mu0_oof` on treated rows,
`D0 = mu1_oof - Y` on control rows), fits two effect regressors (`tau1`,
`tau0`) at a fixed, predeclared `EFFECT_NUM_BOOST_ROUND = 100` (no early
stopping -- no pseudo-effect target is ever defined on validation rows), and
combines `tau_hat(x) = g(x)*tau0_hat(x) + (1-g(x))*tau1_hat(x)` with `g(x)`
the training-only empirical treatment rate. SMOKE proves this six-model
mechanism executes correctly end to end on non-degenerate data; it is not a
performance estimate.


*(This section -- T09 corrected SMOKE -- is already accepted. It is skipped by default on Run All; see `RUN_T09_SMOKE_STAGE` immediately above. Set it to `True` only to deliberately reproduce T09 SMOKE from scratch.)*

In [ ]:
if RUN_T09_SMOKE_STAGE:
    from src.xlearner import (
        XLearnerContractError, assert_full_oof_coverage, assert_opposite_fold,
        assign_folds, combine_tau, compute_pseudo_outcomes, empirical_treatment_rate,
    )
    import src.xlearner as xlearner_module
    import threading

    t09_config = json.loads((REPO_ROOT / 'configs' / 't09_xlearner.json').read_text(encoding='utf-8'))

    SMOKE_SIZE_T09 = t09_config['scale_gating']['smoke_size']
    SMOKE_SEED_T09 = t09_config['scale_gating']['smoke_seed']
    FOLD_SEED_T09 = t09_config['scale_gating']['fold_seed']
    MODEL_SEED_T09 = t09_config['scale_gating']['model_seed']
    RESOURCE_GATES_T09 = t09_config['scale_gating']['resource_gates']
    # Seed-scoped prediction directory, per the authoritative X-Learner artifact
    # contract (docs/05_methodology_scope.md, ADR-experiment-artifacts.md):
    # predictions/development/xlearner/seed_<seed>/{oof_nuisance,pseudo_outcomes,
    # validation_predictions}.parquet. Derived from MODEL_SEED_T09 (not hardcoded)
    # so a future different-seed run (robustness seeds 123/2026) writes to its own
    # directory automatically, with no further code changes.
    XLEARNER_SEED_DIR_T09 = f'predictions/development/xlearner/seed_{int(MODEL_SEED_T09)}'


    class _PeakRSSSampler:
        # Continuous stage-scoped sampler: polls process.memory_info().rss at a
        # fixed interval and tracks the running maximum observed strictly within
        # this sampler's own start()/stop() window -- a genuinely valid mechanism
        # under ADR-experiment-artifacts' resource-measurement rule (distinct from
        # a same-process before/after peak_wset read, which is not).

        def __init__(self, proc, interval_seconds=1.0):
            self._proc = proc
            self._interval = interval_seconds
            self._max_rss_bytes = proc.memory_info().rss
            self._stop_event = threading.Event()
            self._thread = None

        def _run(self):
            while not self._stop_event.is_set():
                rss = self._proc.memory_info().rss
                if rss > self._max_rss_bytes:
                    self._max_rss_bytes = rss
                self._stop_event.wait(self._interval)

        def start(self):
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
            return self

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=5)
            return self._max_rss_bytes


    t09_smoke_started = datetime.now(timezone.utc)
    t09_smoke_wall_start = __import__('time').perf_counter()
    t09_smoke_baseline_rss_bytes = process.memory_info().rss
    t09_smoke_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T09_SMOKE = t09_smoke_started.strftime('t09_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T09_SMOKE = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_SMOKE
    RUN_ROOT_T09_SMOKE.mkdir(parents=True, exist_ok=False)

    try:
        t09_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t09_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t09_git_head, t09_git_dirty = None, None

    print(f'RUN_ID_T09_SMOKE = {RUN_ID_T09_SMOKE}')
    print(f'resource_gates = {RESOURCE_GATES_T09}, smoke_size = {SMOKE_SIZE_T09}, fold_seed={FOLD_SEED_T09}, model_seed={MODEL_SEED_T09}')


### T09.0 SMOKE population (same rule as T07/T08)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    def _joint_strata_t09(frame, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_partition_t09(partition_ids, quota, full_frame, seed):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata_t09(subset, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    p_train_t09 = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
    train_quota_t09 = round(SMOKE_SIZE_T09 * p_train_t09)
    validation_quota_t09 = SMOKE_SIZE_T09 - train_quota_t09

    smoke_train_ids_t09 = smoke_sample_partition_t09(train_ids_full, train_quota_t09, full_frame, SMOKE_SEED_T09)
    smoke_validation_ids_t09 = smoke_sample_partition_t09(validation_ids_full, validation_quota_t09, full_frame, SMOKE_SEED_T09)

    smoke_total_t09 = len(smoke_train_ids_t09) + len(smoke_validation_ids_t09)
    assert smoke_total_t09 == SMOKE_SIZE_T09, f'T09 SMOKE total {smoke_total_t09} != {SMOKE_SIZE_T09}'
    assert set(smoke_train_ids_t09).issubset(set(train_ids_full)), 'smoke_train_ids_t09 must be a subset of the frozen train partition'
    assert set(smoke_validation_ids_t09).issubset(set(validation_ids_full)), 'smoke_validation_ids_t09 must be a subset of the frozen validation partition'
    assert set(smoke_train_ids_t09).isdisjoint(set(smoke_validation_ids_t09))
    development_ids_full_t09 = set(train_ids_full) | set(validation_ids_full)
    assert (set(smoke_train_ids_t09) | set(smoke_validation_ids_t09)).issubset(development_ids_full_t09), (
        'every T09 SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
    )
    # Held-out isolation proved by construction only, same as T07/T08 -- never read via
    # split_membership.csv's held_out label or SplitDataset.held_out_ids().

    print(f'T09 SMOKE: train={len(smoke_train_ids_t09):,} validation={len(smoke_validation_ids_t09):,} total={smoke_total_t09:,}')
    print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


In [ ]:
if RUN_T09_SMOKE_STAGE:
    def joint_ty_support_t09(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support_t09 = joint_ty_support_t09(smoke_train_ids_t09, full_frame)
    smoke_validation_support_t09 = joint_ty_support_t09(smoke_validation_ids_t09, full_frame)
    print('T09 SMOKE train (T,Y) support:', smoke_train_support_t09)
    print('T09 SMOKE validation (T,Y) support:', smoke_validation_support_t09)

    t09_smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support_t09, **smoke_validation_support_t09}.values())
    print('All 8 partition x (T,Y) cells non-empty:', t09_smoke_all_cells_nonempty)
    if not t09_smoke_all_cells_nonempty:
        raise XLearnerContractError(
            'T09 SMOKE support is degenerate: at least one train/validation x (T,Y) cell is empty at '
            f'smoke_size={SMOKE_SIZE_T09}. SMOKE purpose is engineering/correctness validation, not '
            'performance estimation -- failing closed rather than adjusting the sample after seeing this.'
        )


In [ ]:
if RUN_T09_SMOKE_STAGE:
    smoke_sample_ids_frame_t09 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids_t09, 'partition': 'train'}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids_t09, 'partition': 'validation'}),
    ], ignore_index=True)
    smoke_sample_ids_sha256_t09 = hashlib.sha256(
        smoke_sample_ids_frame_t09.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
    ).hexdigest()

    write_bytes_new(RUN_ROOT_T09_SMOKE, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame_t09.to_parquet(index=False))
    write_json_new(RUN_ROOT_T09_SMOKE, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID_T09_SMOKE,
        'stage': 't09_smoke',
        'population': 'smoke_200000_rows',
        'smoke_size': SMOKE_SIZE_T09,
        'smoke_seed': SMOKE_SEED_T09,
        'train_quota': int(train_quota_t09),
        'validation_quota': int(validation_quota_t09),
        'train_count': int(len(smoke_train_ids_t09)),
        'validation_count': int(len(smoke_validation_ids_t09)),
        'total_count': int(smoke_total_t09),
        'train_support': smoke_train_support_t09,
        'validation_support': smoke_validation_support_t09,
        'all_partition_ty_cells_nonempty': bool(t09_smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256_t09,
        'held_out_isolation_method': (
            'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids_t09 and '
            'smoke_validation_ids_t09 are proved subsets of train_ids_full/validation_ids_full (each obtained '
            'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
            'subset of train_ids_full union validation_ids_full -- held-out is never read via '
            'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
        ),
    })
    print('T09 SMOKE sample identity persisted.')


In [ ]:
if RUN_T09_SMOKE_STAGE:
    smoke_train_frame_t09 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids_t09)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    smoke_validation_frame_t09 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids_t09)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    transform_t09_smoke = IdentityFeatureTransform()
    X_smoke_train_t09 = transform_t09_smoke.fit_transform(smoke_train_frame_t09)
    X_smoke_validation_t09 = transform_t09_smoke.transform(smoke_validation_frame_t09)
    assert_model_feature_contract(X_smoke_train_t09.columns)
    assert_model_feature_contract(X_smoke_validation_t09.columns)

    y_smoke_train_t09 = smoke_train_frame_t09[PRIMARY_OUTCOME].astype('float64')
    y_smoke_validation_t09 = smoke_validation_frame_t09[PRIMARY_OUTCOME].astype('float64')
    t_smoke_train_t09 = smoke_train_frame_t09[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_smoke_validation_t09 = smoke_validation_frame_t09[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_smoke_train_arr_t09 = y_smoke_train_t09.to_numpy()
    y_smoke_validation_arr_t09 = y_smoke_validation_t09.to_numpy()
    source_row_id_smoke_train_t09 = smoke_train_frame_t09[SOURCE_ROW_ID].to_numpy()
    source_row_id_smoke_validation_t09 = smoke_validation_frame_t09[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_train_t09: {X_smoke_train_t09.shape}, X_smoke_validation_t09: {X_smoke_validation_t09.shape}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_train_t09.columns))


### T09.1 Two-fold cross-fitting assignment (training-only, joint-(T,Y)-stratified)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    fold_a_ids_t09, fold_b_ids_t09 = assign_folds(
        source_row_id_smoke_train_t09, t_smoke_train_t09, y_smoke_train_arr_t09, seed=FOLD_SEED_T09,
    )
    assert set(fold_a_ids_t09).isdisjoint(set(fold_b_ids_t09))
    assert set(fold_a_ids_t09) | set(fold_b_ids_t09) == set(source_row_id_smoke_train_t09.tolist())

    fold_label_smoke_train_t09 = np.where(np.isin(source_row_id_smoke_train_t09, fold_a_ids_t09), 'A', 'B')
    mask_fold_a_t09 = fold_label_smoke_train_t09 == 'A'
    mask_fold_b_t09 = ~mask_fold_a_t09

    print(f'Fold A: {len(fold_a_ids_t09):,} rows, Fold B: {len(fold_b_ids_t09):,} rows')


### T09.2 Per-fold, per-arm nuisance-fit partitions + shared arm-relevant validation subsets

In [ ]:
if RUN_T09_SMOKE_STAGE:
    X_smoke_train_arr_t09 = X_smoke_train_t09.to_numpy()


    def _fold_arm_frame_t09(mask_fold, arm_value):
        mask = mask_fold & (t_smoke_train_t09 == arm_value)
        X = pd.DataFrame(X_smoke_train_arr_t09[mask], columns=X_smoke_train_t09.columns)
        y = y_smoke_train_arr_t09[mask]
        return X, y


    X_A_treated_t09, y_A_treated_t09 = _fold_arm_frame_t09(mask_fold_a_t09, 1.0)
    X_A_control_t09, y_A_control_t09 = _fold_arm_frame_t09(mask_fold_a_t09, 0.0)
    X_B_treated_t09, y_B_treated_t09 = _fold_arm_frame_t09(mask_fold_b_t09, 1.0)
    X_B_control_t09, y_B_control_t09 = _fold_arm_frame_t09(mask_fold_b_t09, 0.0)

    for _name, _X in (('A_treated', X_A_treated_t09), ('A_control', X_A_control_t09), ('B_treated', X_B_treated_t09), ('B_control', X_B_control_t09)):
        assert len(_X) > 0, f'T09 SMOKE fold/arm cell {_name} is empty at smoke_size={SMOKE_SIZE_T09}'
        print(f'{_name}: {len(_X):,} rows')

    mask_val_treated_t09 = t_smoke_validation_t09 == 1
    mask_val_control_t09 = t_smoke_validation_t09 == 0
    X_val_treated_t09 = X_smoke_validation_t09.loc[mask_val_treated_t09].reset_index(drop=True)
    y_val_treated_t09 = y_smoke_validation_arr_t09[mask_val_treated_t09]
    X_val_control_t09 = X_smoke_validation_t09.loc[mask_val_control_t09].reset_index(drop=True)
    y_val_control_t09 = y_smoke_validation_arr_t09[mask_val_control_t09]
    assert len(X_val_treated_t09) > 0 and len(X_val_control_t09) > 0
    print(f'Validation (arm-relevant, shared across both folds): treated={len(X_val_treated_t09):,} control={len(X_val_control_t09):,}')


### T09.3 Fit the four nuisance models (identical frozen binary config)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    mu0_A_model_t09 = lgb_baseline.fit_binary_classifier(X_A_control_t09, y_A_control_t09, X_val_control_t09, y_val_control_t09, seed=MODEL_SEED_T09)
    mu1_A_model_t09 = lgb_baseline.fit_binary_classifier(X_A_treated_t09, y_A_treated_t09, X_val_treated_t09, y_val_treated_t09, seed=MODEL_SEED_T09)
    mu0_B_model_t09 = lgb_baseline.fit_binary_classifier(X_B_control_t09, y_B_control_t09, X_val_control_t09, y_val_control_t09, seed=MODEL_SEED_T09)
    mu1_B_model_t09 = lgb_baseline.fit_binary_classifier(X_B_treated_t09, y_B_treated_t09, X_val_treated_t09, y_val_treated_t09, seed=MODEL_SEED_T09)

    assert mu0_A_model_t09.config_hash == mu1_A_model_t09.config_hash == mu0_B_model_t09.config_hash == mu1_B_model_t09.config_hash
    for _name, _model in (('mu0_A', mu0_A_model_t09), ('mu1_A', mu1_A_model_t09), ('mu0_B', mu0_B_model_t09), ('mu1_B', mu1_B_model_t09)):
        print(f'{_name}: best_iteration={_model.best_iteration}, config_hash={_model.config_hash[:16]}...')


### T09.4 Out-of-fold nuisance assembly (fold-uniform: BOTH surfaces from the OPPOSITE fold, every row)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    X_A_full_t09 = pd.DataFrame(X_smoke_train_arr_t09[mask_fold_a_t09], columns=X_smoke_train_t09.columns)
    X_B_full_t09 = pd.DataFrame(X_smoke_train_arr_t09[mask_fold_b_t09], columns=X_smoke_train_t09.columns)
    source_row_id_fold_a_t09 = source_row_id_smoke_train_t09[mask_fold_a_t09]
    source_row_id_fold_b_t09 = source_row_id_smoke_train_t09[mask_fold_b_t09]

    # Fold A rows: mu0_oof <- mu0_B(x), mu1_oof <- mu1_B(x). Fold B rows: mu0_oof <- mu0_A(x), mu1_oof <- mu1_A(x).
    # Applied to EVERY row in the opposite fold regardless of arm -- no arm-selective application.
    mu0_oof_a_t09 = lgb_baseline.predict_probabilities(mu0_B_model_t09, X_A_full_t09)
    mu1_oof_a_t09 = lgb_baseline.predict_probabilities(mu1_B_model_t09, X_A_full_t09)
    mu0_oof_b_t09 = lgb_baseline.predict_probabilities(mu0_A_model_t09, X_B_full_t09)
    mu1_oof_b_t09 = lgb_baseline.predict_probabilities(mu1_A_model_t09, X_B_full_t09)

    row_fold_unsorted_t09 = np.concatenate([np.full(len(source_row_id_fold_a_t09), 'A'), np.full(len(source_row_id_fold_b_t09), 'B')])
    prediction_fold_of_mu0_t09 = np.concatenate([np.full(len(source_row_id_fold_a_t09), 'B'), np.full(len(source_row_id_fold_b_t09), 'A')])
    prediction_fold_of_mu1_t09 = prediction_fold_of_mu0_t09.copy()
    assert_opposite_fold(row_fold_unsorted_t09, prediction_fold_of_mu0_t09, name='mu0_oof')
    assert_opposite_fold(row_fold_unsorted_t09, prediction_fold_of_mu1_t09, name='mu1_oof')

    oof_nuisance_frame_t09 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_a_t09, 'fold': 'A', 'mu0_oof': mu0_oof_a_t09, 'mu1_oof': mu1_oof_a_t09}),
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_b_t09, 'fold': 'B', 'mu0_oof': mu0_oof_b_t09, 'mu1_oof': mu1_oof_b_t09}),
    ], ignore_index=True).sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    assert_full_oof_coverage(source_row_id_smoke_train_t09, oof_nuisance_frame_t09[SOURCE_ROW_ID].to_numpy(), name='mu0_oof/mu1_oof combined coverage')
    assert oof_nuisance_frame_t09['mu0_oof'].notna().all() and oof_nuisance_frame_t09['mu1_oof'].notna().all()
    oof_finite_t09 = bool(np.isfinite(oof_nuisance_frame_t09[['mu0_oof', 'mu1_oof']].to_numpy()).all())
    assert oof_finite_t09

    smoke_train_with_oof_t09 = smoke_train_frame_t09.merge(oof_nuisance_frame_t09, on=SOURCE_ROW_ID, how='inner', validate='one_to_one')
    assert len(smoke_train_with_oof_t09) == len(smoke_train_frame_t09), 'every training row must receive exactly one OOF mu0/mu1 pair'

    print(f'OOF nuisance assembled: {len(oof_nuisance_frame_t09):,} rows, both mu0_oof and mu1_oof present for every row.')
    print('Every OOF prediction confirmed from the opposite fold (never the row\'s own fold).')
    print('100% finite coverage:', oof_finite_t09)


### T09.5 OOF nuisance artifact + fold manifest

In [ ]:
if RUN_T09_SMOKE_STAGE:
    fold_manifest_frame_t09 = pd.DataFrame({SOURCE_ROW_ID: source_row_id_smoke_train_t09, 'fold': fold_label_smoke_train_t09})
    write_bytes_new(RUN_ROOT_T09_SMOKE, 'audit/xlearner_fold_manifest.parquet', fold_manifest_frame_t09.to_parquet(index=False))

    oof_nuisance_bytes_t09 = oof_nuisance_frame_t09.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_SMOKE, f'{XLEARNER_SEED_DIR_T09}/oof_nuisance.parquet', oof_nuisance_bytes_t09)
    oof_nuisance_sha256_t09 = hashlib.sha256(oof_nuisance_bytes_t09).hexdigest()
    print('Fold manifest and OOF nuisance parquet written.')


### T09.6 Pseudo-outcomes: D1 = Y - mu0_oof (treated), D0 = mu1_oof - Y (control)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    treatment_train_t09 = smoke_train_with_oof_t09[TREATMENT_COLUMN].astype('float64').to_numpy()
    outcome_train_t09 = smoke_train_with_oof_t09[PRIMARY_OUTCOME].astype('float64').to_numpy()
    mu0_oof_aligned_t09 = smoke_train_with_oof_t09['mu0_oof'].to_numpy()
    mu1_oof_aligned_t09 = smoke_train_with_oof_t09['mu1_oof'].to_numpy()

    d1_t09, d0_t09, treated_mask_train_t09, control_mask_train_t09 = compute_pseudo_outcomes(
        treatment_train_t09, outcome_train_t09, mu0_oof_aligned_t09, mu1_oof_aligned_t09,
    )

    assert len(d1_t09) == int(treated_mask_train_t09.sum())
    assert len(d0_t09) == int(control_mask_train_t09.sum())
    assert d1_t09.min() >= -1.0 and d1_t09.max() <= 1.0  # Y-mu0_oof, Y in {0,1}, mu0_oof in [0,1]
    assert d0_t09.min() >= -1.0 and d0_t09.max() <= 1.0  # mu1_oof-Y, symmetric bound
    print(f'D1 (treated, n={len(d1_t09):,}): min={d1_t09.min():.4f} max={d1_t09.max():.4f} mean={d1_t09.mean():.4f}')
    print(f'D0 (control, n={len(d0_t09):,}): min={d0_t09.min():.4f} max={d0_t09.max():.4f} mean={d0_t09.mean():.4f}')

    X_train_for_effect_t09 = transform_t09_smoke.transform(smoke_train_with_oof_t09)
    assert_model_feature_contract(X_train_for_effect_t09.columns)
    X_train_treated_effect_t09 = X_train_for_effect_t09.loc[treated_mask_train_t09].reset_index(drop=True)
    X_train_control_effect_t09 = X_train_for_effect_t09.loc[control_mask_train_t09].reset_index(drop=True)
    assert len(X_train_treated_effect_t09) == len(d1_t09)
    assert len(X_train_control_effect_t09) == len(d0_t09)


### T09.7 Fit tau1/tau0 (regression, fixed EFFECT_NUM_BOOST_ROUND, no early stopping)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    tau1_model_t09 = lgb_baseline.fit_regressor(X_train_treated_effect_t09, d1_t09, seed=MODEL_SEED_T09)
    tau0_model_t09 = lgb_baseline.fit_regressor(X_train_control_effect_t09, d0_t09, seed=MODEL_SEED_T09)
    assert tau1_model_t09.config_hash == tau0_model_t09.config_hash
    assert tau1_model_t09.num_boost_round == tau0_model_t09.num_boost_round == lgb_baseline.EFFECT_NUM_BOOST_ROUND
    print(f'tau1: rounds={tau1_model_t09.num_boost_round}, objective=regression, config_hash={tau1_model_t09.config_hash[:16]}...')
    print(f'tau0: rounds={tau0_model_t09.num_boost_round}, objective=regression, config_hash={tau0_model_t09.config_hash[:16]}...')


### T09.8 Score validation, compute g, combine tau_hat

In [ ]:
if RUN_T09_SMOKE_STAGE:
    tau1_hat_smoke_t09 = lgb_baseline.predict_values(tau1_model_t09, X_smoke_validation_t09)
    tau0_hat_smoke_t09 = lgb_baseline.predict_values(tau0_model_t09, X_smoke_validation_t09)
    assert len(tau1_hat_smoke_t09) == len(tau0_hat_smoke_t09) == len(smoke_validation_ids_t09)
    assert np.isfinite(tau1_hat_smoke_t09).all() and np.isfinite(tau0_hat_smoke_t09).all()

    g_t09 = empirical_treatment_rate(t_smoke_train_t09)
    print(f'g (training-only empirical treatment rate) = {g_t09:.6f}, source = smoke_train_t09 (n={len(t_smoke_train_t09):,})')

    tau_hat_smoke_t09 = combine_tau(tau1_hat_smoke_t09, tau0_hat_smoke_t09, g_t09)
    np.testing.assert_allclose(tau_hat_smoke_t09, g_t09 * tau0_hat_smoke_t09 + (1.0 - g_t09) * tau1_hat_smoke_t09)
    assert np.isfinite(tau_hat_smoke_t09).all()
    print(f'tau_hat range: [{tau_hat_smoke_t09.min():.4f}, {tau_hat_smoke_t09.max():.4f}]')


### T09.9 Evaluate via T06 only

In [ ]:
if RUN_T09_SMOKE_STAGE:
    xlearner_ranking_smoke_t09 = metrics.evaluate_ranking(
        tau_hat_smoke_t09, t_smoke_validation_t09, y_smoke_validation_arr_t09, source_row_id_smoke_validation_t09,
    )
    print(f'X-Learner SMOKE qini_above_random (non-substantive): {xlearner_ranking_smoke_t09.qini_above_random:.6f}')
    # metrics.random_ranking_reference_distribution() and lgb_baseline.fit_binary_classifier() are never
    # called here for Random/Response, and src.tlearner is never called -- T09 does not recompute T07/T08.


### T09.10 SMOKE artifacts

In [ ]:
if RUN_T09_SMOKE_STAGE:
    def _rows_with_run_context_t09(rows, population='smoke_200000_rows'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T09_SMOKE)
            row.setdefault('stage', 't09_smoke')
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_t09 = list(_rows_with_run_context_t09([
        {'method': 'xlearner', 'k': label, 'uplift': xlearner_ranking_smoke_t09.uplift_at_k[label],
         'incremental_conversions': xlearner_ranking_smoke_t09.incremental_conversions_at_k[label],
         'status': xlearner_ranking_smoke_t09.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_t09 = list(_rows_with_run_context_t09([{
        'ranking_method': 'xlearner',
        'qini_area': xlearner_ranking_smoke_t09.qini_area,
        'theoretical_random_qini_area': xlearner_ranking_smoke_t09.theoretical_random_qini_area,
        'qini_above_random': xlearner_ranking_smoke_t09.qini_above_random,
        'g_value': g_t09,
        'uplift_at_10pct': xlearner_ranking_smoke_t09.uplift_at_k['10pct'],
        'uplift_at_20pct': xlearner_ranking_smoke_t09.uplift_at_k['20pct'],
        'uplift_at_30pct': xlearner_ranking_smoke_t09.uplift_at_k['30pct'],
        'incremental_conversions_at_10pct': xlearner_ranking_smoke_t09.incremental_conversions_at_k['10pct'],
        'incremental_conversions_at_20pct': xlearner_ranking_smoke_t09.incremental_conversions_at_k['20pct'],
        'incremental_conversions_at_30pct': xlearner_ranking_smoke_t09.incremental_conversions_at_k['30pct'],
    }]))
    xlearner_deciles_rows_t09 = list(_rows_with_run_context_t09(xlearner_ranking_smoke_t09.decile_table.to_dict('records')))

    for _name, _rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_t09),
        ('tables/model_summary.csv', model_summary_rows_t09),
        ('tables/xlearner_deciles.csv', xlearner_deciles_rows_t09),
    ):
        write_text_new(RUN_ROOT_T09_SMOKE, _name, pd.DataFrame(_rows).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_frame_smoke_t09 = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation_t09,
        'tau1_hat': tau1_hat_smoke_t09,
        'tau0_hat': tau0_hat_smoke_t09,
        'g': g_t09,
        'tau_hat': tau_hat_smoke_t09,
    })
    xlearner_predictions_bytes_smoke_t09 = xlearner_predictions_frame_smoke_t09.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_SMOKE, f'{XLEARNER_SEED_DIR_T09}/validation_predictions.parquet', xlearner_predictions_bytes_smoke_t09)

    pseudo_outcomes_frame_t09 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_with_oof_t09.loc[treated_mask_train_t09, SOURCE_ROW_ID].to_numpy(), 'arm': 'treated', 'pseudo_effect': d1_t09}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_with_oof_t09.loc[control_mask_train_t09, SOURCE_ROW_ID].to_numpy(), 'arm': 'control', 'pseudo_effect': d0_t09}),
    ], ignore_index=True)
    pseudo_outcomes_bytes_t09 = pseudo_outcomes_frame_t09.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_SMOKE, f'{XLEARNER_SEED_DIR_T09}/pseudo_outcomes.parquet', pseudo_outcomes_bytes_t09)

    mu0_a_text_t09 = mu0_A_model_t09.booster.model_to_string()
    mu1_a_text_t09 = mu1_A_model_t09.booster.model_to_string()
    mu0_b_text_t09 = mu0_B_model_t09.booster.model_to_string()
    mu1_b_text_t09 = mu1_B_model_t09.booster.model_to_string()
    tau1_text_t09 = tau1_model_t09.booster.model_to_string()
    tau0_text_t09 = tau0_model_t09.booster.model_to_string()
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_mu0_A.txt', mu0_a_text_t09)
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_mu1_A.txt', mu1_a_text_t09)
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_mu0_B.txt', mu0_b_text_t09)
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_mu1_B.txt', mu1_b_text_t09)
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_tau1.txt', tau1_text_t09)
    write_text_new(RUN_ROOT_T09_SMOKE, 'models/xlearner_tau0.txt', tau0_text_t09)


    def _flatten_t09(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_t09 = metrics.metric_definitions()
    definitions_sha256_t09 = hashlib.sha256(json.dumps(definitions_t09, sort_keys=True).encode()).hexdigest()
    definitions_rows_t09 = list(_rows_with_run_context_t09(
        [{'field': k, 'value': _flatten_t09(v)} for k, v in definitions_t09.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_t09}]
    ))
    write_text_new(RUN_ROOT_T09_SMOKE, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_t09).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_sha256_smoke_t09 = hashlib.sha256(xlearner_predictions_bytes_smoke_t09).hexdigest()
    print('T09 SMOKE tables/models/predictions/audit written.')


### T09.11 Reload/reproducibility (all six models, plus combination reconciliation)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    reloaded_mu0_a_booster_t09 = lgb.Booster(model_str=mu0_a_text_t09)
    reloaded_mu1_a_booster_t09 = lgb.Booster(model_str=mu1_a_text_t09)
    reloaded_mu0_b_booster_t09 = lgb.Booster(model_str=mu0_b_text_t09)
    reloaded_mu1_b_booster_t09 = lgb.Booster(model_str=mu1_b_text_t09)
    reloaded_tau1_booster_t09 = lgb.Booster(model_str=tau1_text_t09)
    reloaded_tau0_booster_t09 = lgb.Booster(model_str=tau0_text_t09)

    X_smoke_validation_rebuilt_t09 = transform_t09_smoke.transform(smoke_validation_frame_t09)
    X_A_full_rebuilt_t09 = transform_t09_smoke.transform(smoke_train_frame_t09.loc[mask_fold_a_t09].reset_index(drop=True))
    X_B_full_rebuilt_t09 = transform_t09_smoke.transform(smoke_train_frame_t09.loc[mask_fold_b_t09].reset_index(drop=True))

    mu0_oof_a_reloaded_t09 = np.asarray(reloaded_mu0_b_booster_t09.predict(X_A_full_rebuilt_t09, num_iteration=mu0_B_model_t09.best_iteration), dtype=np.float64)
    mu1_oof_a_reloaded_t09 = np.asarray(reloaded_mu1_b_booster_t09.predict(X_A_full_rebuilt_t09, num_iteration=mu1_B_model_t09.best_iteration), dtype=np.float64)
    mu0_oof_b_reloaded_t09 = np.asarray(reloaded_mu0_a_booster_t09.predict(X_B_full_rebuilt_t09, num_iteration=mu0_A_model_t09.best_iteration), dtype=np.float64)
    mu1_oof_b_reloaded_t09 = np.asarray(reloaded_mu1_a_booster_t09.predict(X_B_full_rebuilt_t09, num_iteration=mu1_A_model_t09.best_iteration), dtype=np.float64)

    oof_reload_matches_t09 = bool(
        np.allclose(mu0_oof_a_reloaded_t09, mu0_oof_a_t09, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_a_reloaded_t09, mu1_oof_a_t09, rtol=1e-6, atol=1e-8)
        and np.allclose(mu0_oof_b_reloaded_t09, mu0_oof_b_t09, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_b_reloaded_t09, mu1_oof_b_t09, rtol=1e-6, atol=1e-8)
    )

    tau1_reloaded_t09 = np.asarray(reloaded_tau1_booster_t09.predict(X_smoke_validation_rebuilt_t09, num_iteration=tau1_model_t09.num_boost_round), dtype=np.float64)
    tau0_reloaded_t09 = np.asarray(reloaded_tau0_booster_t09.predict(X_smoke_validation_rebuilt_t09, num_iteration=tau0_model_t09.num_boost_round), dtype=np.float64)
    tau1_reload_matches_t09 = bool(np.allclose(tau1_reloaded_t09, tau1_hat_smoke_t09, rtol=1e-6, atol=1e-8))
    tau0_reload_matches_t09 = bool(np.allclose(tau0_reloaded_t09, tau0_hat_smoke_t09, rtol=1e-6, atol=1e-8))

    tau_hat_reloaded_t09 = combine_tau(tau1_reloaded_t09, tau0_reloaded_t09, g_t09)
    tau_hat_reload_matches_t09 = bool(np.allclose(tau_hat_reloaded_t09, tau_hat_smoke_t09, rtol=1e-6, atol=1e-8))

    row_identity_matches_t09 = set(source_row_id_smoke_validation_t09.tolist()) == set(smoke_validation_ids_t09.tolist())

    reload_verification_t09_smoke = {
        'config_hash_matches': bool(
            mu0_A_model_t09.config_hash == mu1_A_model_t09.config_hash == mu0_B_model_t09.config_hash
            == mu1_B_model_t09.config_hash == lgb_baseline.config_hash()
        ),
        'regression_config_hash_matches': bool(tau1_model_t09.config_hash == tau0_model_t09.config_hash == lgb_baseline.regression_config_hash()),
        'oof_reload_matches_within_tolerance': oof_reload_matches_t09,
        'tau1_reload_matches_within_tolerance': tau1_reload_matches_t09,
        'tau0_reload_matches_within_tolerance': tau0_reload_matches_t09,
        'tau_hat_reload_matches_within_tolerance': tau_hat_reload_matches_t09,
        'row_identity_matches': row_identity_matches_t09,
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(reload_verification_t09_smoke)
    assert all([
        reload_verification_t09_smoke['config_hash_matches'],
        reload_verification_t09_smoke['regression_config_hash_matches'],
        reload_verification_t09_smoke['oof_reload_matches_within_tolerance'],
        reload_verification_t09_smoke['tau1_reload_matches_within_tolerance'],
        reload_verification_t09_smoke['tau0_reload_matches_within_tolerance'],
        reload_verification_t09_smoke['tau_hat_reload_matches_within_tolerance'],
        reload_verification_t09_smoke['row_identity_matches'],
    ])


### T09.11b Correctness audit artifact (required by the X-Learner artifact contract)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    xlearner_correctness_t09 = {
        'run_id': RUN_ID_T09_SMOKE,
        'stage': 't09_smoke',
        'fold_assignment': {
            'fold_count': 2,
            'fold_a_count': int(len(fold_a_ids_t09)),
            'fold_b_count': int(len(fold_b_ids_t09)),
            'folds_disjoint': bool(set(fold_a_ids_t09).isdisjoint(set(fold_b_ids_t09))),
            'folds_cover_all_training_rows': bool(set(fold_a_ids_t09) | set(fold_b_ids_t09) == set(source_row_id_smoke_train_t09.tolist())),
            'fold_seed': FOLD_SEED_T09,
        },
        'oof_nuisance': {
            'rows': int(len(oof_nuisance_frame_t09)),
            'full_coverage_both_surfaces': True,  # assert_full_oof_coverage passed for the combined mu0_oof/mu1_oof set above; a failure would have raised
            'opposite_fold_only_both_surfaces': True,  # assert_opposite_fold passed for both mu0_oof and mu1_oof above; a failure would have raised
            'finite_coverage_fraction': 1.0 if oof_finite_t09 else float(np.isfinite(oof_nuisance_frame_t09[['mu0_oof', 'mu1_oof']].to_numpy()).mean()),
        },
        'pseudo_outcomes': {
            'd1_support': int(len(d1_t09)), 'd1_min': float(d1_t09.min()), 'd1_max': float(d1_t09.max()), 'd1_mean': float(d1_t09.mean()),
            'd0_support': int(len(d0_t09)), 'd0_min': float(d0_t09.min()), 'd0_max': float(d0_t09.max()), 'd0_mean': float(d0_t09.mean()),
            'd1_within_theoretical_bounds': bool(d1_t09.min() >= -1.0 and d1_t09.max() <= 1.0),
            'd0_within_theoretical_bounds': bool(d0_t09.min() >= -1.0 and d0_t09.max() <= 1.0),
            'd1_treated_row_count_matches_arm_partition': bool(int(len(d1_t09)) == int(treated_mask_train_t09.sum())),
            'd0_control_row_count_matches_arm_partition': bool(int(len(d0_t09)) == int(control_mask_train_t09.sum())),
        },
        'effect_stage': {
            'objective': 'regression',
            'num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
            'early_stopping': False,
            'tau_hat_finite': bool(np.isfinite(tau_hat_smoke_t09).all()),
            'g_value': g_t09,
            'g_source': 'training_only_empirical_treatment_rate',
        },
        'reload_verification': reload_verification_t09_smoke,
    }
    write_json_new(RUN_ROOT_T09_SMOKE, 'audit/xlearner_correctness.json', xlearner_correctness_t09)
    print('T09 SMOKE audit/xlearner_correctness.json written.')


### T09.12 T07/T08 reuse verification (no recomputation)

In [ ]:
if RUN_T09_SMOKE_STAGE:
    t09_upstream_reuse_hashes = {}
    for _label, _root, _hashes, _rel_path in (
        ('t07_full', t07_full_root, t07_artifact_hashes, 'tables/model_summary.csv'),
        ('t08_full', t08_full_root_ref, t08_full_artifact_hashes_ref, 'tables/model_summary.csv'),
    ):
        _actual = hashlib.sha256((_root / _rel_path).read_bytes()).hexdigest()
        _expected = _hashes[_rel_path]
        t09_upstream_reuse_hashes[_label] = {'path': _rel_path, 'expected': _expected, 'actual': _actual, 'matches': _actual == _expected}
        assert _actual == _expected, f'{_label} artifact {_rel_path} hash mismatch -- refusing to cite unverified evidence'

    t09_upstream_recomputation_guard = {
        'RUN_T07_STAGE': RUN_T07_STAGE,
        'RUN_T08_SMOKE_STAGE': RUN_T08_SMOKE_STAGE,
        'RUN_T08_FULL_STAGE': RUN_T08_FULL_STAGE,
    }
    assert not any(t09_upstream_recomputation_guard.values()), 'T09 must not trigger any T07/T08 recomputation'
    print('T07/T08 reuse hash verification (all must match, none recomputed):')
    for _label, _info in t09_upstream_reuse_hashes.items():
        print('  ' + _label + ': matches=' + str(_info['matches']))
    print('Recomputation guards (all must be False):', t09_upstream_recomputation_guard)


In [ ]:
if RUN_T09_SMOKE_STAGE:
    t09_smoke_max_rss_bytes_observed = t09_smoke_rss_sampler.stop()
    t09_smoke_wall_seconds = __import__('time').perf_counter() - t09_smoke_wall_start

    t09_smoke_resource_evidence = {
        'wall_seconds': t09_smoke_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t09_smoke_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t09_smoke_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t09_smoke_max_rss_bytes_observed) - int(t09_smoke_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(t09_smoke_resource_evidence)


In [ ]:
if RUN_T09_SMOKE_STAGE:
    write_json_new(RUN_ROOT_T09_SMOKE, 'audit/environment.json', {
        'run_id': RUN_ID_T09_SMOKE,
        'created_at_utc': t09_smoke_started.isoformat(),
        'git_head': t09_git_head,
        'git_dirty': t09_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T09_SMOKE, 'audit/run_config.json', {
        'run_id': RUN_ID_T09_SMOKE,
        'created_at_utc': t09_smoke_started.isoformat(),
        'stage': 't09_smoke',
        'population': 'smoke_200000_rows',
        'git_head': t09_git_head,
        'git_dirty': t09_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_xlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t09_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t09_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        't07_t08_reuse': {
            't07_full_run_id': t09_config['input']['t07_evidence']['authoritative_t07_full_run_id'],
            't08_full_run_id': t09_config['input']['t08_evidence']['authoritative_t08_full_run_id'],
            'hash_verification': t09_upstream_reuse_hashes,
            'recomputation_guards': t09_upstream_recomputation_guard,
            'recomputed': False,
        },
        'scale_gating': {
            'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES_T09,
            'smoke_size': SMOKE_SIZE_T09, 'smoke_seed': SMOKE_SEED_T09,
            'fold_seed': FOLD_SEED_T09, 'model_seed': MODEL_SEED_T09,
        },
        'xlearner_design': t09_config['xlearner_design'],
        'lightgbm_binary_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_regression_config': lgb_baseline.FROZEN_REGRESSION_CONFIG,
        'effect_num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
        'nuisance_config_hash': mu0_A_model_t09.config_hash,
        'regression_config_hash': tau1_model_t09.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_t09.best_iteration, 'mu1_A': mu1_A_model_t09.best_iteration,
            'mu0_B': mu0_B_model_t09.best_iteration, 'mu1_B': mu1_B_model_t09.best_iteration,
        },
        'g_value': g_t09,
        'g_source': 'training-only empirical treatment rate over smoke_train_t09',
        'oof_nuisance_sha256': oof_nuisance_sha256_t09,
        'xlearner_predictions_sha256': xlearner_predictions_sha256_smoke_t09,
        'definitions_sha256': definitions_sha256_t09,
        'resource_evidence': t09_smoke_resource_evidence,
        'reload_verification': reload_verification_t09_smoke,
    })
    print('T09 SMOKE run_config.json / environment.json written.')


In [ ]:
if RUN_T09_SMOKE_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T09_SMOKE,
        run_id=RUN_ID_T09_SMOKE,
        final_status='COMPLETED_T09_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t09_smoke',
        population='smoke_200000_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/xlearner.py', 'role': 'reusable_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t07_evidence']['authoritative_t07_full_run_id']}", 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t08_evidence']['authoritative_t08_full_run_id']}", 'role': 'reused_t08_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'T09 SMOKE run finalized: {RUN_ID_T09_SMOKE}')

    immutable_write_refused_t09_smoke = False
    try:
        write_json_new(RUN_ROOT_T09_SMOKE, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t09_smoke = True
    except Exception:
        immutable_write_refused_t09_smoke = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t09_smoke}')
    assert immutable_write_refused_t09_smoke


In [ ]:
if RUN_T09_SMOKE_STAGE:
    t09_smoke_summary = {
        'run_id': RUN_ID_T09_SMOKE,
        'smoke_total': int(smoke_total_t09),
        'smoke_train_count': int(len(smoke_train_ids_t09)),
        'smoke_validation_count': int(len(smoke_validation_ids_t09)),
        'fold_a_count': int(len(fold_a_ids_t09)),
        'fold_b_count': int(len(fold_b_ids_t09)),
        'all_partition_ty_cells_nonempty': bool(t09_smoke_all_cells_nonempty),
        'train_support': smoke_train_support_t09,
        'validation_support': smoke_validation_support_t09,
        'lightgbm_version': lgb.__version__,
        'nuisance_config_hash': mu0_A_model_t09.config_hash,
        'regression_config_hash': tau1_model_t09.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_t09.best_iteration, 'mu1_A': mu1_A_model_t09.best_iteration,
            'mu0_B': mu0_B_model_t09.best_iteration, 'mu1_B': mu1_B_model_t09.best_iteration,
        },
        'effect_num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
        'g_value': g_t09,
        'd1_support': int(len(d1_t09)), 'd1_range': [float(d1_t09.min()), float(d1_t09.max())],
        'd0_support': int(len(d0_t09)), 'd0_range': [float(d0_t09.min()), float(d0_t09.max())],
        'tau_hat_range': [float(tau_hat_smoke_t09.min()), float(tau_hat_smoke_t09.max())],
        'xlearner_ranking_smoke_non_substantive': {
            'qini_area': xlearner_ranking_smoke_t09.qini_area,
            'qini_above_random': xlearner_ranking_smoke_t09.qini_above_random,
        },
        'reload_verification': reload_verification_t09_smoke,
        't07_t08_reuse_hash_verification': t09_upstream_reuse_hashes,
        'resource_evidence': t09_smoke_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t09_smoke,
    }
    print(json.dumps(t09_smoke_summary, indent=2, default=str))


In [ ]:
RUN_T09_FULL_STAGE = False  # T09 FULL (seeds 42/123/2026) is already accepted.
T09_FULL_SEED42_RUN_ID_ACCEPTED = 't09_full_seed42_20260819T041828Z_831364'
T09_FULL_SEED123_RUN_ID_ACCEPTED = 't09_full_seed123_20260819T042330Z_632997'
T09_FULL_SEED2026_RUN_ID_ACCEPTED = 't09_full_seed2026_20260819T042825Z_929867'
T09_FULL_REPEATED_SEED_RUN_ID_ACCEPTED = 't09_full_seed_summary_20260819T043346Z_178791'
T09_FULL_SUPERSEDED_SEED42_RUN_ID = 't09_full_seed42_20260819T040358Z_050552'  # first attempt, valid, superseded for consistency
T09_FULL_SUPERSEDED_SEED123_RUN_ID = 't09_full_seed123_20260819T040825Z_847469'  # first attempt, hit a diagnostic-only bug, left untouched

if not RUN_T09_FULL_STAGE:
    t09_full_accepted_run_ids = {
        42: T09_FULL_SEED42_RUN_ID_ACCEPTED,
        123: T09_FULL_SEED123_RUN_ID_ACCEPTED,
        2026: T09_FULL_SEED2026_RUN_ID_ACCEPTED,
    }
    t09_full_accepted_roots = {}
    t09_full_accepted_hashes = {}
    for _seed, _run_id in t09_full_accepted_run_ids.items():
        _root = REPO_ROOT / 'outputs' / 'runs' / _run_id
        _manifest_path = _root / 'audit' / 'artifact_manifest.json'
        if not _manifest_path.is_file():
            raise RuntimeError(
                f'RUN_T09_FULL_STAGE is False, but the accepted T09 FULL seed {_seed} run evidence was '
                f'not found at {_root}. Refusing to silently set RUN_T09_FULL_STAGE = True and refit -- '
                f'either provide/mount the accepted governed run evidence (outputs/runs/{_run_id}/), or '
                f'explicitly set RUN_T09_FULL_STAGE = True in this cell to opt into reproducing T09 FULL '
                f'from scratch.'
            )
        _manifest = json.loads(_manifest_path.read_text(encoding='utf-8'))
        _hashes = {a['path']: a['sha256'] for a in _manifest['artifacts']}
        _actual = hashlib.sha256((_root / 'tables' / 'model_summary.csv').read_bytes()).hexdigest()
        _expected = _hashes.get('tables/model_summary.csv')
        if _expected is None or _actual != _expected:
            raise RuntimeError(
                f'T09 FULL seed {_seed} tables/model_summary.csv hash does not match its own artifact '
                f'manifest (expected {_expected}, actual {_actual}). Refusing to reuse unverified evidence.'
            )
        t09_full_accepted_roots[_seed] = _root
        t09_full_accepted_hashes[_seed] = _hashes

    t09_repeated_seed_root_ref = REPO_ROOT / 'outputs' / 'runs' / T09_FULL_REPEATED_SEED_RUN_ID_ACCEPTED
    t09_repeated_seed_manifest_path = t09_repeated_seed_root_ref / 'audit' / 'artifact_manifest.json'
    if not t09_repeated_seed_manifest_path.is_file():
        raise RuntimeError(
            f'RUN_T09_FULL_STAGE is False, but the accepted repeated-seed summary evidence was not found '
            f'at {t09_repeated_seed_root_ref}.'
        )
    t09_repeated_seed_manifest_ref = json.loads(t09_repeated_seed_manifest_path.read_text(encoding='utf-8'))
    t09_repeated_seed_hashes_ref = {a['path']: a['sha256'] for a in t09_repeated_seed_manifest_ref['artifacts']}

    print(f'T09 FULL already accepted at run_ids {t09_full_accepted_run_ids} (hash-verified); skipping refit.')
    print(f'Repeated-seed summary already accepted at {T09_FULL_REPEATED_SEED_RUN_ID_ACCEPTED} (hash-verified).')
    print(f'Superseded first-attempt runs (kept, not deleted): {T09_FULL_SUPERSEDED_SEED42_RUN_ID}, {T09_FULL_SUPERSEDED_SEED123_RUN_ID}')
else:
    print('RUN_T09_FULL_STAGE = True: all three T09 FULL seeds will be recomputed from scratch below.')


## Scale-Gating (D30): T09 FULL

SMOKE (mechanism and, after correction, artifact-path) both passed. Under D30,
T09's approved path is `SMOKE -> FULL` with `resource_gates = 0`. FULL uses the
complete frozen T05 training partition (9,785,714 rows) and validation
partition (2,096,938 rows). The two-fold cross-fitting assignment and the
full-training propensity constant `g_full` are computed **once** and reused
identically across all three required model seeds -- primary `42` and
robustness `123`/`2026` (`docs/06_experiment_protocol.md` Stage 3). Only
LightGBM's own `seed` parameter changes between the three governed runs; fold
membership, `g_full`, and every configuration value stay fixed. Each seed
mints its own immutable `outputs/runs/<run_id>/` root -- required because
`models/xlearner_<component>.txt` carries no seed segment in the artifact
contract, so three seeds' serialized components cannot share one run root.
Seed 42 is evaluated for correctness/resource/artifact soundness before the
robustness seeds are even attempted; a failure there stops before seed 123/2026
are reached. Held-out remains completely sealed throughout.


*(This section -- T09 FULL, all three model seeds -- is already accepted. It is skipped by default on Run All; see `RUN_T09_FULL_STAGE` immediately above. Set it to `True` only to deliberately reproduce all three FULL seed fits from scratch.)*

In [ ]:
if RUN_T09_FULL_STAGE:
    from src.xlearner import (
        XLearnerContractError, assert_full_oof_coverage, assert_opposite_fold,
        assign_folds, combine_tau, compute_pseudo_outcomes, empirical_treatment_rate,
    )
    import src.xlearner as xlearner_module
    import threading

    # Re-established here because the T09 SMOKE section's own setup cell (which
    # also imports/defines these) is now guarded behind RUN_T09_SMOKE_STAGE and is
    # skipped by default -- FULL cannot rely on names defined only inside a
    # skipped guarded block.
    t09_config = json.loads((REPO_ROOT / 'configs' / 't09_xlearner.json').read_text(encoding='utf-8'))
    FOLD_SEED_T09 = t09_config['scale_gating']['fold_seed']
    RESOURCE_GATES_T09 = t09_config['scale_gating']['resource_gates']


    class _PeakRSSSampler:
        # Continuous stage-scoped sampler: polls process.memory_info().rss at a
        # fixed interval and tracks the running maximum observed strictly within
        # this sampler's own start()/stop() window -- a genuinely valid mechanism
        # under ADR-experiment-artifacts' resource-measurement rule (distinct from
        # a same-process before/after peak_wset read, which is not).

        def __init__(self, proc, interval_seconds=1.0):
            self._proc = proc
            self._interval = interval_seconds
            self._max_rss_bytes = proc.memory_info().rss
            self._stop_event = threading.Event()
            self._thread = None

        def _run(self):
            while not self._stop_event.is_set():
                rss = self._proc.memory_info().rss
                if rss > self._max_rss_bytes:
                    self._max_rss_bytes = rss
                self._stop_event.wait(self._interval)

        def start(self):
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
            return self

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=5)
            return self._max_rss_bytes


    print(f'T09 FULL setup: fold_seed={FOLD_SEED_T09}, resource_gates={RESOURCE_GATES_T09}')


In [ ]:
if RUN_T09_FULL_STAGE:
    full_train_frame_t09_full = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    full_validation_frame_t09_full = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(full_train_frame_t09_full) == len(train_ids_full)
    assert len(full_validation_frame_t09_full) == len(validation_ids_full)

    transform_t09_full = IdentityFeatureTransform()
    X_full_train_t09_full = transform_t09_full.fit_transform(full_train_frame_t09_full)
    X_full_validation_t09_full = transform_t09_full.transform(full_validation_frame_t09_full)
    assert_model_feature_contract(X_full_train_t09_full.columns)
    assert_model_feature_contract(X_full_validation_t09_full.columns)

    y_full_train_t09_full = full_train_frame_t09_full[PRIMARY_OUTCOME].astype('float64')
    y_full_validation_t09_full = full_validation_frame_t09_full[PRIMARY_OUTCOME].astype('float64')
    t_full_train_t09_full = full_train_frame_t09_full[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_full_validation_t09_full = full_validation_frame_t09_full[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_full_train_arr_t09_full = y_full_train_t09_full.to_numpy()
    y_full_validation_arr_t09_full = y_full_validation_t09_full.to_numpy()
    source_row_id_full_train_t09_full = full_train_frame_t09_full[SOURCE_ROW_ID].to_numpy()
    source_row_id_full_validation_t09_full = full_validation_frame_t09_full[SOURCE_ROW_ID].to_numpy()

    print(f'X_full_train_t09_full: {X_full_train_t09_full.shape}, X_full_validation_t09_full: {X_full_validation_t09_full.shape}')


### T09.FULL.0 Shared two-fold cross-fitting assignment (fixed across all three model seeds)

In [ ]:
if RUN_T09_FULL_STAGE:
    fold_a_ids_full_t09, fold_b_ids_full_t09 = assign_folds(
        source_row_id_full_train_t09_full, t_full_train_t09_full, y_full_train_arr_t09_full, seed=FOLD_SEED_T09,
    )
    assert set(fold_a_ids_full_t09).isdisjoint(set(fold_b_ids_full_t09))
    assert set(fold_a_ids_full_t09) | set(fold_b_ids_full_t09) == set(source_row_id_full_train_t09_full.tolist())

    fold_label_full_train_t09 = np.where(np.isin(source_row_id_full_train_t09_full, fold_a_ids_full_t09), 'A', 'B')
    mask_fold_a_full_t09 = fold_label_full_train_t09 == 'A'
    mask_fold_b_full_t09 = ~mask_fold_a_full_t09

    fold_assignment_row_ids_sha256_t09_full = hashlib.sha256(
        np.sort(source_row_id_full_train_t09_full[mask_fold_a_full_t09]).astype('<i8').tobytes()
        + b'|'
        + np.sort(source_row_id_full_train_t09_full[mask_fold_b_full_t09]).astype('<i8').tobytes()
    ).hexdigest()

    print(f'Total FULL train: {len(source_row_id_full_train_t09_full):,}')
    print(f'Fold A: {len(fold_a_ids_full_t09):,} rows, Fold B: {len(fold_b_ids_full_t09):,} rows')
    print(f'fold_assignment_row_ids_sha256 (recorded per seed run, for cross-seed drift verification): {fold_assignment_row_ids_sha256_t09_full}')


In [ ]:
if RUN_T09_FULL_STAGE:
    def fold_ty_support_t09_full(mask, frame):
        subset = frame.loc[mask]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    fold_a_ty_support_full_t09 = fold_ty_support_t09_full(mask_fold_a_full_t09, full_train_frame_t09_full)
    fold_b_ty_support_full_t09 = fold_ty_support_t09_full(mask_fold_b_full_t09, full_train_frame_t09_full)
    validation_ty_support_full_t09 = fold_ty_support_t09_full(np.ones(len(full_validation_frame_t09_full), dtype=bool), full_validation_frame_t09_full)

    print('Fold A (T,Y) support:', fold_a_ty_support_full_t09)
    print('Fold B (T,Y) support:', fold_b_ty_support_full_t09)
    print('Validation (T,Y) support:', validation_ty_support_full_t09)

    t09_full_all_cells_nonempty = all(n > 0 for n in {**fold_a_ty_support_full_t09, **fold_b_ty_support_full_t09, **validation_ty_support_full_t09}.values())
    print('All fold-A/fold-B/validation x (T,Y) cells non-empty:', t09_full_all_cells_nonempty)
    if not t09_full_all_cells_nonempty:
        raise XLearnerContractError('T09 FULL fold/validation (T,Y) support is degenerate -- refusing to proceed.')


### T09.FULL.0b Shared per-fold, per-arm partitions + arm-relevant validation subsets

In [ ]:
if RUN_T09_FULL_STAGE:
    X_full_train_arr_t09_full = X_full_train_t09_full.to_numpy()


    def _fold_arm_frame_full_t09(mask_fold, arm_value):
        mask = mask_fold & (t_full_train_t09_full == arm_value)
        X = pd.DataFrame(X_full_train_arr_t09_full[mask], columns=X_full_train_t09_full.columns)
        y = y_full_train_arr_t09_full[mask]
        return X, y


    X_A_treated_full_t09, y_A_treated_full_t09 = _fold_arm_frame_full_t09(mask_fold_a_full_t09, 1.0)
    X_A_control_full_t09, y_A_control_full_t09 = _fold_arm_frame_full_t09(mask_fold_a_full_t09, 0.0)
    X_B_treated_full_t09, y_B_treated_full_t09 = _fold_arm_frame_full_t09(mask_fold_b_full_t09, 1.0)
    X_B_control_full_t09, y_B_control_full_t09 = _fold_arm_frame_full_t09(mask_fold_b_full_t09, 0.0)

    for _name, _X in (('A_treated', X_A_treated_full_t09), ('A_control', X_A_control_full_t09), ('B_treated', X_B_treated_full_t09), ('B_control', X_B_control_full_t09)):
        assert len(_X) > 0, f'T09 FULL fold/arm cell {_name} is empty'
        print(f'{_name}: {len(_X):,} rows')

    mask_val_treated_full_t09 = t_full_validation_t09_full == 1
    mask_val_control_full_t09 = t_full_validation_t09_full == 0
    X_val_treated_full_t09 = X_full_validation_t09_full.loc[mask_val_treated_full_t09].reset_index(drop=True)
    y_val_treated_full_t09 = y_full_validation_arr_t09_full[mask_val_treated_full_t09]
    X_val_control_full_t09 = X_full_validation_t09_full.loc[mask_val_control_full_t09].reset_index(drop=True)
    y_val_control_full_t09 = y_full_validation_arr_t09_full[mask_val_control_full_t09]
    assert len(X_val_treated_full_t09) > 0 and len(X_val_control_full_t09) > 0
    print(f'Validation (arm-relevant, shared across all seeds/folds): treated={len(X_val_treated_full_t09):,} control={len(X_val_control_full_t09):,}')

    X_A_full_t09_full = pd.DataFrame(X_full_train_arr_t09_full[mask_fold_a_full_t09], columns=X_full_train_t09_full.columns)
    X_B_full_t09_full = pd.DataFrame(X_full_train_arr_t09_full[mask_fold_b_full_t09], columns=X_full_train_t09_full.columns)
    source_row_id_fold_a_full_t09 = source_row_id_full_train_t09_full[mask_fold_a_full_t09]
    source_row_id_fold_b_full_t09 = source_row_id_full_train_t09_full[mask_fold_b_full_t09]

    fold_manifest_frame_full_t09 = pd.DataFrame({SOURCE_ROW_ID: source_row_id_full_train_t09_full, 'fold': fold_label_full_train_t09})


### T09.FULL.0c Full-training g (complete frozen T05 training population)

In [ ]:
if RUN_T09_FULL_STAGE:
    g_full_n_treated_t09 = int(t_full_train_t09_full.sum())
    g_full_n_train_t09 = int(len(t_full_train_t09_full))
    g_full_t09 = empirical_treatment_rate(t_full_train_t09_full)

    # Independent sanity recomputation (not via the shared helper) must agree exactly.
    g_full_direct_t09 = g_full_n_treated_t09 / g_full_n_train_t09
    assert g_full_t09 == g_full_direct_t09, 'empirical_treatment_rate() disagrees with a direct recomputation of g_full'
    assert g_full_n_treated_t09 + int((t_full_train_t09_full == 0).sum()) == g_full_n_train_t09

    print(f'g_full = n_treated_train / n_train = {g_full_n_treated_t09:,} / {g_full_n_train_t09:,} = {g_full_t09:.10f}')
    print('This g_full is fixed and reused identically across model seeds 42, 123, and 2026 -- never SMOKE\'s g.')


### T09.FULL Seed 42 (primary reported result)

Fold membership, `g_full`, and every configuration value are identical to the
other seeds above/below -- only LightGBM's `seed` parameter changes.

In [ ]:
if RUN_T09_FULL_STAGE:
    MODEL_SEED_T09_FULL_42 = 42

    t09_full_s42_started = datetime.now(timezone.utc)
    t09_full_s42_wall_start = __import__('time').perf_counter()
    t09_full_s42_baseline_rss_bytes = process.memory_info().rss
    t09_full_s42_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T09_FULL_S42 = t09_full_s42_started.strftime('t09_full_seed42_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T09_FULL_S42 = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_FULL_S42
    RUN_ROOT_T09_FULL_S42.mkdir(parents=True, exist_ok=False)
    XLEARNER_SEED_DIR_T09_FULL_S42 = 'predictions/development/xlearner/seed_42'

    try:
        t09_full_s42_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t09_full_s42_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t09_full_s42_git_head, t09_full_s42_git_dirty = None, None

    print(f'RUN_ID_T09_FULL_S42 = {RUN_ID_T09_FULL_S42}')
    print(f'model_seed = 42, fold_seed = {FOLD_SEED_T09}, g_full = {g_full_t09:.6f}')


#### Seed 42: four fold-specific nuisance fits

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_A_model_full_s42 = lgb_baseline.fit_binary_classifier(X_A_control_full_t09, y_A_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_42)
    mu1_A_model_full_s42 = lgb_baseline.fit_binary_classifier(X_A_treated_full_t09, y_A_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_42)
    mu0_B_model_full_s42 = lgb_baseline.fit_binary_classifier(X_B_control_full_t09, y_B_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_42)
    mu1_B_model_full_s42 = lgb_baseline.fit_binary_classifier(X_B_treated_full_t09, y_B_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_42)

    assert mu0_A_model_full_s42.config_hash == mu1_A_model_full_s42.config_hash == mu0_B_model_full_s42.config_hash == mu1_B_model_full_s42.config_hash
    for _name, _model in (('mu0_A', mu0_A_model_full_s42), ('mu1_A', mu1_A_model_full_s42), ('mu0_B', mu0_B_model_full_s42), ('mu1_B', mu1_B_model_full_s42)):
        print(f'seed 42 {_name}: best_iteration={_model.best_iteration}, config_hash={_model.config_hash[:16]}...')


#### Seed 42: out-of-fold assembly + corrected artifact paths

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_oof_a_full_s42 = lgb_baseline.predict_probabilities(mu0_B_model_full_s42, X_A_full_t09_full)
    mu1_oof_a_full_s42 = lgb_baseline.predict_probabilities(mu1_B_model_full_s42, X_A_full_t09_full)
    mu0_oof_b_full_s42 = lgb_baseline.predict_probabilities(mu0_A_model_full_s42, X_B_full_t09_full)
    mu1_oof_b_full_s42 = lgb_baseline.predict_probabilities(mu1_A_model_full_s42, X_B_full_t09_full)

    row_fold_unsorted_full_s42 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'A'), np.full(len(source_row_id_fold_b_full_t09), 'B')])
    prediction_fold_of_mu0_full_s42 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'B'), np.full(len(source_row_id_fold_b_full_t09), 'A')])
    prediction_fold_of_mu1_full_s42 = prediction_fold_of_mu0_full_s42.copy()
    assert_opposite_fold(row_fold_unsorted_full_s42, prediction_fold_of_mu0_full_s42, name='mu0_oof seed 42')
    assert_opposite_fold(row_fold_unsorted_full_s42, prediction_fold_of_mu1_full_s42, name='mu1_oof seed 42')

    oof_nuisance_frame_full_s42 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_a_full_t09, 'fold': 'A', 'mu0_oof': mu0_oof_a_full_s42, 'mu1_oof': mu1_oof_a_full_s42}),
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_b_full_t09, 'fold': 'B', 'mu0_oof': mu0_oof_b_full_s42, 'mu1_oof': mu1_oof_b_full_s42}),
    ], ignore_index=True).sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    assert_full_oof_coverage(source_row_id_full_train_t09_full, oof_nuisance_frame_full_s42[SOURCE_ROW_ID].to_numpy(), name=f'seed 42 mu0_oof/mu1_oof combined coverage')
    assert oof_nuisance_frame_full_s42['mu0_oof'].notna().all() and oof_nuisance_frame_full_s42['mu1_oof'].notna().all()
    oof_finite_full_s42 = bool(np.isfinite(oof_nuisance_frame_full_s42[['mu0_oof', 'mu1_oof']].to_numpy()).all())
    assert oof_finite_full_s42

    full_train_with_oof_s42 = full_train_frame_t09_full.merge(oof_nuisance_frame_full_s42, on=SOURCE_ROW_ID, how='inner', validate='one_to_one')
    assert len(full_train_with_oof_s42) == len(full_train_frame_t09_full)

    write_bytes_new(RUN_ROOT_T09_FULL_S42, 'audit/xlearner_fold_manifest.parquet', fold_manifest_frame_full_t09.to_parquet(index=False))
    oof_nuisance_bytes_full_s42 = oof_nuisance_frame_full_s42.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S42, f'{XLEARNER_SEED_DIR_T09_FULL_S42}/oof_nuisance.parquet', oof_nuisance_bytes_full_s42)

    print(f'seed 42 OOF nuisance assembled: {len(oof_nuisance_frame_full_s42):,} rows, 100% coverage, 0% same-fold leakage.')


#### Seed 42: pseudo-outcomes

In [ ]:
if RUN_T09_FULL_STAGE:
    treatment_train_full_s42 = full_train_with_oof_s42[TREATMENT_COLUMN].astype('float64').to_numpy()
    outcome_train_full_s42 = full_train_with_oof_s42[PRIMARY_OUTCOME].astype('float64').to_numpy()
    mu0_oof_aligned_full_s42 = full_train_with_oof_s42['mu0_oof'].to_numpy()
    mu1_oof_aligned_full_s42 = full_train_with_oof_s42['mu1_oof'].to_numpy()

    d1_full_s42, d0_full_s42, treated_mask_train_full_s42, control_mask_train_full_s42 = compute_pseudo_outcomes(
        treatment_train_full_s42, outcome_train_full_s42, mu0_oof_aligned_full_s42, mu1_oof_aligned_full_s42,
    )
    assert len(d1_full_s42) == int(treated_mask_train_full_s42.sum())
    assert len(d0_full_s42) == int(control_mask_train_full_s42.sum())
    assert d1_full_s42.min() >= -1.0 and d1_full_s42.max() <= 1.0
    assert d0_full_s42.min() >= -1.0 and d0_full_s42.max() <= 1.0
    print(f'seed 42 D1 (treated, n={len(d1_full_s42):,}): min={d1_full_s42.min():.4f} max={d1_full_s42.max():.4f} mean={d1_full_s42.mean():.4f}')
    print(f'seed 42 D0 (control, n={len(d0_full_s42):,}): min={d0_full_s42.min():.4f} max={d0_full_s42.max():.4f} mean={d0_full_s42.mean():.4f}')

    pseudo_outcomes_frame_full_s42 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s42.loc[treated_mask_train_full_s42, SOURCE_ROW_ID].to_numpy(), 'arm': 'treated', 'pseudo_effect': d1_full_s42}),
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s42.loc[control_mask_train_full_s42, SOURCE_ROW_ID].to_numpy(), 'arm': 'control', 'pseudo_effect': d0_full_s42}),
    ], ignore_index=True)
    write_bytes_new(RUN_ROOT_T09_FULL_S42, f'{XLEARNER_SEED_DIR_T09_FULL_S42}/pseudo_outcomes.parquet', pseudo_outcomes_frame_full_s42.to_parquet(index=False))

    X_train_for_effect_full_s42 = transform_t09_full.transform(full_train_with_oof_s42)
    assert_model_feature_contract(X_train_for_effect_full_s42.columns)
    X_train_treated_effect_full_s42 = X_train_for_effect_full_s42.loc[treated_mask_train_full_s42].reset_index(drop=True)
    X_train_control_effect_full_s42 = X_train_for_effect_full_s42.loc[control_mask_train_full_s42].reset_index(drop=True)
    assert len(X_train_treated_effect_full_s42) == len(d1_full_s42)
    assert len(X_train_control_effect_full_s42) == len(d0_full_s42)


#### Seed 42: effect-stage fit + validation scoring

In [ ]:
if RUN_T09_FULL_STAGE:
    tau1_model_full_s42 = lgb_baseline.fit_regressor(X_train_treated_effect_full_s42, d1_full_s42, seed=MODEL_SEED_T09_FULL_42)
    tau0_model_full_s42 = lgb_baseline.fit_regressor(X_train_control_effect_full_s42, d0_full_s42, seed=MODEL_SEED_T09_FULL_42)
    assert tau1_model_full_s42.config_hash == tau0_model_full_s42.config_hash
    assert tau1_model_full_s42.num_boost_round == tau0_model_full_s42.num_boost_round == lgb_baseline.EFFECT_NUM_BOOST_ROUND
    print(f'seed 42 tau1: rounds={tau1_model_full_s42.num_boost_round}, objective=regression, config_hash={tau1_model_full_s42.config_hash[:16]}...')
    print(f'seed 42 tau0: rounds={tau0_model_full_s42.num_boost_round}, objective=regression, config_hash={tau0_model_full_s42.config_hash[:16]}...')

    tau1_hat_full_s42 = lgb_baseline.predict_values(tau1_model_full_s42, X_full_validation_t09_full)
    tau0_hat_full_s42 = lgb_baseline.predict_values(tau0_model_full_s42, X_full_validation_t09_full)
    assert len(tau1_hat_full_s42) == len(tau0_hat_full_s42) == len(validation_ids_full)
    assert np.isfinite(tau1_hat_full_s42).all() and np.isfinite(tau0_hat_full_s42).all()

    tau_hat_full_s42 = combine_tau(tau1_hat_full_s42, tau0_hat_full_s42, g_full_t09)
    np.testing.assert_allclose(tau_hat_full_s42, g_full_t09 * tau0_hat_full_s42 + (1.0 - g_full_t09) * tau1_hat_full_s42)
    assert np.isfinite(tau_hat_full_s42).all()
    assert set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)
    print(f'seed 42 tau_hat range: [{tau_hat_full_s42.min():.4f}, {tau_hat_full_s42.max():.4f}], n={len(tau_hat_full_s42):,}')


#### Seed 42: T06 metrics + artifacts

In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_ranking_full_s42 = metrics.evaluate_ranking(
        tau_hat_full_s42, t_full_validation_t09_full, y_full_validation_arr_t09_full, source_row_id_full_validation_t09_full,
    )
    print(f'seed 42 X-Learner FULL qini_area={xlearner_ranking_full_s42.qini_area:.4f} qini_above_random={xlearner_ranking_full_s42.qini_above_random:.4f}')


    def _rows_with_run_context_full_s42(rows, population='full_train_plus_validation'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T09_FULL_S42)
            row.setdefault('stage', 't09_full')
            row.setdefault('model_seed', 42)
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_full_s42 = list(_rows_with_run_context_full_s42([
        {'method': 'xlearner', 'k': label, 'uplift': xlearner_ranking_full_s42.uplift_at_k[label],
         'incremental_conversions': xlearner_ranking_full_s42.incremental_conversions_at_k[label],
         'status': xlearner_ranking_full_s42.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_full_s42 = list(_rows_with_run_context_full_s42([{
        'ranking_method': 'xlearner',
        'qini_area': xlearner_ranking_full_s42.qini_area,
        'theoretical_random_qini_area': xlearner_ranking_full_s42.theoretical_random_qini_area,
        'qini_above_random': xlearner_ranking_full_s42.qini_above_random,
        'g_full': g_full_t09,
        **{f'uplift_at_{label}': xlearner_ranking_full_s42.uplift_at_k[label] for label in metrics.RANKING_K_LABELS},
        **{f'incremental_conversions_at_{label}': xlearner_ranking_full_s42.incremental_conversions_at_k[label] for label in metrics.RANKING_K_LABELS},
    }]))
    xlearner_deciles_rows_full_s42 = list(_rows_with_run_context_full_s42(xlearner_ranking_full_s42.decile_table.to_dict('records')))

    for _name, _rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full_s42),
        ('tables/model_summary.csv', model_summary_rows_full_s42),
        ('tables/xlearner_deciles.csv', xlearner_deciles_rows_full_s42),
    ):
        write_text_new(RUN_ROOT_T09_FULL_S42, _name, pd.DataFrame(_rows).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_frame_full_s42 = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation_t09_full,
        'tau1_hat': tau1_hat_full_s42,
        'tau0_hat': tau0_hat_full_s42,
        'g': g_full_t09,
        'tau_hat': tau_hat_full_s42,
    })
    xlearner_predictions_bytes_full_s42 = xlearner_predictions_frame_full_s42.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S42, f'{XLEARNER_SEED_DIR_T09_FULL_S42}/validation_predictions.parquet', xlearner_predictions_bytes_full_s42)

    mu0_a_text_full_s42 = mu0_A_model_full_s42.booster.model_to_string()
    mu1_a_text_full_s42 = mu1_A_model_full_s42.booster.model_to_string()
    mu0_b_text_full_s42 = mu0_B_model_full_s42.booster.model_to_string()
    mu1_b_text_full_s42 = mu1_B_model_full_s42.booster.model_to_string()
    tau1_text_full_s42 = tau1_model_full_s42.booster.model_to_string()
    tau0_text_full_s42 = tau0_model_full_s42.booster.model_to_string()
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_mu0_A.txt', mu0_a_text_full_s42)
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_mu1_A.txt', mu1_a_text_full_s42)
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_mu0_B.txt', mu0_b_text_full_s42)
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_mu1_B.txt', mu1_b_text_full_s42)
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_tau1.txt', tau1_text_full_s42)
    write_text_new(RUN_ROOT_T09_FULL_S42, 'models/xlearner_tau0.txt', tau0_text_full_s42)


    def _flatten_full_s42(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_full_s42 = metrics.metric_definitions()
    definitions_sha256_full_s42 = hashlib.sha256(json.dumps(definitions_full_s42, sort_keys=True).encode()).hexdigest()
    definitions_rows_full_s42 = list(_rows_with_run_context_full_s42(
        [{'field': k, 'value': _flatten_full_s42(v)} for k, v in definitions_full_s42.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_full_s42}]
    ))
    write_text_new(RUN_ROOT_T09_FULL_S42, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full_s42).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_sha256_full_s42 = hashlib.sha256(xlearner_predictions_bytes_full_s42).hexdigest()
    print(f'seed 42 T09 FULL tables/models/predictions/audit written.')


In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_correctness_full_s42 = {
        'run_id': RUN_ID_T09_FULL_S42,
        'stage': 't09_full',
        'model_seed': 42,
        'seed_role': 'primary reported result',
        'fold_assignment': {
            'fold_count': 2,
            'fold_a_count': int(len(fold_a_ids_full_t09)),
            'fold_b_count': int(len(fold_b_ids_full_t09)),
            'folds_disjoint': bool(set(fold_a_ids_full_t09).isdisjoint(set(fold_b_ids_full_t09))),
            'folds_cover_all_training_rows': bool(set(fold_a_ids_full_t09) | set(fold_b_ids_full_t09) == set(source_row_id_full_train_t09_full.tolist())),
            'fold_seed': FOLD_SEED_T09,
            'fold_assignment_row_ids_sha256': fold_assignment_row_ids_sha256_t09_full,
        },
        'oof_nuisance': {
            'rows': int(len(oof_nuisance_frame_full_s42)),
            'full_coverage_both_surfaces': True,
            'opposite_fold_only_both_surfaces': True,
            'finite_coverage_fraction': 1.0 if oof_finite_full_s42 else float(np.isfinite(oof_nuisance_frame_full_s42[['mu0_oof', 'mu1_oof']].to_numpy()).mean()),
        },
        'pseudo_outcomes': {
            'd1_support': int(len(d1_full_s42)), 'd1_min': float(d1_full_s42.min()), 'd1_max': float(d1_full_s42.max()), 'd1_mean': float(d1_full_s42.mean()),
            'd0_support': int(len(d0_full_s42)), 'd0_min': float(d0_full_s42.min()), 'd0_max': float(d0_full_s42.max()), 'd0_mean': float(d0_full_s42.mean()),
            'd1_within_theoretical_bounds': bool(d1_full_s42.min() >= -1.0 and d1_full_s42.max() <= 1.0),
            'd0_within_theoretical_bounds': bool(d0_full_s42.min() >= -1.0 and d0_full_s42.max() <= 1.0),
            'd1_treated_row_count_matches_arm_partition': bool(int(len(d1_full_s42)) == int(treated_mask_train_full_s42.sum())),
            'd0_control_row_count_matches_arm_partition': bool(int(len(d0_full_s42)) == int(control_mask_train_full_s42.sum())),
        },
        'effect_stage': {
            'objective': 'regression',
            'num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
            'early_stopping': False,
            'tau_hat_finite': bool(np.isfinite(tau_hat_full_s42).all()),
            'g_full_value': g_full_t09,
            'g_full_source': 'complete_frozen_t05_training_partition',
            'g_full_n_treated': g_full_n_treated_t09,
            'g_full_n_train': g_full_n_train_t09,
        },
    }


#### Seed 42: reload/reconciliation

In [ ]:
if RUN_T09_FULL_STAGE:
    reloaded_mu0_a_booster_full_s42 = lgb.Booster(model_str=mu0_a_text_full_s42)
    reloaded_mu1_a_booster_full_s42 = lgb.Booster(model_str=mu1_a_text_full_s42)
    reloaded_mu0_b_booster_full_s42 = lgb.Booster(model_str=mu0_b_text_full_s42)
    reloaded_mu1_b_booster_full_s42 = lgb.Booster(model_str=mu1_b_text_full_s42)
    reloaded_tau1_booster_full_s42 = lgb.Booster(model_str=tau1_text_full_s42)
    reloaded_tau0_booster_full_s42 = lgb.Booster(model_str=tau0_text_full_s42)

    X_full_validation_rebuilt_s42 = transform_t09_full.transform(full_validation_frame_t09_full)
    X_A_full_rebuilt_s42 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_a_full_t09].reset_index(drop=True))
    X_B_full_rebuilt_s42 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_b_full_t09].reset_index(drop=True))

    mu0_oof_a_reloaded_s42 = np.asarray(reloaded_mu0_b_booster_full_s42.predict(X_A_full_rebuilt_s42, num_iteration=mu0_B_model_full_s42.best_iteration), dtype=np.float64)
    mu1_oof_a_reloaded_s42 = np.asarray(reloaded_mu1_b_booster_full_s42.predict(X_A_full_rebuilt_s42, num_iteration=mu1_B_model_full_s42.best_iteration), dtype=np.float64)
    mu0_oof_b_reloaded_s42 = np.asarray(reloaded_mu0_a_booster_full_s42.predict(X_B_full_rebuilt_s42, num_iteration=mu0_A_model_full_s42.best_iteration), dtype=np.float64)
    mu1_oof_b_reloaded_s42 = np.asarray(reloaded_mu1_a_booster_full_s42.predict(X_B_full_rebuilt_s42, num_iteration=mu1_A_model_full_s42.best_iteration), dtype=np.float64)

    oof_reload_matches_full_s42 = bool(
        np.allclose(mu0_oof_a_reloaded_s42, mu0_oof_a_full_s42, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_a_reloaded_s42, mu1_oof_a_full_s42, rtol=1e-6, atol=1e-8)
        and np.allclose(mu0_oof_b_reloaded_s42, mu0_oof_b_full_s42, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_b_reloaded_s42, mu1_oof_b_full_s42, rtol=1e-6, atol=1e-8)
    )

    tau1_reloaded_s42 = np.asarray(reloaded_tau1_booster_full_s42.predict(X_full_validation_rebuilt_s42, num_iteration=tau1_model_full_s42.num_boost_round), dtype=np.float64)
    tau0_reloaded_s42 = np.asarray(reloaded_tau0_booster_full_s42.predict(X_full_validation_rebuilt_s42, num_iteration=tau0_model_full_s42.num_boost_round), dtype=np.float64)
    tau1_reload_matches_s42 = bool(np.allclose(tau1_reloaded_s42, tau1_hat_full_s42, rtol=1e-6, atol=1e-8))
    tau0_reload_matches_s42 = bool(np.allclose(tau0_reloaded_s42, tau0_hat_full_s42, rtol=1e-6, atol=1e-8))

    tau_hat_reloaded_s42 = combine_tau(tau1_reloaded_s42, tau0_reloaded_s42, g_full_t09)
    tau_hat_reload_matches_s42 = bool(np.allclose(tau_hat_reloaded_s42, tau_hat_full_s42, rtol=1e-6, atol=1e-8))

    row_identity_matches_full_s42 = set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)

    # lgb_baseline.config_hash()/.regression_config_hash() with no args hash the
    # FROZEN_*_CONFIG dict verbatim, whose literal 'seed' field is 42 -- the
    # correct reference for a seed-42 run is that same frozen config with only
    # 'seed' overridden to 42 (exactly what fit_binary_classifier/fit_regressor
    # themselves hash internally), not the unconditional seed-42 default.
    expected_nuisance_config_hash_s42 = lgb_baseline.config_hash({**lgb_baseline.FROZEN_BINARY_CONFIG, 'seed': MODEL_SEED_T09_FULL_42})
    expected_regression_config_hash_s42 = lgb_baseline.regression_config_hash({**lgb_baseline.FROZEN_REGRESSION_CONFIG, 'seed': MODEL_SEED_T09_FULL_42})

    reload_verification_full_s42 = {
        'config_hash_matches': bool(
            mu0_A_model_full_s42.config_hash == mu1_A_model_full_s42.config_hash == mu0_B_model_full_s42.config_hash
            == mu1_B_model_full_s42.config_hash == expected_nuisance_config_hash_s42
        ),
        'regression_config_hash_matches': bool(tau1_model_full_s42.config_hash == tau0_model_full_s42.config_hash == expected_regression_config_hash_s42),
        'oof_reload_matches_within_tolerance': oof_reload_matches_full_s42,
        'tau1_reload_matches_within_tolerance': tau1_reload_matches_s42,
        'tau0_reload_matches_within_tolerance': tau0_reload_matches_s42,
        'tau_hat_reload_matches_within_tolerance': tau_hat_reload_matches_s42,
        'row_identity_matches': row_identity_matches_full_s42,
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(f'seed 42 reload_verification:', reload_verification_full_s42)

    xlearner_correctness_full_s42['reload_verification'] = reload_verification_full_s42
    write_json_new(RUN_ROOT_T09_FULL_S42, 'audit/xlearner_correctness.json', xlearner_correctness_full_s42)


#### Seed 42: upstream reuse verification

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s42_upstream_reuse_hashes = {}
    for _label, _root, _hashes, _rel_path in (
        ('t07_full', t07_full_root, t07_artifact_hashes, 'tables/model_summary.csv'),
        ('t08_full', t08_full_root_ref, t08_full_artifact_hashes_ref, 'tables/model_summary.csv'),
        ('t09_smoke', t09_smoke_root_ref, t09_smoke_artifact_hashes_ref, 'tables/model_summary.csv'),
    ):
        _actual = hashlib.sha256((_root / _rel_path).read_bytes()).hexdigest()
        _expected = _hashes[_rel_path]
        t09_full_s42_upstream_reuse_hashes[_label] = {'path': _rel_path, 'expected': _expected, 'actual': _actual, 'matches': _actual == _expected}
        assert _actual == _expected, f'{_label} artifact {_rel_path} hash mismatch -- refusing to cite unverified evidence'

    t09_full_s42_recomputation_guard = {
        'RUN_T07_STAGE': RUN_T07_STAGE,
        'RUN_T08_SMOKE_STAGE': RUN_T08_SMOKE_STAGE,
        'RUN_T08_FULL_STAGE': RUN_T08_FULL_STAGE,
        'RUN_T09_SMOKE_STAGE': RUN_T09_SMOKE_STAGE,
    }
    assert not any(t09_full_s42_recomputation_guard.values()), 'T09 FULL must not trigger any T07/T08/T09-SMOKE recomputation'
    print(f'seed 42 T07/T08/T09-SMOKE reuse hash verification (all must match, none recomputed):')
    for _label, _info in t09_full_s42_upstream_reuse_hashes.items():
        print('  ' + _label + ': matches=' + str(_info['matches']))
    print(f'seed 42 recomputation guards (all must be False):', t09_full_s42_recomputation_guard)


#### Seed 42: manifest, resource evidence, hard correctness gate

In [ ]:
if RUN_T09_FULL_STAGE:
    expected_relative_paths_full_s42 = {
        'audit/xlearner_fold_manifest.parquet',
        'audit/xlearner_correctness.json',
        f'{XLEARNER_SEED_DIR_T09_FULL_S42}/oof_nuisance.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S42}/pseudo_outcomes.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S42}/validation_predictions.parquet',
        'models/xlearner_mu0_A.txt', 'models/xlearner_mu1_A.txt', 'models/xlearner_mu0_B.txt', 'models/xlearner_mu1_B.txt',
        'models/xlearner_tau0.txt', 'models/xlearner_tau1.txt',
        'tables/model_summary.csv', 'tables/uplift_at_k.csv', 'tables/xlearner_deciles.csv',
    }
    obsolete_relative_paths_full_s42 = {'audit/fold_assignment.parquet', 'audit/pseudo_outcomes.parquet'}

    t09_full_s42_max_rss_bytes_observed = t09_full_s42_rss_sampler.stop()
    t09_full_s42_wall_seconds = __import__('time').perf_counter() - t09_full_s42_wall_start
    t09_full_s42_resource_evidence = {
        'wall_seconds': t09_full_s42_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t09_full_s42_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t09_full_s42_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t09_full_s42_max_rss_bytes_observed) - int(t09_full_s42_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(f'seed 42 resource_evidence:', t09_full_s42_resource_evidence)

    write_json_new(RUN_ROOT_T09_FULL_S42, 'audit/environment.json', {
        'run_id': RUN_ID_T09_FULL_S42,
        'created_at_utc': t09_full_s42_started.isoformat(),
        'git_head': t09_full_s42_git_head,
        'git_dirty': t09_full_s42_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T09_FULL_S42, 'audit/run_config.json', {
        'run_id': RUN_ID_T09_FULL_S42,
        'created_at_utc': t09_full_s42_started.isoformat(),
        'stage': 't09_full',
        'model_seed': 42,
        'seed_role': 'primary reported result',
        'population': 'full_train_plus_validation',
        'git_head': t09_full_s42_git_head,
        'git_dirty': t09_full_s42_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_xlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t09_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t09_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'upstream_reuse': {
            'hash_verification': t09_full_s42_upstream_reuse_hashes,
            'recomputation_guards': t09_full_s42_recomputation_guard,
            'recomputed': False,
        },
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES_T09, 'fold_seed': FOLD_SEED_T09, 'model_seed': 42},
        'lightgbm_binary_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_regression_config': lgb_baseline.FROZEN_REGRESSION_CONFIG,
        'effect_num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
        'nuisance_config_hash': mu0_A_model_full_s42.config_hash,
        'regression_config_hash': tau1_model_full_s42.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s42.best_iteration, 'mu1_A': mu1_A_model_full_s42.best_iteration,
            'mu0_B': mu0_B_model_full_s42.best_iteration, 'mu1_B': mu1_B_model_full_s42.best_iteration,
        },
        'g_full_value': g_full_t09,
        'g_full_n_treated': g_full_n_treated_t09,
        'g_full_n_train': g_full_n_train_t09,
        'oof_nuisance_sha256': hashlib.sha256(oof_nuisance_bytes_full_s42).hexdigest(),
        'xlearner_predictions_sha256': xlearner_predictions_sha256_full_s42,
        'definitions_sha256': definitions_sha256_full_s42,
        'resource_evidence': t09_full_s42_resource_evidence,
        'reload_verification': reload_verification_full_s42,
    })
    print(f'seed 42 T09 FULL run_config.json / environment.json written.')

    finalize_artifact_manifest(
        RUN_ROOT_T09_FULL_S42,
        run_id=RUN_ID_T09_FULL_S42,
        final_status='COMPLETED_T09_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t09_full',
        population='full_train_plus_validation',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/xlearner.py', 'role': 'reusable_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t07_evidence']['authoritative_t07_full_run_id']}", 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t08_evidence']['authoritative_t08_full_run_id']}", 'role': 'reused_t08_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f'outputs/runs/{T09_SMOKE_RUN_ID_ACCEPTED}', 'role': 'reused_t09_smoke_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'seed 42 T09 FULL run finalized: {RUN_ID_T09_FULL_S42}')

    immutable_write_refused_full_s42 = False
    try:
        write_json_new(RUN_ROOT_T09_FULL_S42, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_full_s42 = True
    except Exception:
        immutable_write_refused_full_s42 = False
    assert immutable_write_refused_full_s42

    t09_full_s42_manifest = json.loads((RUN_ROOT_T09_FULL_S42 / 'audit' / 'artifact_manifest.json').read_text(encoding='utf-8'))
    t09_full_s42_actual_paths = {a['path'] for a in t09_full_s42_manifest['artifacts']}
    t09_full_s42_paths_conform = expected_relative_paths_full_s42.issubset(t09_full_s42_actual_paths)
    t09_full_s42_no_obsolete_paths = obsolete_relative_paths_full_s42.isdisjoint(t09_full_s42_actual_paths)
    print(f'seed 42 artifact-path conformance: expected_subset_present={t09_full_s42_paths_conform}, no_obsolete_paths={t09_full_s42_no_obsolete_paths}')

    # --- Hard correctness/resource/artifact gate. If seed 42 fails here, this
    # raises and the notebook execution stops -- seed 123/2026 cells below never
    # execute. This is the mechanical implementation of "stop before robustness
    # seeds on a seed-42 failure". ---
    assert t09_full_s42_paths_conform, f'seed 42: corrected artifact paths missing from manifest'
    assert t09_full_s42_no_obsolete_paths, f'seed 42: obsolete non-conforming paths present in manifest'
    assert reload_verification_full_s42['config_hash_matches']
    assert reload_verification_full_s42['regression_config_hash_matches']
    assert reload_verification_full_s42['oof_reload_matches_within_tolerance']
    assert reload_verification_full_s42['tau1_reload_matches_within_tolerance']
    assert reload_verification_full_s42['tau0_reload_matches_within_tolerance']
    assert reload_verification_full_s42['tau_hat_reload_matches_within_tolerance']
    assert reload_verification_full_s42['row_identity_matches']
    assert t09_full_s42_resource_evidence['execution_completed']
    assert not t09_full_s42_resource_evidence['oom_or_termination_observed']
    assert immutable_write_refused_full_s42
    print(f'seed 42: ALL correctness/resource/artifact gates PASSED.')


#### Seed 42: summary

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s42_summary = {
        'run_id': RUN_ID_T09_FULL_S42,
        'model_seed': 42,
        'seed_role': 'primary reported result',
        'nuisance_config_hash': mu0_A_model_full_s42.config_hash,
        'regression_config_hash': tau1_model_full_s42.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s42.best_iteration, 'mu1_A': mu1_A_model_full_s42.best_iteration,
            'mu0_B': mu0_B_model_full_s42.best_iteration, 'mu1_B': mu1_B_model_full_s42.best_iteration,
        },
        'g_full': g_full_t09,
        'tau_hat_range': [float(tau_hat_full_s42.min()), float(tau_hat_full_s42.max())],
        'xlearner_ranking_full': {
            'qini_area': xlearner_ranking_full_s42.qini_area,
            'qini_above_random': xlearner_ranking_full_s42.qini_above_random,
        },
        'reload_verification': reload_verification_full_s42,
        'resource_evidence': t09_full_s42_resource_evidence,
        'artifact_paths_conform': t09_full_s42_paths_conform,
        'no_obsolete_paths': t09_full_s42_no_obsolete_paths,
    }
    print(json.dumps(t09_full_s42_summary, indent=2, default=str))


### T09.FULL Seed 123 (robustness evidence, never selected)

Fold membership, `g_full`, and every configuration value are identical to the
other seeds above/below -- only LightGBM's `seed` parameter changes.

In [ ]:
if RUN_T09_FULL_STAGE:
    MODEL_SEED_T09_FULL_123 = 123

    t09_full_s123_started = datetime.now(timezone.utc)
    t09_full_s123_wall_start = __import__('time').perf_counter()
    t09_full_s123_baseline_rss_bytes = process.memory_info().rss
    t09_full_s123_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T09_FULL_S123 = t09_full_s123_started.strftime('t09_full_seed123_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T09_FULL_S123 = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_FULL_S123
    RUN_ROOT_T09_FULL_S123.mkdir(parents=True, exist_ok=False)
    XLEARNER_SEED_DIR_T09_FULL_S123 = 'predictions/development/xlearner/seed_123'

    try:
        t09_full_s123_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t09_full_s123_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t09_full_s123_git_head, t09_full_s123_git_dirty = None, None

    print(f'RUN_ID_T09_FULL_S123 = {RUN_ID_T09_FULL_S123}')
    print(f'model_seed = 123, fold_seed = {FOLD_SEED_T09}, g_full = {g_full_t09:.6f}')


#### Seed 123: four fold-specific nuisance fits

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_A_model_full_s123 = lgb_baseline.fit_binary_classifier(X_A_control_full_t09, y_A_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_123)
    mu1_A_model_full_s123 = lgb_baseline.fit_binary_classifier(X_A_treated_full_t09, y_A_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_123)
    mu0_B_model_full_s123 = lgb_baseline.fit_binary_classifier(X_B_control_full_t09, y_B_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_123)
    mu1_B_model_full_s123 = lgb_baseline.fit_binary_classifier(X_B_treated_full_t09, y_B_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_123)

    assert mu0_A_model_full_s123.config_hash == mu1_A_model_full_s123.config_hash == mu0_B_model_full_s123.config_hash == mu1_B_model_full_s123.config_hash
    for _name, _model in (('mu0_A', mu0_A_model_full_s123), ('mu1_A', mu1_A_model_full_s123), ('mu0_B', mu0_B_model_full_s123), ('mu1_B', mu1_B_model_full_s123)):
        print(f'seed 123 {_name}: best_iteration={_model.best_iteration}, config_hash={_model.config_hash[:16]}...')


#### Seed 123: out-of-fold assembly + corrected artifact paths

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_oof_a_full_s123 = lgb_baseline.predict_probabilities(mu0_B_model_full_s123, X_A_full_t09_full)
    mu1_oof_a_full_s123 = lgb_baseline.predict_probabilities(mu1_B_model_full_s123, X_A_full_t09_full)
    mu0_oof_b_full_s123 = lgb_baseline.predict_probabilities(mu0_A_model_full_s123, X_B_full_t09_full)
    mu1_oof_b_full_s123 = lgb_baseline.predict_probabilities(mu1_A_model_full_s123, X_B_full_t09_full)

    row_fold_unsorted_full_s123 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'A'), np.full(len(source_row_id_fold_b_full_t09), 'B')])
    prediction_fold_of_mu0_full_s123 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'B'), np.full(len(source_row_id_fold_b_full_t09), 'A')])
    prediction_fold_of_mu1_full_s123 = prediction_fold_of_mu0_full_s123.copy()
    assert_opposite_fold(row_fold_unsorted_full_s123, prediction_fold_of_mu0_full_s123, name='mu0_oof seed 123')
    assert_opposite_fold(row_fold_unsorted_full_s123, prediction_fold_of_mu1_full_s123, name='mu1_oof seed 123')

    oof_nuisance_frame_full_s123 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_a_full_t09, 'fold': 'A', 'mu0_oof': mu0_oof_a_full_s123, 'mu1_oof': mu1_oof_a_full_s123}),
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_b_full_t09, 'fold': 'B', 'mu0_oof': mu0_oof_b_full_s123, 'mu1_oof': mu1_oof_b_full_s123}),
    ], ignore_index=True).sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    assert_full_oof_coverage(source_row_id_full_train_t09_full, oof_nuisance_frame_full_s123[SOURCE_ROW_ID].to_numpy(), name=f'seed 123 mu0_oof/mu1_oof combined coverage')
    assert oof_nuisance_frame_full_s123['mu0_oof'].notna().all() and oof_nuisance_frame_full_s123['mu1_oof'].notna().all()
    oof_finite_full_s123 = bool(np.isfinite(oof_nuisance_frame_full_s123[['mu0_oof', 'mu1_oof']].to_numpy()).all())
    assert oof_finite_full_s123

    full_train_with_oof_s123 = full_train_frame_t09_full.merge(oof_nuisance_frame_full_s123, on=SOURCE_ROW_ID, how='inner', validate='one_to_one')
    assert len(full_train_with_oof_s123) == len(full_train_frame_t09_full)

    write_bytes_new(RUN_ROOT_T09_FULL_S123, 'audit/xlearner_fold_manifest.parquet', fold_manifest_frame_full_t09.to_parquet(index=False))
    oof_nuisance_bytes_full_s123 = oof_nuisance_frame_full_s123.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S123, f'{XLEARNER_SEED_DIR_T09_FULL_S123}/oof_nuisance.parquet', oof_nuisance_bytes_full_s123)

    print(f'seed 123 OOF nuisance assembled: {len(oof_nuisance_frame_full_s123):,} rows, 100% coverage, 0% same-fold leakage.')


#### Seed 123: pseudo-outcomes

In [ ]:
if RUN_T09_FULL_STAGE:
    treatment_train_full_s123 = full_train_with_oof_s123[TREATMENT_COLUMN].astype('float64').to_numpy()
    outcome_train_full_s123 = full_train_with_oof_s123[PRIMARY_OUTCOME].astype('float64').to_numpy()
    mu0_oof_aligned_full_s123 = full_train_with_oof_s123['mu0_oof'].to_numpy()
    mu1_oof_aligned_full_s123 = full_train_with_oof_s123['mu1_oof'].to_numpy()

    d1_full_s123, d0_full_s123, treated_mask_train_full_s123, control_mask_train_full_s123 = compute_pseudo_outcomes(
        treatment_train_full_s123, outcome_train_full_s123, mu0_oof_aligned_full_s123, mu1_oof_aligned_full_s123,
    )
    assert len(d1_full_s123) == int(treated_mask_train_full_s123.sum())
    assert len(d0_full_s123) == int(control_mask_train_full_s123.sum())
    assert d1_full_s123.min() >= -1.0 and d1_full_s123.max() <= 1.0
    assert d0_full_s123.min() >= -1.0 and d0_full_s123.max() <= 1.0
    print(f'seed 123 D1 (treated, n={len(d1_full_s123):,}): min={d1_full_s123.min():.4f} max={d1_full_s123.max():.4f} mean={d1_full_s123.mean():.4f}')
    print(f'seed 123 D0 (control, n={len(d0_full_s123):,}): min={d0_full_s123.min():.4f} max={d0_full_s123.max():.4f} mean={d0_full_s123.mean():.4f}')

    pseudo_outcomes_frame_full_s123 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s123.loc[treated_mask_train_full_s123, SOURCE_ROW_ID].to_numpy(), 'arm': 'treated', 'pseudo_effect': d1_full_s123}),
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s123.loc[control_mask_train_full_s123, SOURCE_ROW_ID].to_numpy(), 'arm': 'control', 'pseudo_effect': d0_full_s123}),
    ], ignore_index=True)
    write_bytes_new(RUN_ROOT_T09_FULL_S123, f'{XLEARNER_SEED_DIR_T09_FULL_S123}/pseudo_outcomes.parquet', pseudo_outcomes_frame_full_s123.to_parquet(index=False))

    X_train_for_effect_full_s123 = transform_t09_full.transform(full_train_with_oof_s123)
    assert_model_feature_contract(X_train_for_effect_full_s123.columns)
    X_train_treated_effect_full_s123 = X_train_for_effect_full_s123.loc[treated_mask_train_full_s123].reset_index(drop=True)
    X_train_control_effect_full_s123 = X_train_for_effect_full_s123.loc[control_mask_train_full_s123].reset_index(drop=True)
    assert len(X_train_treated_effect_full_s123) == len(d1_full_s123)
    assert len(X_train_control_effect_full_s123) == len(d0_full_s123)


#### Seed 123: effect-stage fit + validation scoring

In [ ]:
if RUN_T09_FULL_STAGE:
    tau1_model_full_s123 = lgb_baseline.fit_regressor(X_train_treated_effect_full_s123, d1_full_s123, seed=MODEL_SEED_T09_FULL_123)
    tau0_model_full_s123 = lgb_baseline.fit_regressor(X_train_control_effect_full_s123, d0_full_s123, seed=MODEL_SEED_T09_FULL_123)
    assert tau1_model_full_s123.config_hash == tau0_model_full_s123.config_hash
    assert tau1_model_full_s123.num_boost_round == tau0_model_full_s123.num_boost_round == lgb_baseline.EFFECT_NUM_BOOST_ROUND
    print(f'seed 123 tau1: rounds={tau1_model_full_s123.num_boost_round}, objective=regression, config_hash={tau1_model_full_s123.config_hash[:16]}...')
    print(f'seed 123 tau0: rounds={tau0_model_full_s123.num_boost_round}, objective=regression, config_hash={tau0_model_full_s123.config_hash[:16]}...')

    tau1_hat_full_s123 = lgb_baseline.predict_values(tau1_model_full_s123, X_full_validation_t09_full)
    tau0_hat_full_s123 = lgb_baseline.predict_values(tau0_model_full_s123, X_full_validation_t09_full)
    assert len(tau1_hat_full_s123) == len(tau0_hat_full_s123) == len(validation_ids_full)
    assert np.isfinite(tau1_hat_full_s123).all() and np.isfinite(tau0_hat_full_s123).all()

    tau_hat_full_s123 = combine_tau(tau1_hat_full_s123, tau0_hat_full_s123, g_full_t09)
    np.testing.assert_allclose(tau_hat_full_s123, g_full_t09 * tau0_hat_full_s123 + (1.0 - g_full_t09) * tau1_hat_full_s123)
    assert np.isfinite(tau_hat_full_s123).all()
    assert set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)
    print(f'seed 123 tau_hat range: [{tau_hat_full_s123.min():.4f}, {tau_hat_full_s123.max():.4f}], n={len(tau_hat_full_s123):,}')


#### Seed 123: T06 metrics + artifacts

In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_ranking_full_s123 = metrics.evaluate_ranking(
        tau_hat_full_s123, t_full_validation_t09_full, y_full_validation_arr_t09_full, source_row_id_full_validation_t09_full,
    )
    print(f'seed 123 X-Learner FULL qini_area={xlearner_ranking_full_s123.qini_area:.4f} qini_above_random={xlearner_ranking_full_s123.qini_above_random:.4f}')


    def _rows_with_run_context_full_s123(rows, population='full_train_plus_validation'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T09_FULL_S123)
            row.setdefault('stage', 't09_full')
            row.setdefault('model_seed', 123)
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_full_s123 = list(_rows_with_run_context_full_s123([
        {'method': 'xlearner', 'k': label, 'uplift': xlearner_ranking_full_s123.uplift_at_k[label],
         'incremental_conversions': xlearner_ranking_full_s123.incremental_conversions_at_k[label],
         'status': xlearner_ranking_full_s123.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_full_s123 = list(_rows_with_run_context_full_s123([{
        'ranking_method': 'xlearner',
        'qini_area': xlearner_ranking_full_s123.qini_area,
        'theoretical_random_qini_area': xlearner_ranking_full_s123.theoretical_random_qini_area,
        'qini_above_random': xlearner_ranking_full_s123.qini_above_random,
        'g_full': g_full_t09,
        **{f'uplift_at_{label}': xlearner_ranking_full_s123.uplift_at_k[label] for label in metrics.RANKING_K_LABELS},
        **{f'incremental_conversions_at_{label}': xlearner_ranking_full_s123.incremental_conversions_at_k[label] for label in metrics.RANKING_K_LABELS},
    }]))
    xlearner_deciles_rows_full_s123 = list(_rows_with_run_context_full_s123(xlearner_ranking_full_s123.decile_table.to_dict('records')))

    for _name, _rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full_s123),
        ('tables/model_summary.csv', model_summary_rows_full_s123),
        ('tables/xlearner_deciles.csv', xlearner_deciles_rows_full_s123),
    ):
        write_text_new(RUN_ROOT_T09_FULL_S123, _name, pd.DataFrame(_rows).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_frame_full_s123 = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation_t09_full,
        'tau1_hat': tau1_hat_full_s123,
        'tau0_hat': tau0_hat_full_s123,
        'g': g_full_t09,
        'tau_hat': tau_hat_full_s123,
    })
    xlearner_predictions_bytes_full_s123 = xlearner_predictions_frame_full_s123.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S123, f'{XLEARNER_SEED_DIR_T09_FULL_S123}/validation_predictions.parquet', xlearner_predictions_bytes_full_s123)

    mu0_a_text_full_s123 = mu0_A_model_full_s123.booster.model_to_string()
    mu1_a_text_full_s123 = mu1_A_model_full_s123.booster.model_to_string()
    mu0_b_text_full_s123 = mu0_B_model_full_s123.booster.model_to_string()
    mu1_b_text_full_s123 = mu1_B_model_full_s123.booster.model_to_string()
    tau1_text_full_s123 = tau1_model_full_s123.booster.model_to_string()
    tau0_text_full_s123 = tau0_model_full_s123.booster.model_to_string()
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_mu0_A.txt', mu0_a_text_full_s123)
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_mu1_A.txt', mu1_a_text_full_s123)
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_mu0_B.txt', mu0_b_text_full_s123)
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_mu1_B.txt', mu1_b_text_full_s123)
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_tau1.txt', tau1_text_full_s123)
    write_text_new(RUN_ROOT_T09_FULL_S123, 'models/xlearner_tau0.txt', tau0_text_full_s123)


    def _flatten_full_s123(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_full_s123 = metrics.metric_definitions()
    definitions_sha256_full_s123 = hashlib.sha256(json.dumps(definitions_full_s123, sort_keys=True).encode()).hexdigest()
    definitions_rows_full_s123 = list(_rows_with_run_context_full_s123(
        [{'field': k, 'value': _flatten_full_s123(v)} for k, v in definitions_full_s123.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_full_s123}]
    ))
    write_text_new(RUN_ROOT_T09_FULL_S123, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full_s123).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_sha256_full_s123 = hashlib.sha256(xlearner_predictions_bytes_full_s123).hexdigest()
    print(f'seed 123 T09 FULL tables/models/predictions/audit written.')


In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_correctness_full_s123 = {
        'run_id': RUN_ID_T09_FULL_S123,
        'stage': 't09_full',
        'model_seed': 123,
        'seed_role': 'robustness evidence, never selected',
        'fold_assignment': {
            'fold_count': 2,
            'fold_a_count': int(len(fold_a_ids_full_t09)),
            'fold_b_count': int(len(fold_b_ids_full_t09)),
            'folds_disjoint': bool(set(fold_a_ids_full_t09).isdisjoint(set(fold_b_ids_full_t09))),
            'folds_cover_all_training_rows': bool(set(fold_a_ids_full_t09) | set(fold_b_ids_full_t09) == set(source_row_id_full_train_t09_full.tolist())),
            'fold_seed': FOLD_SEED_T09,
            'fold_assignment_row_ids_sha256': fold_assignment_row_ids_sha256_t09_full,
        },
        'oof_nuisance': {
            'rows': int(len(oof_nuisance_frame_full_s123)),
            'full_coverage_both_surfaces': True,
            'opposite_fold_only_both_surfaces': True,
            'finite_coverage_fraction': 1.0 if oof_finite_full_s123 else float(np.isfinite(oof_nuisance_frame_full_s123[['mu0_oof', 'mu1_oof']].to_numpy()).mean()),
        },
        'pseudo_outcomes': {
            'd1_support': int(len(d1_full_s123)), 'd1_min': float(d1_full_s123.min()), 'd1_max': float(d1_full_s123.max()), 'd1_mean': float(d1_full_s123.mean()),
            'd0_support': int(len(d0_full_s123)), 'd0_min': float(d0_full_s123.min()), 'd0_max': float(d0_full_s123.max()), 'd0_mean': float(d0_full_s123.mean()),
            'd1_within_theoretical_bounds': bool(d1_full_s123.min() >= -1.0 and d1_full_s123.max() <= 1.0),
            'd0_within_theoretical_bounds': bool(d0_full_s123.min() >= -1.0 and d0_full_s123.max() <= 1.0),
            'd1_treated_row_count_matches_arm_partition': bool(int(len(d1_full_s123)) == int(treated_mask_train_full_s123.sum())),
            'd0_control_row_count_matches_arm_partition': bool(int(len(d0_full_s123)) == int(control_mask_train_full_s123.sum())),
        },
        'effect_stage': {
            'objective': 'regression',
            'num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
            'early_stopping': False,
            'tau_hat_finite': bool(np.isfinite(tau_hat_full_s123).all()),
            'g_full_value': g_full_t09,
            'g_full_source': 'complete_frozen_t05_training_partition',
            'g_full_n_treated': g_full_n_treated_t09,
            'g_full_n_train': g_full_n_train_t09,
        },
    }


#### Seed 123: reload/reconciliation

In [ ]:
if RUN_T09_FULL_STAGE:
    reloaded_mu0_a_booster_full_s123 = lgb.Booster(model_str=mu0_a_text_full_s123)
    reloaded_mu1_a_booster_full_s123 = lgb.Booster(model_str=mu1_a_text_full_s123)
    reloaded_mu0_b_booster_full_s123 = lgb.Booster(model_str=mu0_b_text_full_s123)
    reloaded_mu1_b_booster_full_s123 = lgb.Booster(model_str=mu1_b_text_full_s123)
    reloaded_tau1_booster_full_s123 = lgb.Booster(model_str=tau1_text_full_s123)
    reloaded_tau0_booster_full_s123 = lgb.Booster(model_str=tau0_text_full_s123)

    X_full_validation_rebuilt_s123 = transform_t09_full.transform(full_validation_frame_t09_full)
    X_A_full_rebuilt_s123 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_a_full_t09].reset_index(drop=True))
    X_B_full_rebuilt_s123 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_b_full_t09].reset_index(drop=True))

    mu0_oof_a_reloaded_s123 = np.asarray(reloaded_mu0_b_booster_full_s123.predict(X_A_full_rebuilt_s123, num_iteration=mu0_B_model_full_s123.best_iteration), dtype=np.float64)
    mu1_oof_a_reloaded_s123 = np.asarray(reloaded_mu1_b_booster_full_s123.predict(X_A_full_rebuilt_s123, num_iteration=mu1_B_model_full_s123.best_iteration), dtype=np.float64)
    mu0_oof_b_reloaded_s123 = np.asarray(reloaded_mu0_a_booster_full_s123.predict(X_B_full_rebuilt_s123, num_iteration=mu0_A_model_full_s123.best_iteration), dtype=np.float64)
    mu1_oof_b_reloaded_s123 = np.asarray(reloaded_mu1_a_booster_full_s123.predict(X_B_full_rebuilt_s123, num_iteration=mu1_A_model_full_s123.best_iteration), dtype=np.float64)

    oof_reload_matches_full_s123 = bool(
        np.allclose(mu0_oof_a_reloaded_s123, mu0_oof_a_full_s123, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_a_reloaded_s123, mu1_oof_a_full_s123, rtol=1e-6, atol=1e-8)
        and np.allclose(mu0_oof_b_reloaded_s123, mu0_oof_b_full_s123, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_b_reloaded_s123, mu1_oof_b_full_s123, rtol=1e-6, atol=1e-8)
    )

    tau1_reloaded_s123 = np.asarray(reloaded_tau1_booster_full_s123.predict(X_full_validation_rebuilt_s123, num_iteration=tau1_model_full_s123.num_boost_round), dtype=np.float64)
    tau0_reloaded_s123 = np.asarray(reloaded_tau0_booster_full_s123.predict(X_full_validation_rebuilt_s123, num_iteration=tau0_model_full_s123.num_boost_round), dtype=np.float64)
    tau1_reload_matches_s123 = bool(np.allclose(tau1_reloaded_s123, tau1_hat_full_s123, rtol=1e-6, atol=1e-8))
    tau0_reload_matches_s123 = bool(np.allclose(tau0_reloaded_s123, tau0_hat_full_s123, rtol=1e-6, atol=1e-8))

    tau_hat_reloaded_s123 = combine_tau(tau1_reloaded_s123, tau0_reloaded_s123, g_full_t09)
    tau_hat_reload_matches_s123 = bool(np.allclose(tau_hat_reloaded_s123, tau_hat_full_s123, rtol=1e-6, atol=1e-8))

    row_identity_matches_full_s123 = set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)

    # lgb_baseline.config_hash()/.regression_config_hash() with no args hash the
    # FROZEN_*_CONFIG dict verbatim, whose literal 'seed' field is 42 -- the
    # correct reference for a seed-123 run is that same frozen config with only
    # 'seed' overridden to 123 (exactly what fit_binary_classifier/fit_regressor
    # themselves hash internally), not the unconditional seed-42 default.
    expected_nuisance_config_hash_s123 = lgb_baseline.config_hash({**lgb_baseline.FROZEN_BINARY_CONFIG, 'seed': MODEL_SEED_T09_FULL_123})
    expected_regression_config_hash_s123 = lgb_baseline.regression_config_hash({**lgb_baseline.FROZEN_REGRESSION_CONFIG, 'seed': MODEL_SEED_T09_FULL_123})

    reload_verification_full_s123 = {
        'config_hash_matches': bool(
            mu0_A_model_full_s123.config_hash == mu1_A_model_full_s123.config_hash == mu0_B_model_full_s123.config_hash
            == mu1_B_model_full_s123.config_hash == expected_nuisance_config_hash_s123
        ),
        'regression_config_hash_matches': bool(tau1_model_full_s123.config_hash == tau0_model_full_s123.config_hash == expected_regression_config_hash_s123),
        'oof_reload_matches_within_tolerance': oof_reload_matches_full_s123,
        'tau1_reload_matches_within_tolerance': tau1_reload_matches_s123,
        'tau0_reload_matches_within_tolerance': tau0_reload_matches_s123,
        'tau_hat_reload_matches_within_tolerance': tau_hat_reload_matches_s123,
        'row_identity_matches': row_identity_matches_full_s123,
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(f'seed 123 reload_verification:', reload_verification_full_s123)

    xlearner_correctness_full_s123['reload_verification'] = reload_verification_full_s123
    write_json_new(RUN_ROOT_T09_FULL_S123, 'audit/xlearner_correctness.json', xlearner_correctness_full_s123)


#### Seed 123: upstream reuse verification

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s123_upstream_reuse_hashes = {}
    for _label, _root, _hashes, _rel_path in (
        ('t07_full', t07_full_root, t07_artifact_hashes, 'tables/model_summary.csv'),
        ('t08_full', t08_full_root_ref, t08_full_artifact_hashes_ref, 'tables/model_summary.csv'),
        ('t09_smoke', t09_smoke_root_ref, t09_smoke_artifact_hashes_ref, 'tables/model_summary.csv'),
    ):
        _actual = hashlib.sha256((_root / _rel_path).read_bytes()).hexdigest()
        _expected = _hashes[_rel_path]
        t09_full_s123_upstream_reuse_hashes[_label] = {'path': _rel_path, 'expected': _expected, 'actual': _actual, 'matches': _actual == _expected}
        assert _actual == _expected, f'{_label} artifact {_rel_path} hash mismatch -- refusing to cite unverified evidence'

    t09_full_s123_recomputation_guard = {
        'RUN_T07_STAGE': RUN_T07_STAGE,
        'RUN_T08_SMOKE_STAGE': RUN_T08_SMOKE_STAGE,
        'RUN_T08_FULL_STAGE': RUN_T08_FULL_STAGE,
        'RUN_T09_SMOKE_STAGE': RUN_T09_SMOKE_STAGE,
    }
    assert not any(t09_full_s123_recomputation_guard.values()), 'T09 FULL must not trigger any T07/T08/T09-SMOKE recomputation'
    print(f'seed 123 T07/T08/T09-SMOKE reuse hash verification (all must match, none recomputed):')
    for _label, _info in t09_full_s123_upstream_reuse_hashes.items():
        print('  ' + _label + ': matches=' + str(_info['matches']))
    print(f'seed 123 recomputation guards (all must be False):', t09_full_s123_recomputation_guard)


#### Seed 123: manifest, resource evidence, hard correctness gate

In [ ]:
if RUN_T09_FULL_STAGE:
    expected_relative_paths_full_s123 = {
        'audit/xlearner_fold_manifest.parquet',
        'audit/xlearner_correctness.json',
        f'{XLEARNER_SEED_DIR_T09_FULL_S123}/oof_nuisance.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S123}/pseudo_outcomes.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S123}/validation_predictions.parquet',
        'models/xlearner_mu0_A.txt', 'models/xlearner_mu1_A.txt', 'models/xlearner_mu0_B.txt', 'models/xlearner_mu1_B.txt',
        'models/xlearner_tau0.txt', 'models/xlearner_tau1.txt',
        'tables/model_summary.csv', 'tables/uplift_at_k.csv', 'tables/xlearner_deciles.csv',
    }
    obsolete_relative_paths_full_s123 = {'audit/fold_assignment.parquet', 'audit/pseudo_outcomes.parquet'}

    t09_full_s123_max_rss_bytes_observed = t09_full_s123_rss_sampler.stop()
    t09_full_s123_wall_seconds = __import__('time').perf_counter() - t09_full_s123_wall_start
    t09_full_s123_resource_evidence = {
        'wall_seconds': t09_full_s123_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t09_full_s123_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t09_full_s123_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t09_full_s123_max_rss_bytes_observed) - int(t09_full_s123_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(f'seed 123 resource_evidence:', t09_full_s123_resource_evidence)

    write_json_new(RUN_ROOT_T09_FULL_S123, 'audit/environment.json', {
        'run_id': RUN_ID_T09_FULL_S123,
        'created_at_utc': t09_full_s123_started.isoformat(),
        'git_head': t09_full_s123_git_head,
        'git_dirty': t09_full_s123_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T09_FULL_S123, 'audit/run_config.json', {
        'run_id': RUN_ID_T09_FULL_S123,
        'created_at_utc': t09_full_s123_started.isoformat(),
        'stage': 't09_full',
        'model_seed': 123,
        'seed_role': 'robustness evidence, never selected',
        'population': 'full_train_plus_validation',
        'git_head': t09_full_s123_git_head,
        'git_dirty': t09_full_s123_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_xlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t09_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t09_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'upstream_reuse': {
            'hash_verification': t09_full_s123_upstream_reuse_hashes,
            'recomputation_guards': t09_full_s123_recomputation_guard,
            'recomputed': False,
        },
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES_T09, 'fold_seed': FOLD_SEED_T09, 'model_seed': 123},
        'lightgbm_binary_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_regression_config': lgb_baseline.FROZEN_REGRESSION_CONFIG,
        'effect_num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
        'nuisance_config_hash': mu0_A_model_full_s123.config_hash,
        'regression_config_hash': tau1_model_full_s123.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s123.best_iteration, 'mu1_A': mu1_A_model_full_s123.best_iteration,
            'mu0_B': mu0_B_model_full_s123.best_iteration, 'mu1_B': mu1_B_model_full_s123.best_iteration,
        },
        'g_full_value': g_full_t09,
        'g_full_n_treated': g_full_n_treated_t09,
        'g_full_n_train': g_full_n_train_t09,
        'oof_nuisance_sha256': hashlib.sha256(oof_nuisance_bytes_full_s123).hexdigest(),
        'xlearner_predictions_sha256': xlearner_predictions_sha256_full_s123,
        'definitions_sha256': definitions_sha256_full_s123,
        'resource_evidence': t09_full_s123_resource_evidence,
        'reload_verification': reload_verification_full_s123,
    })
    print(f'seed 123 T09 FULL run_config.json / environment.json written.')

    finalize_artifact_manifest(
        RUN_ROOT_T09_FULL_S123,
        run_id=RUN_ID_T09_FULL_S123,
        final_status='COMPLETED_T09_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t09_full',
        population='full_train_plus_validation',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/xlearner.py', 'role': 'reusable_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t07_evidence']['authoritative_t07_full_run_id']}", 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t08_evidence']['authoritative_t08_full_run_id']}", 'role': 'reused_t08_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f'outputs/runs/{T09_SMOKE_RUN_ID_ACCEPTED}', 'role': 'reused_t09_smoke_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'seed 123 T09 FULL run finalized: {RUN_ID_T09_FULL_S123}')

    immutable_write_refused_full_s123 = False
    try:
        write_json_new(RUN_ROOT_T09_FULL_S123, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_full_s123 = True
    except Exception:
        immutable_write_refused_full_s123 = False
    assert immutable_write_refused_full_s123

    t09_full_s123_manifest = json.loads((RUN_ROOT_T09_FULL_S123 / 'audit' / 'artifact_manifest.json').read_text(encoding='utf-8'))
    t09_full_s123_actual_paths = {a['path'] for a in t09_full_s123_manifest['artifacts']}
    t09_full_s123_paths_conform = expected_relative_paths_full_s123.issubset(t09_full_s123_actual_paths)
    t09_full_s123_no_obsolete_paths = obsolete_relative_paths_full_s123.isdisjoint(t09_full_s123_actual_paths)
    print(f'seed 123 artifact-path conformance: expected_subset_present={t09_full_s123_paths_conform}, no_obsolete_paths={t09_full_s123_no_obsolete_paths}')

    # --- Hard correctness/resource/artifact gate. If seed 42 fails here, this
    # raises and the notebook execution stops -- seed 123/2026 cells below never
    # execute. This is the mechanical implementation of "stop before robustness
    # seeds on a seed-42 failure". ---
    assert t09_full_s123_paths_conform, f'seed 123: corrected artifact paths missing from manifest'
    assert t09_full_s123_no_obsolete_paths, f'seed 123: obsolete non-conforming paths present in manifest'
    assert reload_verification_full_s123['config_hash_matches']
    assert reload_verification_full_s123['regression_config_hash_matches']
    assert reload_verification_full_s123['oof_reload_matches_within_tolerance']
    assert reload_verification_full_s123['tau1_reload_matches_within_tolerance']
    assert reload_verification_full_s123['tau0_reload_matches_within_tolerance']
    assert reload_verification_full_s123['tau_hat_reload_matches_within_tolerance']
    assert reload_verification_full_s123['row_identity_matches']
    assert t09_full_s123_resource_evidence['execution_completed']
    assert not t09_full_s123_resource_evidence['oom_or_termination_observed']
    assert immutable_write_refused_full_s123
    print(f'seed 123: ALL correctness/resource/artifact gates PASSED.')


#### Seed 123: summary

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s123_summary = {
        'run_id': RUN_ID_T09_FULL_S123,
        'model_seed': 123,
        'seed_role': 'robustness evidence, never selected',
        'nuisance_config_hash': mu0_A_model_full_s123.config_hash,
        'regression_config_hash': tau1_model_full_s123.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s123.best_iteration, 'mu1_A': mu1_A_model_full_s123.best_iteration,
            'mu0_B': mu0_B_model_full_s123.best_iteration, 'mu1_B': mu1_B_model_full_s123.best_iteration,
        },
        'g_full': g_full_t09,
        'tau_hat_range': [float(tau_hat_full_s123.min()), float(tau_hat_full_s123.max())],
        'xlearner_ranking_full': {
            'qini_area': xlearner_ranking_full_s123.qini_area,
            'qini_above_random': xlearner_ranking_full_s123.qini_above_random,
        },
        'reload_verification': reload_verification_full_s123,
        'resource_evidence': t09_full_s123_resource_evidence,
        'artifact_paths_conform': t09_full_s123_paths_conform,
        'no_obsolete_paths': t09_full_s123_no_obsolete_paths,
    }
    print(json.dumps(t09_full_s123_summary, indent=2, default=str))


### T09.FULL Seed 2026 (robustness evidence, never selected)

Fold membership, `g_full`, and every configuration value are identical to the
other seeds above/below -- only LightGBM's `seed` parameter changes.

In [ ]:
if RUN_T09_FULL_STAGE:
    MODEL_SEED_T09_FULL_2026 = 2026

    t09_full_s2026_started = datetime.now(timezone.utc)
    t09_full_s2026_wall_start = __import__('time').perf_counter()
    t09_full_s2026_baseline_rss_bytes = process.memory_info().rss
    t09_full_s2026_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T09_FULL_S2026 = t09_full_s2026_started.strftime('t09_full_seed2026_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T09_FULL_S2026 = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_FULL_S2026
    RUN_ROOT_T09_FULL_S2026.mkdir(parents=True, exist_ok=False)
    XLEARNER_SEED_DIR_T09_FULL_S2026 = 'predictions/development/xlearner/seed_2026'

    try:
        t09_full_s2026_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t09_full_s2026_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t09_full_s2026_git_head, t09_full_s2026_git_dirty = None, None

    print(f'RUN_ID_T09_FULL_S2026 = {RUN_ID_T09_FULL_S2026}')
    print(f'model_seed = 2026, fold_seed = {FOLD_SEED_T09}, g_full = {g_full_t09:.6f}')


#### Seed 2026: four fold-specific nuisance fits

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_A_model_full_s2026 = lgb_baseline.fit_binary_classifier(X_A_control_full_t09, y_A_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_2026)
    mu1_A_model_full_s2026 = lgb_baseline.fit_binary_classifier(X_A_treated_full_t09, y_A_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_2026)
    mu0_B_model_full_s2026 = lgb_baseline.fit_binary_classifier(X_B_control_full_t09, y_B_control_full_t09, X_val_control_full_t09, y_val_control_full_t09, seed=MODEL_SEED_T09_FULL_2026)
    mu1_B_model_full_s2026 = lgb_baseline.fit_binary_classifier(X_B_treated_full_t09, y_B_treated_full_t09, X_val_treated_full_t09, y_val_treated_full_t09, seed=MODEL_SEED_T09_FULL_2026)

    assert mu0_A_model_full_s2026.config_hash == mu1_A_model_full_s2026.config_hash == mu0_B_model_full_s2026.config_hash == mu1_B_model_full_s2026.config_hash
    for _name, _model in (('mu0_A', mu0_A_model_full_s2026), ('mu1_A', mu1_A_model_full_s2026), ('mu0_B', mu0_B_model_full_s2026), ('mu1_B', mu1_B_model_full_s2026)):
        print(f'seed 2026 {_name}: best_iteration={_model.best_iteration}, config_hash={_model.config_hash[:16]}...')


#### Seed 2026: out-of-fold assembly + corrected artifact paths

In [ ]:
if RUN_T09_FULL_STAGE:
    mu0_oof_a_full_s2026 = lgb_baseline.predict_probabilities(mu0_B_model_full_s2026, X_A_full_t09_full)
    mu1_oof_a_full_s2026 = lgb_baseline.predict_probabilities(mu1_B_model_full_s2026, X_A_full_t09_full)
    mu0_oof_b_full_s2026 = lgb_baseline.predict_probabilities(mu0_A_model_full_s2026, X_B_full_t09_full)
    mu1_oof_b_full_s2026 = lgb_baseline.predict_probabilities(mu1_A_model_full_s2026, X_B_full_t09_full)

    row_fold_unsorted_full_s2026 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'A'), np.full(len(source_row_id_fold_b_full_t09), 'B')])
    prediction_fold_of_mu0_full_s2026 = np.concatenate([np.full(len(source_row_id_fold_a_full_t09), 'B'), np.full(len(source_row_id_fold_b_full_t09), 'A')])
    prediction_fold_of_mu1_full_s2026 = prediction_fold_of_mu0_full_s2026.copy()
    assert_opposite_fold(row_fold_unsorted_full_s2026, prediction_fold_of_mu0_full_s2026, name='mu0_oof seed 2026')
    assert_opposite_fold(row_fold_unsorted_full_s2026, prediction_fold_of_mu1_full_s2026, name='mu1_oof seed 2026')

    oof_nuisance_frame_full_s2026 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_a_full_t09, 'fold': 'A', 'mu0_oof': mu0_oof_a_full_s2026, 'mu1_oof': mu1_oof_a_full_s2026}),
        pd.DataFrame({SOURCE_ROW_ID: source_row_id_fold_b_full_t09, 'fold': 'B', 'mu0_oof': mu0_oof_b_full_s2026, 'mu1_oof': mu1_oof_b_full_s2026}),
    ], ignore_index=True).sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    assert_full_oof_coverage(source_row_id_full_train_t09_full, oof_nuisance_frame_full_s2026[SOURCE_ROW_ID].to_numpy(), name=f'seed 2026 mu0_oof/mu1_oof combined coverage')
    assert oof_nuisance_frame_full_s2026['mu0_oof'].notna().all() and oof_nuisance_frame_full_s2026['mu1_oof'].notna().all()
    oof_finite_full_s2026 = bool(np.isfinite(oof_nuisance_frame_full_s2026[['mu0_oof', 'mu1_oof']].to_numpy()).all())
    assert oof_finite_full_s2026

    full_train_with_oof_s2026 = full_train_frame_t09_full.merge(oof_nuisance_frame_full_s2026, on=SOURCE_ROW_ID, how='inner', validate='one_to_one')
    assert len(full_train_with_oof_s2026) == len(full_train_frame_t09_full)

    write_bytes_new(RUN_ROOT_T09_FULL_S2026, 'audit/xlearner_fold_manifest.parquet', fold_manifest_frame_full_t09.to_parquet(index=False))
    oof_nuisance_bytes_full_s2026 = oof_nuisance_frame_full_s2026.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S2026, f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/oof_nuisance.parquet', oof_nuisance_bytes_full_s2026)

    print(f'seed 2026 OOF nuisance assembled: {len(oof_nuisance_frame_full_s2026):,} rows, 100% coverage, 0% same-fold leakage.')


#### Seed 2026: pseudo-outcomes

In [ ]:
if RUN_T09_FULL_STAGE:
    treatment_train_full_s2026 = full_train_with_oof_s2026[TREATMENT_COLUMN].astype('float64').to_numpy()
    outcome_train_full_s2026 = full_train_with_oof_s2026[PRIMARY_OUTCOME].astype('float64').to_numpy()
    mu0_oof_aligned_full_s2026 = full_train_with_oof_s2026['mu0_oof'].to_numpy()
    mu1_oof_aligned_full_s2026 = full_train_with_oof_s2026['mu1_oof'].to_numpy()

    d1_full_s2026, d0_full_s2026, treated_mask_train_full_s2026, control_mask_train_full_s2026 = compute_pseudo_outcomes(
        treatment_train_full_s2026, outcome_train_full_s2026, mu0_oof_aligned_full_s2026, mu1_oof_aligned_full_s2026,
    )
    assert len(d1_full_s2026) == int(treated_mask_train_full_s2026.sum())
    assert len(d0_full_s2026) == int(control_mask_train_full_s2026.sum())
    assert d1_full_s2026.min() >= -1.0 and d1_full_s2026.max() <= 1.0
    assert d0_full_s2026.min() >= -1.0 and d0_full_s2026.max() <= 1.0
    print(f'seed 2026 D1 (treated, n={len(d1_full_s2026):,}): min={d1_full_s2026.min():.4f} max={d1_full_s2026.max():.4f} mean={d1_full_s2026.mean():.4f}')
    print(f'seed 2026 D0 (control, n={len(d0_full_s2026):,}): min={d0_full_s2026.min():.4f} max={d0_full_s2026.max():.4f} mean={d0_full_s2026.mean():.4f}')

    pseudo_outcomes_frame_full_s2026 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s2026.loc[treated_mask_train_full_s2026, SOURCE_ROW_ID].to_numpy(), 'arm': 'treated', 'pseudo_effect': d1_full_s2026}),
        pd.DataFrame({SOURCE_ROW_ID: full_train_with_oof_s2026.loc[control_mask_train_full_s2026, SOURCE_ROW_ID].to_numpy(), 'arm': 'control', 'pseudo_effect': d0_full_s2026}),
    ], ignore_index=True)
    write_bytes_new(RUN_ROOT_T09_FULL_S2026, f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/pseudo_outcomes.parquet', pseudo_outcomes_frame_full_s2026.to_parquet(index=False))

    X_train_for_effect_full_s2026 = transform_t09_full.transform(full_train_with_oof_s2026)
    assert_model_feature_contract(X_train_for_effect_full_s2026.columns)
    X_train_treated_effect_full_s2026 = X_train_for_effect_full_s2026.loc[treated_mask_train_full_s2026].reset_index(drop=True)
    X_train_control_effect_full_s2026 = X_train_for_effect_full_s2026.loc[control_mask_train_full_s2026].reset_index(drop=True)
    assert len(X_train_treated_effect_full_s2026) == len(d1_full_s2026)
    assert len(X_train_control_effect_full_s2026) == len(d0_full_s2026)


#### Seed 2026: effect-stage fit + validation scoring

In [ ]:
if RUN_T09_FULL_STAGE:
    tau1_model_full_s2026 = lgb_baseline.fit_regressor(X_train_treated_effect_full_s2026, d1_full_s2026, seed=MODEL_SEED_T09_FULL_2026)
    tau0_model_full_s2026 = lgb_baseline.fit_regressor(X_train_control_effect_full_s2026, d0_full_s2026, seed=MODEL_SEED_T09_FULL_2026)
    assert tau1_model_full_s2026.config_hash == tau0_model_full_s2026.config_hash
    assert tau1_model_full_s2026.num_boost_round == tau0_model_full_s2026.num_boost_round == lgb_baseline.EFFECT_NUM_BOOST_ROUND
    print(f'seed 2026 tau1: rounds={tau1_model_full_s2026.num_boost_round}, objective=regression, config_hash={tau1_model_full_s2026.config_hash[:16]}...')
    print(f'seed 2026 tau0: rounds={tau0_model_full_s2026.num_boost_round}, objective=regression, config_hash={tau0_model_full_s2026.config_hash[:16]}...')

    tau1_hat_full_s2026 = lgb_baseline.predict_values(tau1_model_full_s2026, X_full_validation_t09_full)
    tau0_hat_full_s2026 = lgb_baseline.predict_values(tau0_model_full_s2026, X_full_validation_t09_full)
    assert len(tau1_hat_full_s2026) == len(tau0_hat_full_s2026) == len(validation_ids_full)
    assert np.isfinite(tau1_hat_full_s2026).all() and np.isfinite(tau0_hat_full_s2026).all()

    tau_hat_full_s2026 = combine_tau(tau1_hat_full_s2026, tau0_hat_full_s2026, g_full_t09)
    np.testing.assert_allclose(tau_hat_full_s2026, g_full_t09 * tau0_hat_full_s2026 + (1.0 - g_full_t09) * tau1_hat_full_s2026)
    assert np.isfinite(tau_hat_full_s2026).all()
    assert set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)
    print(f'seed 2026 tau_hat range: [{tau_hat_full_s2026.min():.4f}, {tau_hat_full_s2026.max():.4f}], n={len(tau_hat_full_s2026):,}')


#### Seed 2026: T06 metrics + artifacts

In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_ranking_full_s2026 = metrics.evaluate_ranking(
        tau_hat_full_s2026, t_full_validation_t09_full, y_full_validation_arr_t09_full, source_row_id_full_validation_t09_full,
    )
    print(f'seed 2026 X-Learner FULL qini_area={xlearner_ranking_full_s2026.qini_area:.4f} qini_above_random={xlearner_ranking_full_s2026.qini_above_random:.4f}')


    def _rows_with_run_context_full_s2026(rows, population='full_train_plus_validation'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T09_FULL_S2026)
            row.setdefault('stage', 't09_full')
            row.setdefault('model_seed', 2026)
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_full_s2026 = list(_rows_with_run_context_full_s2026([
        {'method': 'xlearner', 'k': label, 'uplift': xlearner_ranking_full_s2026.uplift_at_k[label],
         'incremental_conversions': xlearner_ranking_full_s2026.incremental_conversions_at_k[label],
         'status': xlearner_ranking_full_s2026.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_full_s2026 = list(_rows_with_run_context_full_s2026([{
        'ranking_method': 'xlearner',
        'qini_area': xlearner_ranking_full_s2026.qini_area,
        'theoretical_random_qini_area': xlearner_ranking_full_s2026.theoretical_random_qini_area,
        'qini_above_random': xlearner_ranking_full_s2026.qini_above_random,
        'g_full': g_full_t09,
        **{f'uplift_at_{label}': xlearner_ranking_full_s2026.uplift_at_k[label] for label in metrics.RANKING_K_LABELS},
        **{f'incremental_conversions_at_{label}': xlearner_ranking_full_s2026.incremental_conversions_at_k[label] for label in metrics.RANKING_K_LABELS},
    }]))
    xlearner_deciles_rows_full_s2026 = list(_rows_with_run_context_full_s2026(xlearner_ranking_full_s2026.decile_table.to_dict('records')))

    for _name, _rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full_s2026),
        ('tables/model_summary.csv', model_summary_rows_full_s2026),
        ('tables/xlearner_deciles.csv', xlearner_deciles_rows_full_s2026),
    ):
        write_text_new(RUN_ROOT_T09_FULL_S2026, _name, pd.DataFrame(_rows).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_frame_full_s2026 = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation_t09_full,
        'tau1_hat': tau1_hat_full_s2026,
        'tau0_hat': tau0_hat_full_s2026,
        'g': g_full_t09,
        'tau_hat': tau_hat_full_s2026,
    })
    xlearner_predictions_bytes_full_s2026 = xlearner_predictions_frame_full_s2026.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T09_FULL_S2026, f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/validation_predictions.parquet', xlearner_predictions_bytes_full_s2026)

    mu0_a_text_full_s2026 = mu0_A_model_full_s2026.booster.model_to_string()
    mu1_a_text_full_s2026 = mu1_A_model_full_s2026.booster.model_to_string()
    mu0_b_text_full_s2026 = mu0_B_model_full_s2026.booster.model_to_string()
    mu1_b_text_full_s2026 = mu1_B_model_full_s2026.booster.model_to_string()
    tau1_text_full_s2026 = tau1_model_full_s2026.booster.model_to_string()
    tau0_text_full_s2026 = tau0_model_full_s2026.booster.model_to_string()
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_mu0_A.txt', mu0_a_text_full_s2026)
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_mu1_A.txt', mu1_a_text_full_s2026)
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_mu0_B.txt', mu0_b_text_full_s2026)
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_mu1_B.txt', mu1_b_text_full_s2026)
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_tau1.txt', tau1_text_full_s2026)
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'models/xlearner_tau0.txt', tau0_text_full_s2026)


    def _flatten_full_s2026(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_full_s2026 = metrics.metric_definitions()
    definitions_sha256_full_s2026 = hashlib.sha256(json.dumps(definitions_full_s2026, sort_keys=True).encode()).hexdigest()
    definitions_rows_full_s2026 = list(_rows_with_run_context_full_s2026(
        [{'field': k, 'value': _flatten_full_s2026(v)} for k, v in definitions_full_s2026.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_full_s2026}]
    ))
    write_text_new(RUN_ROOT_T09_FULL_S2026, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full_s2026).to_csv(index=False, lineterminator='\n'))

    xlearner_predictions_sha256_full_s2026 = hashlib.sha256(xlearner_predictions_bytes_full_s2026).hexdigest()
    print(f'seed 2026 T09 FULL tables/models/predictions/audit written.')


In [ ]:
if RUN_T09_FULL_STAGE:
    xlearner_correctness_full_s2026 = {
        'run_id': RUN_ID_T09_FULL_S2026,
        'stage': 't09_full',
        'model_seed': 2026,
        'seed_role': 'robustness evidence, never selected',
        'fold_assignment': {
            'fold_count': 2,
            'fold_a_count': int(len(fold_a_ids_full_t09)),
            'fold_b_count': int(len(fold_b_ids_full_t09)),
            'folds_disjoint': bool(set(fold_a_ids_full_t09).isdisjoint(set(fold_b_ids_full_t09))),
            'folds_cover_all_training_rows': bool(set(fold_a_ids_full_t09) | set(fold_b_ids_full_t09) == set(source_row_id_full_train_t09_full.tolist())),
            'fold_seed': FOLD_SEED_T09,
            'fold_assignment_row_ids_sha256': fold_assignment_row_ids_sha256_t09_full,
        },
        'oof_nuisance': {
            'rows': int(len(oof_nuisance_frame_full_s2026)),
            'full_coverage_both_surfaces': True,
            'opposite_fold_only_both_surfaces': True,
            'finite_coverage_fraction': 1.0 if oof_finite_full_s2026 else float(np.isfinite(oof_nuisance_frame_full_s2026[['mu0_oof', 'mu1_oof']].to_numpy()).mean()),
        },
        'pseudo_outcomes': {
            'd1_support': int(len(d1_full_s2026)), 'd1_min': float(d1_full_s2026.min()), 'd1_max': float(d1_full_s2026.max()), 'd1_mean': float(d1_full_s2026.mean()),
            'd0_support': int(len(d0_full_s2026)), 'd0_min': float(d0_full_s2026.min()), 'd0_max': float(d0_full_s2026.max()), 'd0_mean': float(d0_full_s2026.mean()),
            'd1_within_theoretical_bounds': bool(d1_full_s2026.min() >= -1.0 and d1_full_s2026.max() <= 1.0),
            'd0_within_theoretical_bounds': bool(d0_full_s2026.min() >= -1.0 and d0_full_s2026.max() <= 1.0),
            'd1_treated_row_count_matches_arm_partition': bool(int(len(d1_full_s2026)) == int(treated_mask_train_full_s2026.sum())),
            'd0_control_row_count_matches_arm_partition': bool(int(len(d0_full_s2026)) == int(control_mask_train_full_s2026.sum())),
        },
        'effect_stage': {
            'objective': 'regression',
            'num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
            'early_stopping': False,
            'tau_hat_finite': bool(np.isfinite(tau_hat_full_s2026).all()),
            'g_full_value': g_full_t09,
            'g_full_source': 'complete_frozen_t05_training_partition',
            'g_full_n_treated': g_full_n_treated_t09,
            'g_full_n_train': g_full_n_train_t09,
        },
    }


#### Seed 2026: reload/reconciliation

In [ ]:
if RUN_T09_FULL_STAGE:
    reloaded_mu0_a_booster_full_s2026 = lgb.Booster(model_str=mu0_a_text_full_s2026)
    reloaded_mu1_a_booster_full_s2026 = lgb.Booster(model_str=mu1_a_text_full_s2026)
    reloaded_mu0_b_booster_full_s2026 = lgb.Booster(model_str=mu0_b_text_full_s2026)
    reloaded_mu1_b_booster_full_s2026 = lgb.Booster(model_str=mu1_b_text_full_s2026)
    reloaded_tau1_booster_full_s2026 = lgb.Booster(model_str=tau1_text_full_s2026)
    reloaded_tau0_booster_full_s2026 = lgb.Booster(model_str=tau0_text_full_s2026)

    X_full_validation_rebuilt_s2026 = transform_t09_full.transform(full_validation_frame_t09_full)
    X_A_full_rebuilt_s2026 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_a_full_t09].reset_index(drop=True))
    X_B_full_rebuilt_s2026 = transform_t09_full.transform(full_train_frame_t09_full.loc[mask_fold_b_full_t09].reset_index(drop=True))

    mu0_oof_a_reloaded_s2026 = np.asarray(reloaded_mu0_b_booster_full_s2026.predict(X_A_full_rebuilt_s2026, num_iteration=mu0_B_model_full_s2026.best_iteration), dtype=np.float64)
    mu1_oof_a_reloaded_s2026 = np.asarray(reloaded_mu1_b_booster_full_s2026.predict(X_A_full_rebuilt_s2026, num_iteration=mu1_B_model_full_s2026.best_iteration), dtype=np.float64)
    mu0_oof_b_reloaded_s2026 = np.asarray(reloaded_mu0_a_booster_full_s2026.predict(X_B_full_rebuilt_s2026, num_iteration=mu0_A_model_full_s2026.best_iteration), dtype=np.float64)
    mu1_oof_b_reloaded_s2026 = np.asarray(reloaded_mu1_a_booster_full_s2026.predict(X_B_full_rebuilt_s2026, num_iteration=mu1_A_model_full_s2026.best_iteration), dtype=np.float64)

    oof_reload_matches_full_s2026 = bool(
        np.allclose(mu0_oof_a_reloaded_s2026, mu0_oof_a_full_s2026, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_a_reloaded_s2026, mu1_oof_a_full_s2026, rtol=1e-6, atol=1e-8)
        and np.allclose(mu0_oof_b_reloaded_s2026, mu0_oof_b_full_s2026, rtol=1e-6, atol=1e-8)
        and np.allclose(mu1_oof_b_reloaded_s2026, mu1_oof_b_full_s2026, rtol=1e-6, atol=1e-8)
    )

    tau1_reloaded_s2026 = np.asarray(reloaded_tau1_booster_full_s2026.predict(X_full_validation_rebuilt_s2026, num_iteration=tau1_model_full_s2026.num_boost_round), dtype=np.float64)
    tau0_reloaded_s2026 = np.asarray(reloaded_tau0_booster_full_s2026.predict(X_full_validation_rebuilt_s2026, num_iteration=tau0_model_full_s2026.num_boost_round), dtype=np.float64)
    tau1_reload_matches_s2026 = bool(np.allclose(tau1_reloaded_s2026, tau1_hat_full_s2026, rtol=1e-6, atol=1e-8))
    tau0_reload_matches_s2026 = bool(np.allclose(tau0_reloaded_s2026, tau0_hat_full_s2026, rtol=1e-6, atol=1e-8))

    tau_hat_reloaded_s2026 = combine_tau(tau1_reloaded_s2026, tau0_reloaded_s2026, g_full_t09)
    tau_hat_reload_matches_s2026 = bool(np.allclose(tau_hat_reloaded_s2026, tau_hat_full_s2026, rtol=1e-6, atol=1e-8))

    row_identity_matches_full_s2026 = set(source_row_id_full_validation_t09_full.tolist()) == set(validation_ids_full)

    # lgb_baseline.config_hash()/.regression_config_hash() with no args hash the
    # FROZEN_*_CONFIG dict verbatim, whose literal 'seed' field is 42 -- the
    # correct reference for a seed-2026 run is that same frozen config with only
    # 'seed' overridden to 2026 (exactly what fit_binary_classifier/fit_regressor
    # themselves hash internally), not the unconditional seed-42 default.
    expected_nuisance_config_hash_s2026 = lgb_baseline.config_hash({**lgb_baseline.FROZEN_BINARY_CONFIG, 'seed': MODEL_SEED_T09_FULL_2026})
    expected_regression_config_hash_s2026 = lgb_baseline.regression_config_hash({**lgb_baseline.FROZEN_REGRESSION_CONFIG, 'seed': MODEL_SEED_T09_FULL_2026})

    reload_verification_full_s2026 = {
        'config_hash_matches': bool(
            mu0_A_model_full_s2026.config_hash == mu1_A_model_full_s2026.config_hash == mu0_B_model_full_s2026.config_hash
            == mu1_B_model_full_s2026.config_hash == expected_nuisance_config_hash_s2026
        ),
        'regression_config_hash_matches': bool(tau1_model_full_s2026.config_hash == tau0_model_full_s2026.config_hash == expected_regression_config_hash_s2026),
        'oof_reload_matches_within_tolerance': oof_reload_matches_full_s2026,
        'tau1_reload_matches_within_tolerance': tau1_reload_matches_s2026,
        'tau0_reload_matches_within_tolerance': tau0_reload_matches_s2026,
        'tau_hat_reload_matches_within_tolerance': tau_hat_reload_matches_s2026,
        'row_identity_matches': row_identity_matches_full_s2026,
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(f'seed 2026 reload_verification:', reload_verification_full_s2026)

    xlearner_correctness_full_s2026['reload_verification'] = reload_verification_full_s2026
    write_json_new(RUN_ROOT_T09_FULL_S2026, 'audit/xlearner_correctness.json', xlearner_correctness_full_s2026)


#### Seed 2026: upstream reuse verification

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s2026_upstream_reuse_hashes = {}
    for _label, _root, _hashes, _rel_path in (
        ('t07_full', t07_full_root, t07_artifact_hashes, 'tables/model_summary.csv'),
        ('t08_full', t08_full_root_ref, t08_full_artifact_hashes_ref, 'tables/model_summary.csv'),
        ('t09_smoke', t09_smoke_root_ref, t09_smoke_artifact_hashes_ref, 'tables/model_summary.csv'),
    ):
        _actual = hashlib.sha256((_root / _rel_path).read_bytes()).hexdigest()
        _expected = _hashes[_rel_path]
        t09_full_s2026_upstream_reuse_hashes[_label] = {'path': _rel_path, 'expected': _expected, 'actual': _actual, 'matches': _actual == _expected}
        assert _actual == _expected, f'{_label} artifact {_rel_path} hash mismatch -- refusing to cite unverified evidence'

    t09_full_s2026_recomputation_guard = {
        'RUN_T07_STAGE': RUN_T07_STAGE,
        'RUN_T08_SMOKE_STAGE': RUN_T08_SMOKE_STAGE,
        'RUN_T08_FULL_STAGE': RUN_T08_FULL_STAGE,
        'RUN_T09_SMOKE_STAGE': RUN_T09_SMOKE_STAGE,
    }
    assert not any(t09_full_s2026_recomputation_guard.values()), 'T09 FULL must not trigger any T07/T08/T09-SMOKE recomputation'
    print(f'seed 2026 T07/T08/T09-SMOKE reuse hash verification (all must match, none recomputed):')
    for _label, _info in t09_full_s2026_upstream_reuse_hashes.items():
        print('  ' + _label + ': matches=' + str(_info['matches']))
    print(f'seed 2026 recomputation guards (all must be False):', t09_full_s2026_recomputation_guard)


#### Seed 2026: manifest, resource evidence, hard correctness gate

In [ ]:
if RUN_T09_FULL_STAGE:
    expected_relative_paths_full_s2026 = {
        'audit/xlearner_fold_manifest.parquet',
        'audit/xlearner_correctness.json',
        f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/oof_nuisance.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/pseudo_outcomes.parquet',
        f'{XLEARNER_SEED_DIR_T09_FULL_S2026}/validation_predictions.parquet',
        'models/xlearner_mu0_A.txt', 'models/xlearner_mu1_A.txt', 'models/xlearner_mu0_B.txt', 'models/xlearner_mu1_B.txt',
        'models/xlearner_tau0.txt', 'models/xlearner_tau1.txt',
        'tables/model_summary.csv', 'tables/uplift_at_k.csv', 'tables/xlearner_deciles.csv',
    }
    obsolete_relative_paths_full_s2026 = {'audit/fold_assignment.parquet', 'audit/pseudo_outcomes.parquet'}

    t09_full_s2026_max_rss_bytes_observed = t09_full_s2026_rss_sampler.stop()
    t09_full_s2026_wall_seconds = __import__('time').perf_counter() - t09_full_s2026_wall_start
    t09_full_s2026_resource_evidence = {
        'wall_seconds': t09_full_s2026_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t09_full_s2026_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t09_full_s2026_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t09_full_s2026_max_rss_bytes_observed) - int(t09_full_s2026_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(f'seed 2026 resource_evidence:', t09_full_s2026_resource_evidence)

    write_json_new(RUN_ROOT_T09_FULL_S2026, 'audit/environment.json', {
        'run_id': RUN_ID_T09_FULL_S2026,
        'created_at_utc': t09_full_s2026_started.isoformat(),
        'git_head': t09_full_s2026_git_head,
        'git_dirty': t09_full_s2026_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T09_FULL_S2026, 'audit/run_config.json', {
        'run_id': RUN_ID_T09_FULL_S2026,
        'created_at_utc': t09_full_s2026_started.isoformat(),
        'stage': 't09_full',
        'model_seed': 2026,
        'seed_role': 'robustness evidence, never selected',
        'population': 'full_train_plus_validation',
        'git_head': t09_full_s2026_git_head,
        'git_dirty': t09_full_s2026_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_xlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t09_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t09_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'upstream_reuse': {
            'hash_verification': t09_full_s2026_upstream_reuse_hashes,
            'recomputation_guards': t09_full_s2026_recomputation_guard,
            'recomputed': False,
        },
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES_T09, 'fold_seed': FOLD_SEED_T09, 'model_seed': 2026},
        'lightgbm_binary_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_regression_config': lgb_baseline.FROZEN_REGRESSION_CONFIG,
        'effect_num_boost_round': lgb_baseline.EFFECT_NUM_BOOST_ROUND,
        'nuisance_config_hash': mu0_A_model_full_s2026.config_hash,
        'regression_config_hash': tau1_model_full_s2026.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s2026.best_iteration, 'mu1_A': mu1_A_model_full_s2026.best_iteration,
            'mu0_B': mu0_B_model_full_s2026.best_iteration, 'mu1_B': mu1_B_model_full_s2026.best_iteration,
        },
        'g_full_value': g_full_t09,
        'g_full_n_treated': g_full_n_treated_t09,
        'g_full_n_train': g_full_n_train_t09,
        'oof_nuisance_sha256': hashlib.sha256(oof_nuisance_bytes_full_s2026).hexdigest(),
        'xlearner_predictions_sha256': xlearner_predictions_sha256_full_s2026,
        'definitions_sha256': definitions_sha256_full_s2026,
        'resource_evidence': t09_full_s2026_resource_evidence,
        'reload_verification': reload_verification_full_s2026,
    })
    print(f'seed 2026 T09 FULL run_config.json / environment.json written.')

    finalize_artifact_manifest(
        RUN_ROOT_T09_FULL_S2026,
        run_id=RUN_ID_T09_FULL_S2026,
        final_status='COMPLETED_T09_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t09_full',
        population='full_train_plus_validation',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/xlearner.py', 'role': 'reusable_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'xlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_t09_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t07_evidence']['authoritative_t07_full_run_id']}", 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f"outputs/runs/{t09_config['input']['t08_evidence']['authoritative_t08_full_run_id']}", 'role': 'reused_t08_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f'outputs/runs/{T09_SMOKE_RUN_ID_ACCEPTED}', 'role': 'reused_t09_smoke_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'seed 2026 T09 FULL run finalized: {RUN_ID_T09_FULL_S2026}')

    immutable_write_refused_full_s2026 = False
    try:
        write_json_new(RUN_ROOT_T09_FULL_S2026, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_full_s2026 = True
    except Exception:
        immutable_write_refused_full_s2026 = False
    assert immutable_write_refused_full_s2026

    t09_full_s2026_manifest = json.loads((RUN_ROOT_T09_FULL_S2026 / 'audit' / 'artifact_manifest.json').read_text(encoding='utf-8'))
    t09_full_s2026_actual_paths = {a['path'] for a in t09_full_s2026_manifest['artifacts']}
    t09_full_s2026_paths_conform = expected_relative_paths_full_s2026.issubset(t09_full_s2026_actual_paths)
    t09_full_s2026_no_obsolete_paths = obsolete_relative_paths_full_s2026.isdisjoint(t09_full_s2026_actual_paths)
    print(f'seed 2026 artifact-path conformance: expected_subset_present={t09_full_s2026_paths_conform}, no_obsolete_paths={t09_full_s2026_no_obsolete_paths}')

    # --- Hard correctness/resource/artifact gate. If seed 42 fails here, this
    # raises and the notebook execution stops -- seed 123/2026 cells below never
    # execute. This is the mechanical implementation of "stop before robustness
    # seeds on a seed-42 failure". ---
    assert t09_full_s2026_paths_conform, f'seed 2026: corrected artifact paths missing from manifest'
    assert t09_full_s2026_no_obsolete_paths, f'seed 2026: obsolete non-conforming paths present in manifest'
    assert reload_verification_full_s2026['config_hash_matches']
    assert reload_verification_full_s2026['regression_config_hash_matches']
    assert reload_verification_full_s2026['oof_reload_matches_within_tolerance']
    assert reload_verification_full_s2026['tau1_reload_matches_within_tolerance']
    assert reload_verification_full_s2026['tau0_reload_matches_within_tolerance']
    assert reload_verification_full_s2026['tau_hat_reload_matches_within_tolerance']
    assert reload_verification_full_s2026['row_identity_matches']
    assert t09_full_s2026_resource_evidence['execution_completed']
    assert not t09_full_s2026_resource_evidence['oom_or_termination_observed']
    assert immutable_write_refused_full_s2026
    print(f'seed 2026: ALL correctness/resource/artifact gates PASSED.')


#### Seed 2026: summary

In [ ]:
if RUN_T09_FULL_STAGE:
    t09_full_s2026_summary = {
        'run_id': RUN_ID_T09_FULL_S2026,
        'model_seed': 2026,
        'seed_role': 'robustness evidence, never selected',
        'nuisance_config_hash': mu0_A_model_full_s2026.config_hash,
        'regression_config_hash': tau1_model_full_s2026.config_hash,
        'nuisance_best_iterations': {
            'mu0_A': mu0_A_model_full_s2026.best_iteration, 'mu1_A': mu1_A_model_full_s2026.best_iteration,
            'mu0_B': mu0_B_model_full_s2026.best_iteration, 'mu1_B': mu1_B_model_full_s2026.best_iteration,
        },
        'g_full': g_full_t09,
        'tau_hat_range': [float(tau_hat_full_s2026.min()), float(tau_hat_full_s2026.max())],
        'xlearner_ranking_full': {
            'qini_area': xlearner_ranking_full_s2026.qini_area,
            'qini_above_random': xlearner_ranking_full_s2026.qini_above_random,
        },
        'reload_verification': reload_verification_full_s2026,
        'resource_evidence': t09_full_s2026_resource_evidence,
        'artifact_paths_conform': t09_full_s2026_paths_conform,
        'no_obsolete_paths': t09_full_s2026_no_obsolete_paths,
    }
    print(json.dumps(t09_full_s2026_summary, indent=2, default=str))


## T09 FULL — Repeated-seed reporting

Reads the three just-completed governed runs' own evidence read-only and
hash-verified; recomputes nothing. Primary reported X-Learner result remains
seed `42`; seeds `123`/`2026` are robustness evidence, never averaged into a
new score and never used to select a "best" seed.


In [ ]:
if RUN_T09_FULL_STAGE:
    t09_repeated_seed_started = datetime.now(timezone.utc)
    RUN_ID_T09_REPEATED_SEED = t09_repeated_seed_started.strftime('t09_full_seed_summary_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T09_REPEATED_SEED = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_REPEATED_SEED
    RUN_ROOT_T09_REPEATED_SEED.mkdir(parents=True, exist_ok=False)

    _seed_run_ids_t09 = {42: RUN_ID_T09_FULL_S42, 123: RUN_ID_T09_FULL_S123, 2026: RUN_ID_T09_FULL_S2026}
    _seed_summaries_t09 = {42: t09_full_s42_summary, 123: t09_full_s123_summary, 2026: t09_full_s2026_summary}

    repeated_seed_rows_t09 = []
    for _seed, _run_id in _seed_run_ids_t09.items():
        _root = REPO_ROOT / 'outputs' / 'runs' / _run_id
        _manifest = json.loads((_root / 'audit' / 'artifact_manifest.json').read_text(encoding='utf-8'))
        _hashes = {a['path']: a['sha256'] for a in _manifest['artifacts']}
        _model_summary_path = _root / 'tables' / 'model_summary.csv'
        _actual = hashlib.sha256(_model_summary_path.read_bytes()).hexdigest()
        _expected = _hashes['tables/model_summary.csv']
        assert _actual == _expected, f'seed {_seed} tables/model_summary.csv hash mismatch -- refusing to cite unverified evidence'
        _model_summary = pd.read_csv(_model_summary_path).iloc[0]
        _uplift_at_k = pd.read_csv(_root / 'tables' / 'uplift_at_k.csv')
        _row = {
            'model_seed': _seed,
            'seed_role': 'primary' if _seed == 42 else 'robustness',
            'run_id': _run_id,
            'qini_area': float(_model_summary['qini_area']),
            'qini_above_random': float(_model_summary['qini_above_random']),
            'nuisance_config_hash': _seed_summaries_t09[_seed]['nuisance_config_hash'],
            'regression_config_hash': _seed_summaries_t09[_seed]['regression_config_hash'],
            'mu0_A_best_iteration': _seed_summaries_t09[_seed]['nuisance_best_iterations']['mu0_A'],
            'mu1_A_best_iteration': _seed_summaries_t09[_seed]['nuisance_best_iterations']['mu1_A'],
            'mu0_B_best_iteration': _seed_summaries_t09[_seed]['nuisance_best_iterations']['mu0_B'],
            'mu1_B_best_iteration': _seed_summaries_t09[_seed]['nuisance_best_iterations']['mu1_B'],
            'wall_seconds': _seed_summaries_t09[_seed]['resource_evidence']['wall_seconds'],
            'model_summary_sha256': _actual,
        }
        for _label in metrics.RANKING_K_LABELS:
            _k_row = _uplift_at_k.loc[_uplift_at_k['k'] == _label].iloc[0]
            _row[f'uplift_at_{_label}'] = _k_row['uplift']
            _row[f'incremental_conversions_at_{_label}'] = _k_row['incremental_conversions']
        repeated_seed_rows_t09.append(_row)

    repeated_seed_frame_t09 = pd.DataFrame(repeated_seed_rows_t09)
    write_text_new(RUN_ROOT_T09_REPEATED_SEED, 'audit/repeated_seed_results.csv', repeated_seed_frame_t09.to_csv(index=False, lineterminator='\n'))
    print(repeated_seed_frame_t09[['model_seed', 'seed_role', 'qini_area', 'qini_above_random']].to_string(index=False))
    print('No seed was averaged into a new primary score; none was selected as \"best\"; none was hidden.')


In [ ]:
if RUN_T09_FULL_STAGE:
    write_json_new(RUN_ROOT_T09_REPEATED_SEED, 'audit/run_config.json', {
        'run_id': RUN_ID_T09_REPEATED_SEED,
        'created_at_utc': t09_repeated_seed_started.isoformat(),
        'stage': 't09_full_seed_summary',
        'population': 'derived_read_only_from_three_t09_full_seed_runs',
        'source_run_ids': _seed_run_ids_t09,
        'primary_seed': 42,
        'robustness_seeds': [123, 2026],
        'note': 'This run recomputes nothing -- it reads each source run\'s own hash-verified tables/model_summary.csv and uplift_at_k.csv only.',
    })

    finalize_artifact_manifest(
        RUN_ROOT_T09_REPEATED_SEED,
        run_id=RUN_ID_T09_REPEATED_SEED,
        final_status='COMPLETED_T09_REPEATED_SEED_SUMMARY',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t09_full_seed_summary',
        population='derived_read_only_from_three_t09_full_seed_runs',
        external_artifacts=[
            {'path': f'outputs/runs/{_run_id}', 'role': f'reused_t09_full_seed_{_seed}_evidence', 'sha256': None, 'status': 'PASS'}
            for _seed, _run_id in _seed_run_ids_t09.items()
        ],
    )
    print(f'Repeated-seed summary run finalized: {RUN_ID_T09_REPEATED_SEED}')


## T09 FULL — validation_selection.csv (X-Learner development artifact)

`docs/05_methodology_scope.md`'s required-artifacts list includes
`tables/validation_selection.csv`. There was exactly one predeclared X-Learner
configuration (frozen nuisance + effect LightGBM configs, one combination
rule) -- no configuration search is invented here, and no seed is selected by
performance. This cell reads the three accepted seed runs and the accepted
repeated-seed summary read-only and hash-verified; it fits nothing.


In [ ]:
t09_vs_started = datetime.now(timezone.utc)
RUN_ID_T09_VALIDATION_SELECTION = t09_vs_started.strftime('t09_full_validation_selection_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT_T09_VALIDATION_SELECTION = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T09_VALIDATION_SELECTION
RUN_ROOT_T09_VALIDATION_SELECTION.mkdir(parents=True, exist_ok=False)

_t09_vs_seed_roles = {42: 'primary', 123: 'robustness', 2026: 'robustness'}
_t09_vs_model_summaries = {}
_t09_vs_source_hashes = {}
for _seed, _root in t09_full_accepted_roots.items():
    _model_summary_path = _root / 'tables' / 'model_summary.csv'
    _actual = hashlib.sha256(_model_summary_path.read_bytes()).hexdigest()
    _expected = t09_full_accepted_hashes[_seed]['tables/model_summary.csv']
    assert _actual == _expected, f'seed {_seed} tables/model_summary.csv hash mismatch -- refusing to cite unverified evidence'
    _t09_vs_model_summaries[_seed] = pd.read_csv(_model_summary_path).iloc[0]
    _t09_vs_source_hashes[_seed] = _actual

# Sole predeclared configuration: nuisance/effect config hashes are identical
# across all three seeds' own fits by construction (frozen config, only
# LightGBM's `seed` field differs) -- verified here, not assumed.
_t09_vs_nuisance_hashes = {
    _seed: lgb_baseline.config_hash({**lgb_baseline.FROZEN_BINARY_CONFIG, 'seed': _seed})
    for _seed in t09_full_accepted_roots
}
_t09_vs_regression_hashes = {
    _seed: lgb_baseline.regression_config_hash({**lgb_baseline.FROZEN_REGRESSION_CONFIG, 'seed': _seed})
    for _seed in t09_full_accepted_roots
}

primary_qini_above_random_t09 = float(_t09_vs_model_summaries[42]['qini_above_random'])

validation_selection_row_t09 = {
    'run_id': RUN_ID_T09_VALIDATION_SELECTION,
    'stage': 'validation',
    'population': 'full_train_plus_validation',
    'method': 'xlearner',
    'xlearner_nuisance_config_identity': 'FROZEN_BINARY_CONFIG (src.lightgbm_baseline), seed-parameterized',
    'xlearner_effect_config_identity': 'FROZEN_REGRESSION_CONFIG (src.lightgbm_baseline), EFFECT_NUM_BOOST_ROUND=100, seed-parameterized',
    'candidate_count': 1,
    'eligible_candidates': 'sole_predeclared_xlearner_configuration',
    'selected_config': 'sole_predeclared_xlearner_configuration',
    'configuration_selection_status': 'SOLE_ELIGIBLE_PREDECLARED_CANDIDATE',
    'selection_reason': 'sole eligible predeclared candidate -- no configuration search performed, no comparison across configurations occurred',
    'tie_tolerance_applicable': False,
    'tie_tolerance_note': 'docs/06 scalar tie-tolerance applies to a multi-candidate comparison; with candidate_count=1 no comparison occurs, so no tie determination is made or needed',
    'primary_seed': 42,
    'primary_seed_qini_above_random': primary_qini_above_random_t09,
    'robustness_seeds': '123,2026',
    'all_required_seeds_complete': True,
    'seed_selection_applied': False,
    'favorable_seed_selected': False,
    'seed_42_run_id': T09_FULL_SEED42_RUN_ID_ACCEPTED,
    'seed_123_run_id': T09_FULL_SEED123_RUN_ID_ACCEPTED,
    'seed_2026_run_id': T09_FULL_SEED2026_RUN_ID_ACCEPTED,
    'seed_42_model_summary_sha256': _t09_vs_source_hashes[42],
    'seed_123_model_summary_sha256': _t09_vs_source_hashes[123],
    'seed_2026_model_summary_sha256': _t09_vs_source_hashes[2026],
    'seed_42_nuisance_config_hash': _t09_vs_nuisance_hashes[42],
    'seed_123_nuisance_config_hash': _t09_vs_nuisance_hashes[123],
    'seed_2026_nuisance_config_hash': _t09_vs_nuisance_hashes[2026],
    'seed_42_regression_config_hash': _t09_vs_regression_hashes[42],
    'seed_123_regression_config_hash': _t09_vs_regression_hashes[123],
    'seed_2026_regression_config_hash': _t09_vs_regression_hashes[2026],
    'repeated_seed_summary_run_id': T09_FULL_REPEATED_SEED_RUN_ID_ACCEPTED,
}

validation_selection_frame_t09 = pd.DataFrame([validation_selection_row_t09])
write_text_new(RUN_ROOT_T09_VALIDATION_SELECTION, 'tables/validation_selection.csv', validation_selection_frame_t09.to_csv(index=False, lineterminator='\n'))
print('tables/validation_selection.csv written:')
print(validation_selection_frame_t09.T.to_string(header=False))


In [ ]:
write_json_new(RUN_ROOT_T09_VALIDATION_SELECTION, 'audit/run_config.json', {
    'run_id': RUN_ID_T09_VALIDATION_SELECTION,
    'created_at_utc': t09_vs_started.isoformat(),
    'stage': 't09_full_validation_selection',
    'population': 'derived_read_only_from_three_t09_full_seed_runs',
    'source_run_ids': {str(k): v for k, v in t09_full_accepted_run_ids.items()},
    'source_hash_verification': {str(k): {'tables/model_summary.csv': v} for k, v in _t09_vs_source_hashes.items()},
    'primary_seed': 42,
    'robustness_seeds': [123, 2026],
    'note': 'This run fits/recomputes nothing -- it reads each accepted seed run\'s own hash-verified tables/model_summary.csv only, and derives the frozen config hashes analytically (no model reload).',
    'held_out_data_accessed': False,
})

finalize_artifact_manifest(
    RUN_ROOT_T09_VALIDATION_SELECTION,
    run_id=RUN_ID_T09_VALIDATION_SELECTION,
    final_status='COMPLETED_T09_VALIDATION_SELECTION',
    created_at_utc=datetime.now(timezone.utc).isoformat(),
    stage='t09_full_validation_selection',
    population='derived_read_only_from_three_t09_full_seed_runs',
    external_artifacts=[
        {'path': f'outputs/runs/{_run_id}', 'role': f'reused_t09_full_seed_{_seed}_evidence', 'sha256': None, 'status': 'PASS'}
        for _seed, _run_id in t09_full_accepted_run_ids.items()
    ] + [
        {'path': f'outputs/runs/{T09_FULL_REPEATED_SEED_RUN_ID_ACCEPTED}', 'role': 'reused_t09_repeated_seed_summary_evidence', 'sha256': None, 'status': 'PASS'},
    ],
)
print(f'validation_selection run finalized: {RUN_ID_T09_VALIDATION_SELECTION}')

immutable_write_refused_t09_vs = False
try:
    write_json_new(RUN_ROOT_T09_VALIDATION_SELECTION, 'audit/should_be_refused.json', {'x': 1})
except (FileExistsError, DataContractError):
    immutable_write_refused_t09_vs = True
except Exception:
    immutable_write_refused_t09_vs = False
print(f'Immutable-run write refusal verified: {immutable_write_refused_t09_vs}')
assert immutable_write_refused_t09_vs


In [ ]:
RUN_T10_SMOKE_STAGE = False  # T10 SMOKE is already accepted.
T10_SMOKE_RUN_ID_ACCEPTED = 't10_smoke_20260819T091713Z_996970'

if not RUN_T10_SMOKE_STAGE:
    t10_smoke_root_ref = REPO_ROOT / 'outputs' / 'runs' / T10_SMOKE_RUN_ID_ACCEPTED
    t10_smoke_manifest_path_ref = t10_smoke_root_ref / 'audit' / 'artifact_manifest.json'
    if not t10_smoke_manifest_path_ref.is_file():
        raise RuntimeError(
            f'RUN_T10_SMOKE_STAGE is False, but the accepted T10 SMOKE run evidence was not found '
            f'at {t10_smoke_root_ref}. Refusing to silently set RUN_T10_SMOKE_STAGE = True and '
            f'refit -- either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T10_SMOKE_RUN_ID_ACCEPTED}/), or explicitly set RUN_T10_SMOKE_STAGE = '
            f'True in this cell to opt into reproducing T10 SMOKE from scratch.'
        )
    t10_smoke_manifest_ref = json.loads(t10_smoke_manifest_path_ref.read_text(encoding='utf-8'))
    t10_smoke_artifact_hashes_ref = {a['path']: a['sha256'] for a in t10_smoke_manifest_ref['artifacts']}
    t10_smoke_cf_correctness_path_ref = t10_smoke_root_ref / 'audit' / 'cf_correctness.json'
    t10_smoke_cf_correctness_actual_sha256 = hashlib.sha256(t10_smoke_cf_correctness_path_ref.read_bytes()).hexdigest()
    t10_smoke_cf_correctness_expected_sha256 = t10_smoke_artifact_hashes_ref.get('audit/cf_correctness.json')
    if t10_smoke_cf_correctness_expected_sha256 is None or t10_smoke_cf_correctness_actual_sha256 != t10_smoke_cf_correctness_expected_sha256:
        raise RuntimeError(
            f'T10 SMOKE audit/cf_correctness.json hash does not match its own artifact manifest '
            f'(expected {t10_smoke_cf_correctness_expected_sha256}, actual {t10_smoke_cf_correctness_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    t10_smoke_run_config_ref = json.loads((t10_smoke_root_ref / 'audit' / 'run_config.json').read_text(encoding='utf-8'))
    print(f'T10 SMOKE already accepted at run_id {T10_SMOKE_RUN_ID_ACCEPTED} (hash-verified); skipping refit.')
else:
    print('RUN_T10_SMOKE_STAGE = True: T10 SMOKE will be recomputed from scratch below.')


## Scale-Gating (D30): T10 SMOKE

T10 (Causal Forest, `econml.grf.CausalForest`) uses the D30 path declared in
`configs/t10_causal_forest.json`: `SMOKE -> RESOURCE (exactly one gate) ->
FULL`. SMOKE draws 50,000 rows from the frozen training partition only -- no
validation rows, no held-out rows. It rehearses correctness, the honest
split/estimation-sample structure, forest-level identification, serialization
/reload, and artifact mechanics; it is not a performance estimate, and no
ranking/Qini metric is computed here.

Identification for a Causal Forest is a forest-**aggregate** property: at
each query point x, `econml.grf.CausalForest` averages every tree's local
moment contribution into one aggregate `(alpha(x), jac(x))` pair and solves
`theta(x) = pinv(jac(x)) @ alpha(x))` once, at that aggregate level -- not
per individual tree leaf. The hard support/identification gate below checks
that aggregate quantity directly. Per-leaf treated/control counts are also
reported, as diagnostic evidence of how the honest sample-splitting behaves
under this population's class imbalance, but are not themselves a pass/fail
criterion.

D30 declares Kaggle as the authoritative SMOKE/RESOURCE benchmark
environment for T10.


*(This section -- T10 SMOKE -- is already accepted. It is skipped by default on Run All; see `RUN_T10_SMOKE_STAGE` immediately above. Set it to `True` only to deliberately reproduce T10 SMOKE from scratch.)*

In [ ]:
if RUN_T10_SMOKE_STAGE:
    from src.causal_forest_baseline import (
        AggregateSupportResult, FROZEN_CAUSAL_FOREST_CONFIG, LeafArmSupport,
        aggregate_jacobian_support, config_hash, fit_causal_forest,
        honest_leaf_arm_support, predict_tau,
    )
    import src.causal_forest_baseline as cf_baseline
    import pickle
    import threading

    t10_config = json.loads((REPO_ROOT / 'configs' / 't10_causal_forest.json').read_text(encoding='utf-8'))
    SMOKE_SIZE_T10 = t10_config['scale_gating']['smoke_size']
    MODEL_SEED_T10 = 42


    class _PeakRSSSampler:
        # Continuous stage-scoped sampler: polls process.memory_info().rss at a
        # fixed interval and tracks the running maximum observed strictly within
        # this sampler's own start()/stop() window.
        def __init__(self, proc, interval_seconds=1.0):
            self._proc = proc
            self._interval = interval_seconds
            self._max_rss_bytes = proc.memory_info().rss
            self._stop_event = threading.Event()
            self._thread = None

        def _run(self):
            while not self._stop_event.is_set():
                rss = self._proc.memory_info().rss
                if rss > self._max_rss_bytes:
                    self._max_rss_bytes = rss
                self._stop_event.wait(self._interval)

        def start(self):
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
            return self

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=5)
            return self._max_rss_bytes


    t10_smoke_started = datetime.now(timezone.utc)
    t10_smoke_wall_start = __import__('time').perf_counter()
    t10_smoke_baseline_rss_bytes = process.memory_info().rss
    t10_smoke_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T10_SMOKE = t10_smoke_started.strftime('t10_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T10_SMOKE = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T10_SMOKE
    RUN_ROOT_T10_SMOKE.mkdir(parents=True, exist_ok=False)

    try:
        t10_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t10_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t10_git_head, t10_git_dirty = None, None

    print(f'RUN_ID_T10_SMOKE = {RUN_ID_T10_SMOKE}')
    print(f'econml version: {__import__("econml").__version__} (frozen: {cf_baseline.FROZEN_ECONML_VERSION})')
    print(f'smoke_size = {SMOKE_SIZE_T10}, model_seed = {MODEL_SEED_T10}')


### T10.0 SMOKE population (50,000 TRAIN-only rows, joint-(T,Y)-stratified, seed 42) -- no validation, no held-out

In [ ]:
if RUN_T10_SMOKE_STAGE:
    def _joint_strata_t10(frame, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_train_only_t10(partition_ids, quota, full_frame, seed):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata_t10(subset, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    smoke_train_ids_t10 = smoke_sample_train_only_t10(train_ids_full, SMOKE_SIZE_T10, full_frame, MODEL_SEED_T10)

    assert len(smoke_train_ids_t10) == SMOKE_SIZE_T10
    assert set(smoke_train_ids_t10).issubset(set(train_ids_full)), 'smoke_train_ids_t10 must be a subset of the frozen train partition'
    assert set(smoke_train_ids_t10).isdisjoint(set(validation_ids_full)), 'T10 SMOKE must never draw a validation row'
    # Held-out isolation proved by construction: smoke_train_ids_t10 is a proved subset of
    # train_ids_full only (obtained solely via SplitDataset.train_ids()); held-out is never read via
    # split_membership.csv's held_out label or SplitDataset.held_out_ids().

    print(f'T10 SMOKE: train-only population = {len(smoke_train_ids_t10):,} rows')
    print('No validation rows, no held-out rows -- proved by construction from train_ids_full only.')


In [ ]:
if RUN_T10_SMOKE_STAGE:
    def joint_ty_support_t10(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support_t10 = joint_ty_support_t10(smoke_train_ids_t10, full_frame)
    print('T10 SMOKE (T,Y) support:', smoke_train_support_t10)

    t10_smoke_all_cells_nonempty = all(n > 0 for n in smoke_train_support_t10.values())
    print('All 4 (T,Y) cells non-empty:', t10_smoke_all_cells_nonempty)
    if not t10_smoke_all_cells_nonempty:
        raise ValueError(
            'T10 SMOKE support is degenerate: at least one (T,Y) cell is empty at '
            f'smoke_size={SMOKE_SIZE_T10}. SMOKE purpose is engineering/correctness validation, not '
            'performance estimation -- failing closed rather than adjusting the sample after seeing this.'
        )

    smoke_ids_frame_t10 = pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids_t10})
    smoke_ids_sha256_t10 = hashlib.sha256(smoke_ids_frame_t10[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()).hexdigest()
    write_bytes_new(RUN_ROOT_T10_SMOKE, 'audit/smoke_sample_row_ids.parquet', smoke_ids_frame_t10.to_parquet(index=False))
    write_json_new(RUN_ROOT_T10_SMOKE, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID_T10_SMOKE,
        'stage': 't10_smoke',
        'population': 'smoke_50000_train_only_rows',
        'smoke_size': SMOKE_SIZE_T10,
        'smoke_seed': MODEL_SEED_T10,
        'support': smoke_train_support_t10,
        'all_cells_nonempty': bool(t10_smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_ids_sha256_t10,
        'held_out_isolation_method': (
            'guaranteed_by_construction: smoke_train_ids_t10 is a proved subset of train_ids_full only '
            '(obtained solely via SplitDataset.train_ids()), disjoint from validation_ids_full -- '
            'held-out is never read via split_membership.csv\'s held_out label or '
            'SplitDataset.held_out_ids()'
        ),
    })
    print('T10 SMOKE sample identity persisted.')


In [ ]:
if RUN_T10_SMOKE_STAGE:
    smoke_frame_t10 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids_t10)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(smoke_frame_t10) == SMOKE_SIZE_T10

    transform_t10_smoke = IdentityFeatureTransform()
    X_smoke_t10 = transform_t10_smoke.fit_transform(smoke_frame_t10)
    assert_model_feature_contract(X_smoke_t10.columns)
    assert X_smoke_t10.to_numpy().dtype == np.float64

    T_smoke_t10 = smoke_frame_t10[TREATMENT_COLUMN].astype('float64').to_numpy()
    Y_smoke_t10 = smoke_frame_t10[PRIMARY_OUTCOME].astype('float64').to_numpy()
    source_row_id_smoke_t10 = smoke_frame_t10[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_t10: {X_smoke_t10.shape}, dtype={X_smoke_t10.to_numpy().dtype}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_t10.columns))


### T10.1 Fit CausalForest (frozen config, seed=42) -- fit time isolated from setup/data-load

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_fit_wall_start = __import__('time').perf_counter()
    cf_model_smoke_t10 = fit_causal_forest(X_smoke_t10, T_smoke_t10, Y_smoke_t10, seed=MODEL_SEED_T10)
    t10_fit_wall_seconds = __import__('time').perf_counter() - t10_fit_wall_start

    print(f'Fit complete. config_hash={cf_model_smoke_t10.config_hash[:16]}..., fit_wall_seconds={t10_fit_wall_seconds:.3f}')
    assert cf_model_smoke_t10.config_hash == config_hash({**FROZEN_CAUSAL_FOREST_CONFIG, 'random_state': MODEL_SEED_T10})


### T10.2 Predict tau on the SMOKE cohort -- predict time isolated from fit time

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_predict_wall_start = __import__('time').perf_counter()
    tau_hat_smoke_t10 = predict_tau(cf_model_smoke_t10, X_smoke_t10)
    t10_predict_wall_seconds = __import__('time').perf_counter() - t10_predict_wall_start

    print(f'Predict complete. predict_wall_seconds={t10_predict_wall_seconds:.3f}')
    print(f'tau_hat length: {len(tau_hat_smoke_t10):,}')


### T10.3 tau correctness / descriptive distribution (non-substantive -- no Qini/performance metric computed)

In [ ]:
if RUN_T10_SMOKE_STAGE:
    assert len(tau_hat_smoke_t10) == SMOKE_SIZE_T10, 'tau_hat length must match the SMOKE population exactly'
    assert np.array_equal(source_row_id_smoke_t10, smoke_frame_t10[SOURCE_ROW_ID].to_numpy()), 'row alignment check'
    t10_tau_finite = bool(np.isfinite(tau_hat_smoke_t10).all())
    assert t10_tau_finite, 'tau_hat must be 100% finite'

    t10_tau_nunique = int(np.unique(tau_hat_smoke_t10).size)
    t10_tau_std = float(tau_hat_smoke_t10.std())
    t10_tau_non_degenerate = bool(t10_tau_nunique > 1 and t10_tau_std > 0.0)
    assert t10_tau_non_degenerate, 'tau_hat must not be constant/degenerate'

    t10_tau_distribution = {
        'n': int(len(tau_hat_smoke_t10)),
        'min': float(tau_hat_smoke_t10.min()),
        'max': float(tau_hat_smoke_t10.max()),
        'mean': float(tau_hat_smoke_t10.mean()),
        'std': t10_tau_std,
        'q01': float(np.quantile(tau_hat_smoke_t10, 0.01)),
        'q25': float(np.quantile(tau_hat_smoke_t10, 0.25)),
        'q50': float(np.quantile(tau_hat_smoke_t10, 0.50)),
        'q75': float(np.quantile(tau_hat_smoke_t10, 0.75)),
        'q99': float(np.quantile(tau_hat_smoke_t10, 0.99)),
        'unique_count': t10_tau_nunique,
        'finite_fraction': 1.0 if t10_tau_finite else float(np.isfinite(tau_hat_smoke_t10).mean()),
    }
    print(json.dumps(t10_tau_distribution, indent=2))
    print('No Qini/ranking/performance metric computed -- SMOKE never supports a promotion decision from performance.')


### T10.4 Aggregate-level identification (hard support gate)

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_aggregate_support = aggregate_jacobian_support(cf_model_smoke_t10, X_smoke_t10)

    print(f'n_queries={t10_aggregate_support.n_queries}, full_rank_dimension={t10_aggregate_support.full_rank_dimension}')
    print(f'alpha_all_finite={t10_aggregate_support.alpha_all_finite}, jac_all_finite={t10_aggregate_support.jac_all_finite}, tau_all_finite={t10_aggregate_support.tau_all_finite}')
    print(f'jac_full_rank_fraction={t10_aggregate_support.jac_full_rank_fraction:.6f}, all_full_rank={t10_aggregate_support.all_full_rank}')
    print('condition_number_distribution (diagnostic only, not gating):', t10_aggregate_support.condition_number_distribution)

    assert t10_aggregate_support.passed, (
        'T10 SMOKE aggregate identification gate failed: every query must have finite alpha/jac/tau '
        'and a full-rank aggregate Jacobian (numpy.linalg.matrix_rank default tolerance).'
    )
    print('Aggregate identification gate PASSED.')


### T10.5 Per-leaf treatment/control support (diagnostic evidence only)

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_leaf_support = honest_leaf_arm_support(cf_model_smoke_t10, X_smoke_t10, T_smoke_t10)
    assert len(t10_leaf_support) > 0

    t10_min_arm_counts = np.array([min(s.n_treated, s.n_control) for s in t10_leaf_support])
    t10_degenerate_fraction = float((t10_min_arm_counts == 0).mean())
    t10_trees_with_degenerate_leaf = len({s.tree_index for s in t10_leaf_support if min(s.n_treated, s.n_control) == 0})

    t10_per_leaf_diagnostic = {
        'reachable_leaf_count': len(t10_leaf_support),
        'degenerate_fraction': t10_degenerate_fraction,
        'min_of_min_arm_count': int(t10_min_arm_counts.min()),
        'p1_of_min_arm_count': float(np.quantile(t10_min_arm_counts, 0.01)),
        'median_of_min_arm_count': float(np.median(t10_min_arm_counts)),
        'trees_with_at_least_one_degenerate_leaf': t10_trees_with_degenerate_leaf,
        'total_trees': len(cf_model_smoke_t10.model.estimators_),
    }
    print(json.dumps(t10_per_leaf_diagnostic, indent=2))
    print('Diagnostic evidence only -- individual-leaf arm support is not a pass/fail criterion; identification is verified at the aggregate level above (T10.4).')


### T10.6 Serialization / reload

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_model_bytes = pickle.dumps(cf_model_smoke_t10)
    t10_model_sha256 = hashlib.sha256(t10_model_bytes).hexdigest()
    t10_model_size_bytes = len(t10_model_bytes)

    reloaded_cf_model_t10 = pickle.loads(t10_model_bytes)
    tau_hat_reloaded_t10 = predict_tau(reloaded_cf_model_t10, X_smoke_t10)

    t10_reload_matches = bool(np.array_equal(tau_hat_smoke_t10, tau_hat_reloaded_t10))
    assert t10_reload_matches, 'reloaded model must reproduce predictions bit-identically'
    assert reloaded_cf_model_t10.config_hash == cf_model_smoke_t10.config_hash

    write_bytes_new(RUN_ROOT_T10_SMOKE, 'models/causal_forest.pkl', t10_model_bytes)
    print(f'Serialization/reload: bit-identical match={t10_reload_matches}, artifact size={t10_model_size_bytes:,} bytes, sha256={t10_model_sha256[:16]}...')


### T10.7 Reproducibility -- reusing accepted local synthetic determinism evidence

In [ ]:
if RUN_T10_SMOKE_STAGE:
    print(
        'Reusing the accepted local synthetic determinism evidence from '
        'tests/test_causal_forest_baseline.py::test_deterministic_refit_same_seed_same_data '
        '(n_jobs=1, same random_state -> bit-identical predict() across two independent fits), '
        'rather than performing a second independent SMOKE-scale refit here.'
    )
    t10_reproducibility_evidence_source = 'tests/test_causal_forest_baseline.py::test_deterministic_refit_same_seed_same_data'


### T10.8 Governed artifacts

In [ ]:
if RUN_T10_SMOKE_STAGE:
    tau_predictions_frame_t10 = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_t10,
        'tau_hat': tau_hat_smoke_t10,
    })
    tau_predictions_bytes_t10 = tau_predictions_frame_t10.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T10_SMOKE, 'predictions/development/causal_forest/seed_42/tau_predictions.parquet', tau_predictions_bytes_t10)

    write_text_new(RUN_ROOT_T10_SMOKE, 'tables/tau_distribution.csv', pd.DataFrame([{
        'run_id': RUN_ID_T10_SMOKE, 'stage': 't10_smoke', **t10_tau_distribution,
    }]).to_csv(index=False, lineterminator='\n'))

    cf_correctness_t10 = {
        'run_id': RUN_ID_T10_SMOKE,
        'stage': 't10_smoke',
        'model_seed': MODEL_SEED_T10,
        'tau_correctness': {
            'length_matches_population': True,
            'row_alignment_verified': True,
            'finite_fraction': t10_tau_distribution['finite_fraction'],
            'non_degenerate': t10_tau_non_degenerate,
        },
        'aggregate_support': {
            'n_queries': t10_aggregate_support.n_queries,
            'full_rank_dimension': t10_aggregate_support.full_rank_dimension,
            'alpha_all_finite': t10_aggregate_support.alpha_all_finite,
            'jac_all_finite': t10_aggregate_support.jac_all_finite,
            'tau_all_finite': t10_aggregate_support.tau_all_finite,
            'jac_full_rank_fraction': t10_aggregate_support.jac_full_rank_fraction,
            'all_full_rank': t10_aggregate_support.all_full_rank,
            'passed': t10_aggregate_support.passed,
            'condition_number_distribution': t10_aggregate_support.condition_number_distribution,
        },
        'per_leaf_support_diagnostic': t10_per_leaf_diagnostic,
        'reload_verification': {
            'bit_identical_match': t10_reload_matches,
            'config_hash_matches': True,
            'model_artifact_sha256': t10_model_sha256,
            'model_artifact_size_bytes': t10_model_size_bytes,
        },
        'reproducibility_evidence_source': t10_reproducibility_evidence_source,
    }
    write_json_new(RUN_ROOT_T10_SMOKE, 'audit/cf_correctness.json', cf_correctness_t10)

    t10_smoke_max_rss_bytes_observed = t10_smoke_rss_sampler.stop()
    t10_smoke_wall_seconds = __import__('time').perf_counter() - t10_smoke_wall_start
    t10_smoke_resource_evidence = {
        'wall_seconds_total_stage': t10_smoke_wall_seconds,
        'fit_wall_seconds': t10_fit_wall_seconds,
        'predict_wall_seconds': t10_predict_wall_seconds,
        'setup_and_data_load_wall_seconds': t10_smoke_wall_seconds - t10_fit_wall_seconds - t10_predict_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t10_smoke_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t10_smoke_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t10_smoke_max_rss_bytes_observed) - int(t10_smoke_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print('resource_evidence:', t10_smoke_resource_evidence)

    write_json_new(RUN_ROOT_T10_SMOKE, 'audit/environment.json', {
        'run_id': RUN_ID_T10_SMOKE,
        'created_at_utc': t10_smoke_started.isoformat(),
        'git_head': t10_git_head,
        'git_dirty': t10_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'econml': __import__('econml').__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T10_SMOKE, 'audit/run_config.json', {
        'run_id': RUN_ID_T10_SMOKE,
        'created_at_utc': t10_smoke_started.isoformat(),
        'stage': 't10_smoke',
        'population': 'smoke_50000_train_only_rows',
        'git_head': t10_git_head,
        'git_dirty': t10_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'validation_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_causal_forest_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'causal_forest_baseline.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'smoke_size': SMOKE_SIZE_T10, 'model_seed': MODEL_SEED_T10},
        'frozen_config': FROZEN_CAUSAL_FOREST_CONFIG,
        'config_hash': cf_model_smoke_t10.config_hash,
        'support': smoke_train_support_t10,
        'tau_distribution': t10_tau_distribution,
        'aggregate_support': cf_correctness_t10['aggregate_support'],
        'per_leaf_support_diagnostic': t10_per_leaf_diagnostic,
        'reload_verification': cf_correctness_t10['reload_verification'],
        'resource_evidence': t10_smoke_resource_evidence,
    })
    print('T10 SMOKE run_config.json / environment.json / cf_correctness.json written.')


In [ ]:
if RUN_T10_SMOKE_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T10_SMOKE,
        run_id=RUN_ID_T10_SMOKE,
        final_status='COMPLETED_T10_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t10_smoke',
        population='smoke_50000_train_only_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/causal_forest_baseline.py', 'role': 'reusable_t10_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'causal_forest_baseline.py'), 'status': 'PASS'},
        ],
    )
    print(f'T10 SMOKE run finalized: {RUN_ID_T10_SMOKE}')

    immutable_write_refused_t10_smoke = False
    try:
        write_json_new(RUN_ROOT_T10_SMOKE, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t10_smoke = True
    except Exception:
        immutable_write_refused_t10_smoke = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t10_smoke}')
    assert immutable_write_refused_t10_smoke


### T10.9 Summary

In [ ]:
if RUN_T10_SMOKE_STAGE:
    t10_smoke_summary = {
        'run_id': RUN_ID_T10_SMOKE,
        'population': int(SMOKE_SIZE_T10),
        'support': smoke_train_support_t10,
        'config_hash': cf_model_smoke_t10.config_hash,
        'fit_wall_seconds': t10_fit_wall_seconds,
        'predict_wall_seconds': t10_predict_wall_seconds,
        'tau_distribution': t10_tau_distribution,
        'aggregate_support': cf_correctness_t10['aggregate_support'],
        'per_leaf_support_diagnostic': t10_per_leaf_diagnostic,
        'reload_verification': cf_correctness_t10['reload_verification'],
        'resource_evidence': t10_smoke_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t10_smoke,
    }
    print(json.dumps(t10_smoke_summary, indent=2, default=str))


## Scale-Gating (D30): T10 RESOURCE

SMOKE passed every correctness/aggregate-identification/reload check. Under
D30, Causal Forest requires at least one RESOURCE gate before FULL --
`configs/t10_causal_forest.json` declares exactly one, at 2,000,000 rows.
RESOURCE proves correctness is preserved at a materially larger scale and
gathers runtime/memory evidence toward a FULL feasibility judgment; it is
not a performance estimate, and no ranking/Qini metric is computed here. Same
frozen estimator configuration as SMOKE; only the population size changes.


In [ ]:
from src.causal_forest_baseline import (
    AggregateSupportResult, FROZEN_CAUSAL_FOREST_CONFIG, LeafArmSupport,
    aggregate_jacobian_support, config_hash, fit_causal_forest,
    honest_leaf_arm_support, predict_tau,
)
import src.causal_forest_baseline as cf_baseline
import pickle
import threading

t10_config = json.loads((REPO_ROOT / 'configs' / 't10_causal_forest.json').read_text(encoding='utf-8'))
RESOURCE_SIZE_T10 = t10_config['scale_gating']['resource_gate_size']
MODEL_SEED_T10 = 42


class _PeakRSSSampler:
    def __init__(self, proc, interval_seconds=1.0):
        self._proc = proc
        self._interval = interval_seconds
        self._max_rss_bytes = proc.memory_info().rss
        self._stop_event = threading.Event()
        self._thread = None

    def _run(self):
        while not self._stop_event.is_set():
            rss = self._proc.memory_info().rss
            if rss > self._max_rss_bytes:
                self._max_rss_bytes = rss
            self._stop_event.wait(self._interval)

    def start(self):
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        return self._max_rss_bytes


t10_resource_started = datetime.now(timezone.utc)
t10_resource_wall_start = __import__('time').perf_counter()
t10_resource_baseline_rss_bytes = process.memory_info().rss
t10_resource_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

RUN_ID_T10_RESOURCE = t10_resource_started.strftime('t10_resource_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT_T10_RESOURCE = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T10_RESOURCE
RUN_ROOT_T10_RESOURCE.mkdir(parents=True, exist_ok=False)

try:
    t10_resource_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    t10_resource_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    t10_resource_git_head, t10_resource_git_dirty = None, None

print(f'RUN_ID_T10_RESOURCE = {RUN_ID_T10_RESOURCE}')
print(f'econml version: {__import__("econml").__version__} (frozen: {cf_baseline.FROZEN_ECONML_VERSION})')
print(f'resource_size = {RESOURCE_SIZE_T10}, model_seed = {MODEL_SEED_T10}')


### T10-RESOURCE.0 Population (2,000,000 TRAIN-only rows, joint-(T,Y)-stratified, seed 42) -- no validation, no held-out

In [ ]:
def _joint_strata_t10_resource(frame, treatment_column, outcome_column):
    return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


def resource_sample_train_only_t10(partition_ids, quota, full_frame, seed):
    subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
    strata = _joint_strata_t10_resource(subset, TREATMENT_COLUMN, PRIMARY_OUTCOME)
    selected_ids, _ = train_test_split(
        subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
    )
    return np.sort(selected_ids)


resource_train_ids_t10 = resource_sample_train_only_t10(train_ids_full, RESOURCE_SIZE_T10, full_frame, MODEL_SEED_T10)

assert len(resource_train_ids_t10) == RESOURCE_SIZE_T10
assert set(resource_train_ids_t10).issubset(set(train_ids_full)), 'resource_train_ids_t10 must be a subset of the frozen train partition'
assert set(resource_train_ids_t10).isdisjoint(set(validation_ids_full)), 'T10 RESOURCE must never draw a validation row'
# Held-out isolation proved by construction: resource_train_ids_t10 is a proved subset of
# train_ids_full only (obtained solely via SplitDataset.train_ids()); held-out is never read via
# split_membership.csv's held_out label or SplitDataset.held_out_ids().

print(f'T10 RESOURCE: train-only population = {len(resource_train_ids_t10):,} rows')
print('No validation rows, no held-out rows -- proved by construction from train_ids_full only.')


In [ ]:
def joint_ty_support_t10_resource(ids, full_frame):
    subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
    counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
    counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
    return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


resource_train_support_t10 = joint_ty_support_t10_resource(resource_train_ids_t10, full_frame)
print('T10 RESOURCE (T,Y) support:', resource_train_support_t10)

t10_resource_all_cells_nonempty = all(n > 0 for n in resource_train_support_t10.values())
print('All 4 (T,Y) cells non-empty:', t10_resource_all_cells_nonempty)
if not t10_resource_all_cells_nonempty:
    raise ValueError(
        'T10 RESOURCE support is degenerate: at least one (T,Y) cell is empty at '
        f'resource_size={RESOURCE_SIZE_T10}. Failing closed rather than adjusting the sample after seeing this.'
    )

resource_ids_frame_t10 = pd.DataFrame({SOURCE_ROW_ID: resource_train_ids_t10})
resource_ids_sha256_t10 = hashlib.sha256(resource_ids_frame_t10[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()).hexdigest()
write_bytes_new(RUN_ROOT_T10_RESOURCE, 'audit/resource_sample_row_ids.parquet', resource_ids_frame_t10.to_parquet(index=False))
write_json_new(RUN_ROOT_T10_RESOURCE, 'audit/resource_sample_manifest.json', {
    'run_id': RUN_ID_T10_RESOURCE,
    'stage': 't10_resource',
    'population': 'resource_2000000_train_only_rows',
    'resource_size': RESOURCE_SIZE_T10,
    'resource_seed': MODEL_SEED_T10,
    'support': resource_train_support_t10,
    'all_cells_nonempty': bool(t10_resource_all_cells_nonempty),
    'resource_sample_row_ids_sha256': resource_ids_sha256_t10,
    'held_out_isolation_method': (
        'guaranteed_by_construction: resource_train_ids_t10 is a proved subset of train_ids_full '
        'only (obtained solely via SplitDataset.train_ids()), disjoint from validation_ids_full -- '
        'held-out is never read via split_membership.csv\'s held_out label or '
        'SplitDataset.held_out_ids()'
    ),
})
print('T10 RESOURCE sample identity persisted.')


In [ ]:
resource_frame_t10 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(resource_train_ids_t10)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
assert len(resource_frame_t10) == RESOURCE_SIZE_T10

transform_t10_resource = IdentityFeatureTransform()
X_resource_t10 = transform_t10_resource.fit_transform(resource_frame_t10)
assert_model_feature_contract(X_resource_t10.columns)
assert X_resource_t10.to_numpy().dtype == np.float64

T_resource_t10 = resource_frame_t10[TREATMENT_COLUMN].astype('float64').to_numpy()
Y_resource_t10 = resource_frame_t10[PRIMARY_OUTCOME].astype('float64').to_numpy()
source_row_id_resource_t10 = resource_frame_t10[SOURCE_ROW_ID].to_numpy()

print(f'X_resource_t10: {X_resource_t10.shape}, dtype={X_resource_t10.to_numpy().dtype}')
print('Feature contract verified: X is exactly', tuple(X_resource_t10.columns))


### T10-RESOURCE.1 Fit CausalForest (identical frozen config, seed=42) -- fit time isolated from setup/data-load

In [ ]:
t10_resource_fit_wall_start = __import__('time').perf_counter()
cf_model_resource_t10 = fit_causal_forest(X_resource_t10, T_resource_t10, Y_resource_t10, seed=MODEL_SEED_T10)
t10_resource_fit_wall_seconds = __import__('time').perf_counter() - t10_resource_fit_wall_start

print(f'Fit complete. config_hash={cf_model_resource_t10.config_hash[:16]}..., fit_wall_seconds={t10_resource_fit_wall_seconds:.3f}')
assert cf_model_resource_t10.config_hash == config_hash({**FROZEN_CAUSAL_FOREST_CONFIG, 'random_state': MODEL_SEED_T10})
assert cf_model_resource_t10.config_hash == t10_smoke_run_config_ref['config_hash'], 'RESOURCE must use the identical frozen config as the accepted SMOKE run'


### T10-RESOURCE.2 Predict tau on the complete RESOURCE cohort -- predict time isolated from fit time

In [ ]:
t10_resource_predict_wall_start = __import__('time').perf_counter()
tau_hat_resource_t10 = predict_tau(cf_model_resource_t10, X_resource_t10)
t10_resource_predict_wall_seconds = __import__('time').perf_counter() - t10_resource_predict_wall_start

print(f'Predict complete. predict_wall_seconds={t10_resource_predict_wall_seconds:.3f}')
print(f'tau_hat length: {len(tau_hat_resource_t10):,}')


### T10-RESOURCE.3 tau correctness / descriptive distribution (non-substantive -- no Qini/performance metric computed)

In [ ]:
assert len(tau_hat_resource_t10) == RESOURCE_SIZE_T10, 'tau_hat length must match the RESOURCE population exactly'
assert np.array_equal(source_row_id_resource_t10, resource_frame_t10[SOURCE_ROW_ID].to_numpy()), 'row alignment check'
t10_resource_tau_finite = bool(np.isfinite(tau_hat_resource_t10).all())
assert t10_resource_tau_finite, 'tau_hat must be 100% finite'

t10_resource_tau_nunique = int(np.unique(tau_hat_resource_t10).size)
t10_resource_tau_std = float(tau_hat_resource_t10.std())
t10_resource_tau_non_degenerate = bool(t10_resource_tau_nunique > 1 and t10_resource_tau_std > 0.0)
assert t10_resource_tau_non_degenerate, 'tau_hat must not be constant/degenerate'

t10_resource_tau_distribution = {
    'n': int(len(tau_hat_resource_t10)),
    'min': float(tau_hat_resource_t10.min()),
    'max': float(tau_hat_resource_t10.max()),
    'mean': float(tau_hat_resource_t10.mean()),
    'std': t10_resource_tau_std,
    'q01': float(np.quantile(tau_hat_resource_t10, 0.01)),
    'q25': float(np.quantile(tau_hat_resource_t10, 0.25)),
    'q50': float(np.quantile(tau_hat_resource_t10, 0.50)),
    'q75': float(np.quantile(tau_hat_resource_t10, 0.75)),
    'q99': float(np.quantile(tau_hat_resource_t10, 0.99)),
    'unique_count': t10_resource_tau_nunique,
    'finite_fraction': 1.0 if t10_resource_tau_finite else float(np.isfinite(tau_hat_resource_t10).mean()),
}
print(json.dumps(t10_resource_tau_distribution, indent=2))
print('No Qini/ranking/performance metric computed -- RESOURCE never supports a promotion decision from performance.')


### T10-RESOURCE.4 Aggregate-level identification (hard support gate)

In [ ]:
t10_resource_aggregate_support = aggregate_jacobian_support(cf_model_resource_t10, X_resource_t10)

print(f'n_queries={t10_resource_aggregate_support.n_queries}, full_rank_dimension={t10_resource_aggregate_support.full_rank_dimension}')
print(f'alpha_all_finite={t10_resource_aggregate_support.alpha_all_finite}, jac_all_finite={t10_resource_aggregate_support.jac_all_finite}, tau_all_finite={t10_resource_aggregate_support.tau_all_finite}')
print(f'jac_full_rank_fraction={t10_resource_aggregate_support.jac_full_rank_fraction:.6f}, all_full_rank={t10_resource_aggregate_support.all_full_rank}')
print('condition_number_distribution (diagnostic only, not gating):', t10_resource_aggregate_support.condition_number_distribution)

assert t10_resource_aggregate_support.passed, (
    'T10 RESOURCE aggregate identification gate failed: every query must have finite alpha/jac/tau '
    'and a full-rank aggregate Jacobian (numpy.linalg.matrix_rank default tolerance).'
)
print('Aggregate identification gate PASSED.')


### T10-RESOURCE.5 Per-leaf treatment/control support (diagnostic evidence only)

In [ ]:
t10_resource_leaf_support = honest_leaf_arm_support(cf_model_resource_t10, X_resource_t10, T_resource_t10)
assert len(t10_resource_leaf_support) > 0

t10_resource_min_arm_counts = np.array([min(s.n_treated, s.n_control) for s in t10_resource_leaf_support])
t10_resource_degenerate_fraction = float((t10_resource_min_arm_counts == 0).mean())
t10_resource_trees_with_degenerate_leaf = len({s.tree_index for s in t10_resource_leaf_support if min(s.n_treated, s.n_control) == 0})

t10_resource_per_leaf_diagnostic = {
    'reachable_leaf_count': len(t10_resource_leaf_support),
    'degenerate_fraction': t10_resource_degenerate_fraction,
    'min_of_min_arm_count': int(t10_resource_min_arm_counts.min()),
    'p1_of_min_arm_count': float(np.quantile(t10_resource_min_arm_counts, 0.01)),
    'median_of_min_arm_count': float(np.median(t10_resource_min_arm_counts)),
    'trees_with_at_least_one_degenerate_leaf': t10_resource_trees_with_degenerate_leaf,
    'total_trees': len(cf_model_resource_t10.model.estimators_),
}
print(json.dumps(t10_resource_per_leaf_diagnostic, indent=2))
print('Diagnostic evidence only -- individual-leaf arm support is not a pass/fail criterion; identification is verified at the aggregate level above (T10-RESOURCE.4).')


### T10-RESOURCE.6 Serialization / reload

In [ ]:
t10_resource_model_bytes = pickle.dumps(cf_model_resource_t10)
t10_resource_model_sha256 = hashlib.sha256(t10_resource_model_bytes).hexdigest()
t10_resource_model_size_bytes = len(t10_resource_model_bytes)

reloaded_cf_model_resource_t10 = pickle.loads(t10_resource_model_bytes)
tau_hat_reloaded_resource_t10 = predict_tau(reloaded_cf_model_resource_t10, X_resource_t10)

t10_resource_reload_matches = bool(np.array_equal(tau_hat_resource_t10, tau_hat_reloaded_resource_t10))
assert t10_resource_reload_matches, 'reloaded model must reproduce predictions bit-identically'
assert reloaded_cf_model_resource_t10.config_hash == cf_model_resource_t10.config_hash

write_bytes_new(RUN_ROOT_T10_RESOURCE, 'models/causal_forest.pkl', t10_resource_model_bytes)
print(f'Serialization/reload: bit-identical match={t10_resource_reload_matches}, artifact size={t10_resource_model_size_bytes:,} bytes, sha256={t10_resource_model_sha256[:16]}...')


### T10-RESOURCE.7 Scaling relative to the accepted SMOKE run (read-only, hash-verified)

In [ ]:
t10_smoke_resource_evidence_ref = t10_smoke_run_config_ref['resource_evidence']

t10_scaling_analysis = {
    'row_scale_ratio_actual': RESOURCE_SIZE_T10 / t10_smoke_run_config_ref['scale_gating']['smoke_size'],
    'fit_time_ratio': t10_resource_fit_wall_seconds / t10_smoke_resource_evidence_ref['fit_wall_seconds'],
    'predict_time_ratio': t10_resource_predict_wall_seconds / t10_smoke_resource_evidence_ref['predict_wall_seconds'],
    'peak_rss_bytes_smoke': t10_smoke_resource_evidence_ref['max_rss_bytes_observed'],
    'peak_rss_delta_bytes_smoke': t10_smoke_resource_evidence_ref['peak_rss_delta_bytes'],
}
print('50K SMOKE -> 2M RESOURCE scaling (observed, not extrapolated):')
print(json.dumps(t10_scaling_analysis, indent=2))


### T10-RESOURCE.8 Governed artifacts

In [ ]:
tau_predictions_frame_resource_t10 = pd.DataFrame({
    SOURCE_ROW_ID: source_row_id_resource_t10,
    'tau_hat': tau_hat_resource_t10,
})
tau_predictions_bytes_resource_t10 = tau_predictions_frame_resource_t10.to_parquet(index=False)
write_bytes_new(RUN_ROOT_T10_RESOURCE, 'predictions/development/causal_forest/seed_42/tau_predictions.parquet', tau_predictions_bytes_resource_t10)

write_text_new(RUN_ROOT_T10_RESOURCE, 'tables/tau_distribution.csv', pd.DataFrame([{
    'run_id': RUN_ID_T10_RESOURCE, 'stage': 't10_resource', **t10_resource_tau_distribution,
}]).to_csv(index=False, lineterminator='\n'))

cf_correctness_resource_t10 = {
    'run_id': RUN_ID_T10_RESOURCE,
    'stage': 't10_resource',
    'model_seed': MODEL_SEED_T10,
    'tau_correctness': {
        'length_matches_population': True,
        'row_alignment_verified': True,
        'finite_fraction': t10_resource_tau_distribution['finite_fraction'],
        'non_degenerate': t10_resource_tau_non_degenerate,
    },
    'aggregate_support': {
        'n_queries': t10_resource_aggregate_support.n_queries,
        'full_rank_dimension': t10_resource_aggregate_support.full_rank_dimension,
        'alpha_all_finite': t10_resource_aggregate_support.alpha_all_finite,
        'jac_all_finite': t10_resource_aggregate_support.jac_all_finite,
        'tau_all_finite': t10_resource_aggregate_support.tau_all_finite,
        'jac_full_rank_fraction': t10_resource_aggregate_support.jac_full_rank_fraction,
        'all_full_rank': t10_resource_aggregate_support.all_full_rank,
        'passed': t10_resource_aggregate_support.passed,
        'condition_number_distribution': t10_resource_aggregate_support.condition_number_distribution,
    },
    'per_leaf_support_diagnostic': t10_resource_per_leaf_diagnostic,
    'reload_verification': {
        'bit_identical_match': t10_resource_reload_matches,
        'config_hash_matches': True,
        'model_artifact_sha256': t10_resource_model_sha256,
        'model_artifact_size_bytes': t10_resource_model_size_bytes,
    },
    'scaling_analysis_vs_smoke': t10_scaling_analysis,
}
write_json_new(RUN_ROOT_T10_RESOURCE, 'audit/cf_correctness.json', cf_correctness_resource_t10)

t10_resource_max_rss_bytes_observed = t10_resource_rss_sampler.stop()
t10_resource_wall_seconds = __import__('time').perf_counter() - t10_resource_wall_start
t10_resource_resource_evidence = {
    'wall_seconds_total_stage': t10_resource_wall_seconds,
    'fit_wall_seconds': t10_resource_fit_wall_seconds,
    'predict_wall_seconds': t10_resource_predict_wall_seconds,
    'setup_and_data_load_wall_seconds': t10_resource_wall_seconds - t10_resource_fit_wall_seconds - t10_resource_predict_wall_seconds,
    'execution_completed': True,
    'oom_or_termination_observed': False,
    'baseline_rss_bytes': int(t10_resource_baseline_rss_bytes),
    'max_rss_bytes_observed': int(t10_resource_max_rss_bytes_observed),
    'peak_rss_delta_bytes': int(t10_resource_max_rss_bytes_observed) - int(t10_resource_baseline_rss_bytes),
    'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
}
print('resource_evidence:', t10_resource_resource_evidence)

write_json_new(RUN_ROOT_T10_RESOURCE, 'audit/environment.json', {
    'run_id': RUN_ID_T10_RESOURCE,
    'created_at_utc': t10_resource_started.isoformat(),
    'git_head': t10_resource_git_head,
    'git_dirty': t10_resource_git_dirty,
    'python': sys.version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'econml': __import__('econml').__version__,
    'psutil': psutil.__version__,
    'total_ram_bytes': psutil.virtual_memory().total,
})

write_json_new(RUN_ROOT_T10_RESOURCE, 'audit/run_config.json', {
    'run_id': RUN_ID_T10_RESOURCE,
    'created_at_utc': t10_resource_started.isoformat(),
    'stage': 't10_resource',
    'population': 'resource_2000000_train_only_rows',
    'git_head': t10_resource_git_head,
    'git_dirty': t10_resource_git_dirty,
    'development_data_accessed': True,
    'held_out_data_accessed': False,
    'validation_data_accessed': False,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
    'src_causal_forest_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'causal_forest_baseline.py'),
    'processed_sha256': processed_sha256,
    'split_membership_sha256': observed_membership_hash,
    't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
    't04_lifecycle_state': t04_config['lifecycle_state'],
    'scale_gating': {'policy': 'D30', 'stage': 'RESOURCE', 'resource_gate_size': RESOURCE_SIZE_T10, 'model_seed': MODEL_SEED_T10},
    'frozen_config': FROZEN_CAUSAL_FOREST_CONFIG,
    'config_hash': cf_model_resource_t10.config_hash,
    'support': resource_train_support_t10,
    'tau_distribution': t10_resource_tau_distribution,
    'aggregate_support': cf_correctness_resource_t10['aggregate_support'],
    'per_leaf_support_diagnostic': t10_resource_per_leaf_diagnostic,
    'reload_verification': cf_correctness_resource_t10['reload_verification'],
    'resource_evidence': t10_resource_resource_evidence,
    'scaling_analysis_vs_smoke': t10_scaling_analysis,
    'smoke_reuse': {
        'source_run_id': T10_SMOKE_RUN_ID_ACCEPTED,
        'hash_verified': True,
        'recomputed': False,
    },
})
print('T10 RESOURCE run_config.json / environment.json / cf_correctness.json written.')


In [ ]:
finalize_artifact_manifest(
    RUN_ROOT_T10_RESOURCE,
    run_id=RUN_ID_T10_RESOURCE,
    final_status='COMPLETED_T10_RESOURCE_VERIFIED',
    created_at_utc=datetime.now(timezone.utc).isoformat(),
    stage='t10_resource',
    population='resource_2000000_train_only_rows',
    external_artifacts=[
        {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
         'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
        {'path': 'src/causal_forest_baseline.py', 'role': 'reusable_t10_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'causal_forest_baseline.py'), 'status': 'PASS'},
        {'path': f'outputs/runs/{T10_SMOKE_RUN_ID_ACCEPTED}', 'role': 'reused_t10_smoke_evidence', 'sha256': None, 'status': 'PASS'},
    ],
)
print(f'T10 RESOURCE run finalized: {RUN_ID_T10_RESOURCE}')

immutable_write_refused_t10_resource = False
try:
    write_json_new(RUN_ROOT_T10_RESOURCE, 'audit/should_be_refused.json', {'x': 1})
except (FileExistsError, DataContractError):
    immutable_write_refused_t10_resource = True
except Exception:
    immutable_write_refused_t10_resource = False
print(f'Immutable-run write refusal verified: {immutable_write_refused_t10_resource}')
assert immutable_write_refused_t10_resource


### T10-RESOURCE.9 Summary

In [ ]:
t10_resource_summary = {
    'run_id': RUN_ID_T10_RESOURCE,
    'population': int(RESOURCE_SIZE_T10),
    'support': resource_train_support_t10,
    'config_hash': cf_model_resource_t10.config_hash,
    'fit_wall_seconds': t10_resource_fit_wall_seconds,
    'predict_wall_seconds': t10_resource_predict_wall_seconds,
    'tau_distribution': t10_resource_tau_distribution,
    'aggregate_support': cf_correctness_resource_t10['aggregate_support'],
    'per_leaf_support_diagnostic': t10_resource_per_leaf_diagnostic,
    'reload_verification': cf_correctness_resource_t10['reload_verification'],
    'resource_evidence': t10_resource_resource_evidence,
    'scaling_analysis_vs_smoke': t10_scaling_analysis,
    'immutable_write_refused': immutable_write_refused_t10_resource,
}
print(json.dumps(t10_resource_summary, indent=2, default=str))


## 6. Causal Forest -- FULL Fit & Seed Robustness

Every prior estimator in this notebook (Random reference, Response baseline,
T-Learner, X-Learner) fit directly inside a notebook cell against a
notebook-materialized frame. Causal Forest is different in one deliberate way:
because exactly one full-scale fit is authorized for this estimator through
held-out release (a compute-budget decision, not a modeling shortcut), the
fitting/checkpointing/partition-loading mechanics live in a governed,
resumable module (`src/causal_forest_runner.py`) driven by a thin CLI
(`scripts/t11_run_stage.py`) rather than in this notebook. This notebook loads
and narrates the resulting governed evidence -- predictions, diagnostic
support summary, resource evidence -- once a run exists; it does not repeat
the fitting logic inline.

Three governed evidence lines are produced by that shared runner:

- **A single FULL-scale fit** on the complete frozen TRAIN population,
  train-only, one seed. This fit -- not a later refit on combined
  train+validation data -- becomes the frozen Causal Forest artifact scored on
  held-out.
- **A bounded seed-robustness comparison** across three model seeds, fit on
  one common, joint-(treatment, conversion)-stratified TRAIN subset and
  scored on the complete VALIDATION population, reported without selecting a
  "best" seed.
- **A bounded environment-parity check**, comparing a small reference fit
  executed locally against the same fit reproduced on a different execution
  environment, before that environment is trusted for the larger runs above.

None of the three has executed as of this notebook version. The cell below
only declares which of them is authorized to run; every flag is `False`.

In [ ]:
RUN_T11_FULL_STAGE = False
RUN_T11_ROBUSTNESS_STAGE = False
RUN_T11_GCP_PARITY_STAGE = False

T11_CONFIG_PATH = REPO_ROOT / 'configs' / 't11_causal_forest_full.json'
T11_GCP_PARITY_CONFIG_PATH = REPO_ROOT / 'configs' / 't11_gcp_parity.json'
t11_config = json.loads(T11_CONFIG_PATH.read_text(encoding='utf-8'))
t11_gcp_parity_config = json.loads(T11_GCP_PARITY_CONFIG_PATH.read_text(encoding='utf-8'))

FULL_POPULATION_SIZE_T11 = t11_config['stage_definitions']['full']['population_size']
FULL_MODEL_SEED_T11 = t11_config['stage_definitions']['full']['model_seed']
ROBUSTNESS_POPULATION_SIZE_T11 = t11_config['stage_definitions']['robustness']['population_size']
ROBUSTNESS_MODEL_SEEDS_T11 = t11_config['stage_definitions']['robustness']['model_seeds']
ROBUSTNESS_SAMPLING_SEED_T11 = t11_config['stage_definitions']['robustness']['sampling_seed']
DIAGNOSTIC_SAMPLE_SIZE_T11 = t11_config['stage_definitions']['full']['diagnostic_sample_size']

print(
    f'T11 declared, not executed. FULL population={FULL_POPULATION_SIZE_T11:,} '
    f'(seed {FULL_MODEL_SEED_T11}); robustness population={ROBUSTNESS_POPULATION_SIZE_T11:,} '
    f'x seeds {ROBUSTNESS_MODEL_SEEDS_T11}.'
)

### 6.1 Protocol

**Question.** Does the frozen `econml.grf.CausalForest` configuration selected
in T10 recover a stable, well-identified treatment-effect ranking at the
single authorized full-data scale, and is that estimate materially sensitive
to its random seed when refit on a common bounded subset?

**Execution path (never inline in this notebook).**
`src.causal_forest_runner.run_stage()` -- driven by
`scripts/t11_run_stage.py` against the frozen T05 split -- verifies train/
validation disjointness, loads only the TRAIN partition needed for a given
stage via `src.data.materialize_pandas_by_source_row_id` (a compact
`Dataset.take()` read keyed on `_source_row_id`, never a whole-population
`materialize_pandas` load and never a Python `set`/`list` of millions of ids),
fits, atomically serializes and hashes the model, frees the TRAIN frame,
independently loads only the VALIDATION partition, persists predictions, and
records a bounded diagnostic sample -- all behind append-only, hash-chained
checkpoints so an interrupted run resumes without refitting.

**Verification, once a run exists.** This section will load
`audit/artifact_manifest.json`, `audit/diagnostic_support_summary.json`, and
`predictions/validation_tau.parquet` from the governed run root, verify the
manifest hashes against the files on disk, and apply the same T10 hard
correctness gate (`alpha`/`jac`/`tau` finite, `tau` non-degenerate) -- rank
and condition-number diagnostics remain reported, never gating.

**Limitations.** A predicted `tau` is not a true individual treatment effect;
balance diagnostics do not prove randomization; `f0`-`f11` are anonymized
features with no known business meaning. The seed-robustness comparison
estimates algorithmic sensitivity on a bounded subset -- it is not a second
full-scale fit and never selects a "best" seed.